# Librerias, conexiones, funiciones

In [128]:
import pandas as pd
import pyodbc
import numpy as np
from sqlalchemy import create_engine
import seaborn as sns
import matplotlib.pyplot as plt
import re
import unicodedata
import os
from pathlib import Path
from statistics import mode, StatisticsError
from typing import Dict, Any, Tuple

In [129]:
inicio = pd.Timestamp('2025-09-24')
fin = pd.Timestamp('2026-02-21')

In [130]:
cnxn = pyodbc.connect('DRIVER=NetSuite Drivers 64bit;\
                      Host=5845631.connect.api.netsuite.com;\
                      Port=1708;\
                      Encrypted=1;\
                      AllowSinglePacketLogout=1;\
                      Truststore=system;\
                      SDSN=NetSuite2.com;\
                      UID=gloria.cabrera@servimeters.com;\
                      PWD=NomegustaNS0;\
                      CustomProperties=AccountID=5845631;\
                      RoleID=1060')
cursor = cnxn.cursor()

# Cargue de datos

In [131]:
pisos=pd.read_excel('pisos.xlsx')
pisos.info()

<class 'pandas.DataFrame'>
RangeIndex: 2172 entries, 0 to 2171
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   perido              2172 non-null   datetime64[us]
 1   ejecutivo           2172 non-null   str           
 2   linea_comision      2148 non-null   str           
 3   meta                2139 non-null   float64       
 4   piso                2139 non-null   float64       
 5   comision_ejecutivo  2169 non-null   float64       
 6   jefe                2145 non-null   str           
 7   comision_jefe       2145 non-null   float64       
 8   gerente             2169 non-null   str           
 9   comision_gerente    2169 non-null   float64       
 10  Condiciones         444 non-null    str           
 11  Unnamed: 11         0 non-null      float64       
 12  Unnamed: 12         1 non-null      str           
 13  Unnamed: 13         1 non-null      float64       
dtypes: 

In [132]:
reemplazos=pd.read_excel('reemplazosProfAnuladas.xlsx')

In [133]:
reemplazos.info()

<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   original   201 non-null    str  
 1   reemplazo  201 non-null    str  
dtypes: str(2)
memory usage: 3.3 KB


In [134]:
pagado=pd.read_excel('comisionespagadas.xlsx')
pagado.info()

<class 'pandas.DataFrame'>
RangeIndex: 33904 entries, 0 to 33903
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   DOCUMENTO               33904 non-null  object 
 1   DESCRIPCION MOVIMIENTO  33904 non-null  str    
 2   %                       16157 non-null  float64
 3   VR. COMISION            16145 non-null  object 
 4   FECHA PAGO              23025 non-null  object 
 5   pagada                  33904 non-null  bool   
dtypes: bool(1), float64(1), object(3), str(1)
memory usage: 1.3+ MB


In [135]:
# creada para eliminar valores en coincidencias que son iguales al dcto_origen

def limpiar_coincidencias(row):
    coincidencias = row['coincidencias']
    dcto = row['dcto_origen']
    
    # Si es lista y contiene solo el mismo valor que dcto, limpiarla
    if isinstance(coincidencias, list):
        coincidencias_filtradas = [c for c in coincidencias if c != dcto]
        return coincidencias_filtradas if coincidencias_filtradas else None
    return coincidencias

In [136]:
def remove_tranid_from_list(row):
    """
    Retorna la lista de coincidencias sin el valor de tranId.
    Si coincidencias no es una lista, la retorna tal cual.
    """
    coincidencias = row['coincidencias']
    tranid = row['tranId']
    
    if isinstance(coincidencias, list):
        # Crea una nueva lista excluyendo el valor igual a tranId
        return [c for c in coincidencias if c != tranid]
    else:
        return coincidencias

In [137]:
def procesar_grupo(grupo):
    if grupo['Valor'].nunique() == 1:
        # Todos los valores iguales → conservar fecha más antigua
        return grupo.sort_values('tranDate', ascending=True).head(1)
    else:
        # Valores diferentes → conservar uno y marcar nota
        fila = grupo.sort_values('tranDate', ascending=True).head(1).copy()
        fila['nota'] = 'tiene más de una nota crédito'
        return fila

## Facturas
Facturas con fecha mayor a 2024-01-01

In [138]:
f="""select t.tranDate,t.dueDate, t.tranId, t.entity,  s.entityid, i.fullName as Servicio, l.rate as Precio, l.quantity, g.amount*(-1) as Valor, createdfrom, t.id, t.status, custbody_sm_factura_reemplazo, t.custbody_ks_sm_proforma, t.type,
i.custitem_sm_lineacomision, k.name as cc, b.name as lote, t.custbody_sm_reemplaza_a, t.memo
    from transaction as t
        inner join transactionLine as l  on t.id=l.transaction
        left join item as i on l.item=i.id
        left join TransactionAccountingLine as u on  u.transactionline = l.id and u.transaction=l.transaction
        inner join account as a on u.account=a.id
        left join classification as c on l.class=c.id
        left join employee as s on t.employee = s.id 
        left join department as d on l.department=d.id    
        left join CUSTOMRECORD_CSEG1 as k on l.cseg1=k.id

        left join CUSTOMRECORD_CSEG2 as b on l.cseg2=b.id
        inner join TransactionAccountingLine as g on l.id=g.transactionline and l.transaction=g.transaction
    where type in ('CustInvc')  and i.custitem_sm_linea_serv not in (18) AND t.tranDate >= TO_DATE('2021-01-01', 'YYYY-MM-DD')"""
    
    #, n.nextDoc
            #, NextTransactionLink.nextdoc as siguiente_doc,          left join NextTransactionLink  on t.id=NextTransactionLink.nextdoc

In [139]:
cursor.execute(f)
nombres_columnas = [column[0] for column in cursor.description]
rows = cursor.fetchall()
resultados_tuplas = [tuple(row) for row in rows]

In [140]:
df = pd.DataFrame(resultados_tuplas, columns=nombres_columnas)
#df['createdfrom'] = df['createdfrom'].fillna(0).astype(int)
f=df.copy()
f

,tranDate,dueDate,tranId,entity,entityid,Servicio,Precio,quantity,Valor,createdfrom,id,status,custbody_sm_factura_reemplazo,custbody_ks_sm_proforma,type,custitem_sm_lineacomision,cc,lote,custbody_sm_reemplaza_a,memo
0,2022-05-20,2022-06-19,SM24994,3234,JESUS DANIEL TORREGROSA EBRATT,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,2447200.0,-1.0,2447200.0,NaN,560420,B,None,F,CustInvc,15.0,RETIE BARRANQUILLA,114427,NaN,Certificación RETIE 20 APTOS - TORRE 45 m2. Fa...
1,2022-08-03,2022-09-02,SM27260,3577,CLARA ISABEL MUÑOZ,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,8702660.0,-1.0,8702660.0,640051.0,643570,B,None,F,CustInvc,15.0,RITEL CALI,110651,NaN,INSP-RIT-110651-2021 / Proyecto: Los Naranjos ...
2,2022-08-22,2022-09-21,SM27887,3222,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,5100020.0,-1.0,5100020.0,NaN,664951,B,None,F,CustInvc,15.0,RITEL BOGOTA,111412,NaN,INSP-RIT-111412-2021 / PROYECTO: BIO ET1 OBR. ...
3,2022-09-16,2022-09-26,SM28758,35456,CLARA ISABEL MUÑOZ,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,5600000.0,-1.0,5600000.0,693459.0,695890,B,None,F,CustInvc,15.0,RITEL CALI,123008,NaN,INSP-RIT-123008-2022 / Proyecto: EL GUAYACAN /...
4,2022-12-18,2022-12-28,SM32030,85626,LUISA FERNANDA CAMACHO CUADRADO,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,2700000.0,-1.0,2700000.0,639914.0,799023,B,None,T,CustInvc,15.0,RITEL MEDELLIN,122878,NaN,PEDIDO 613999 - INSP-RIT-122878-2022-RITEL MON...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80310,2025-04-03,2025-04-03,SM57665,30514,LAURA ELENA RAMIREZ ROJAS,Servicios de calibración con laboratorios subc...,174000.0,-8.0,1392000.0,1804020.0,1805659,B,None,F,CustInvc,NaN,SUBCONTRATADOS LVV,179078,NaN,COT CONTRATO NO. 1153029. CONTRATO NO. 1153029
80311,2025-04-03,2025-04-03,SM57665,30514,LAURA ELENA RAMIREZ ROJAS,Servicios de calibración con laboratorios subc...,610000.0,-8.0,4880000.0,1804020.0,1805659,B,None,F,CustInvc,NaN,SUBCONTRATADOS LVV,179078,NaN,COT CONTRATO NO. 1153029. CONTRATO NO. 1153029
80312,2025-03-10,2025-03-10,SM56906,16986,LAURA ELENA RAMIREZ ROJAS,Servicios de calibración con laboratorios subc...,182000.0,-1.0,182000.0,1784381.0,1786711,B,None,F,CustInvc,NaN,SUBCONTRATADOS LVV,174683,NaN,"COT 174683, OC 4158"
80313,2025-03-12,2025-04-11,SM56989,85597,LAURA ELENA RAMIREZ ROJAS,Servicios de calibración con laboratorios subc...,217620.0,-2.0,435240.0,1787574.0,1788430,B,None,F,CustInvc,NaN,SUBCONTRATADOS LVV,176197,NaN,COT 176195 -176197


In [141]:
c="""select t.tranDate,t.dueDate, t.tranId, t.entity,  s.entityid, i.fullName as Servicio, l.rate as Precio, l.quantity, g.amount*(-1) as Valor, createdfrom, t.id, t.status, custbody_sm_factura_reemplazo, t.custbody_ks_sm_proforma, t.type,
i.custitem_sm_lineacomision, k.name as cc, b.name as lote, t.custbody_sm_reemplaza_a, t.memo
    from transaction as t
        inner join transactionLine as l  on t.id=l.transaction
        left join item as i on l.item=i.id
        left join TransactionAccountingLine as u on  u.transactionline = l.id and u.transaction=l.transaction
        inner join account as a on u.account=a.id
        left join classification as c on l.class=c.id
        left join employee as s on t.employee = s.id 
        left join department as d on l.department=d.id    
        left join CUSTOMRECORD_CSEG1 as k on l.cseg1=k.id

        left join CUSTOMRECORD_CSEG2 as b on l.cseg2=b.id
        inner join TransactionAccountingLine as g on l.id=g.transactionline and l.transaction=g.transaction
    where type in ('PurchOrd')  AND t.tranDate >= TO_DATE('2021-01-01', 'YYYY-MM-DD')
    AND t.status NOT IN ('H', 'C')"""
    

In [142]:
cursor.execute(c)
nombres_columnas = [column[0] for column in cursor.description]
rows = cursor.fetchall()
resultados_tuplas = [tuple(row) for row in rows]

In [143]:
df = pd.DataFrame(resultados_tuplas, columns=nombres_columnas)
#df['createdfrom'] = df['createdfrom'].fillna(0).astype(int)
c=df.copy()
c = c.dropna(subset=['lote'])
c

,tranDate,dueDate,tranId,entity,entityid,Servicio,Precio,quantity,Valor,createdfrom,id,status,custbody_sm_factura_reemplazo,custbody_ks_sm_proforma,type,custitem_sm_lineacomision,cc,lote,custbody_sm_reemplaza_a,memo
1,2021-02-15,NaT,166000,11258,NaN,SERVICIOS FREELANCE,160000.0,1.0,-160000.0,NaN,94077,G,None,None,PurchOrd,NaN,RETIE CONVENIOS,98223,None,NaN
3,2021-02-19,NaT,166091,43317,MARIA CONSUELO JIMENEZ AVILA,ALMUERZO OPERATIVO,15000.0,1.0,-15000.0,93182.0,99631,G,None,None,PurchOrd,NaN,ESCALERAS Y ASCENSORES BARRANQUILLA,97510,None,Visita N°2 DE 2 VISITAS - FACTURA VENTA 11917 ...
4,2021-02-19,NaT,166091,43317,MARIA CONSUELO JIMENEZ AVILA,HIDRATACION OPERATIVO,6000.0,1.0,-6000.0,93182.0,99631,G,None,None,PurchOrd,NaN,ESCALERAS Y ASCENSORES BARRANQUILLA,97510,None,Visita N°2 DE 2 VISITAS - FACTURA VENTA 11917 ...
5,2021-02-19,NaT,166091,43317,MARIA CONSUELO JIMENEZ AVILA,TRANSPORTES INTERMUNICIPALES OPERATIVO,100000.0,1.0,-100000.0,93182.0,99631,G,None,None,PurchOrd,NaN,ESCALERAS Y ASCENSORES BARRANQUILLA,97510,None,Visita N°2 DE 2 VISITAS - FACTURA VENTA 11917 ...
7,2021-02-19,NaT,166091,43317,MARIA CONSUELO JIMENEZ AVILA,TRANSPORTES INTERNOS OPERATIVO,40000.0,1.0,-40000.0,93182.0,99631,G,None,None,PurchOrd,NaN,ESCALERAS Y ASCENSORES BARRANQUILLA,97510,None,Visita N°2 DE 2 VISITAS - FACTURA VENTA 11917 ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145334,2026-06-01,NaT,192934,43273,GEOVANNY ANDRES QUIMBAYO FLORIDO,MEDIOS DE TRANSPORTE (T) PRO,60000.0,1.0,-60000.0,2093028.0,2103541,F,None,None,PurchOrd,NaN,RETIE BOGOTA,197769,None,Comercial Lorena Vasquez; Desplazamiento de Bo...
145336,2026-05-27,2026-05-26,192858,17563,GEOVANNY ANDRES QUIMBAYO FLORIDO,MEDIOS DE TRANSPORTE (M) PRO,23000.0,1.0,-23000.0,2100003.0,2100373,F,None,None,PurchOrd,NaN,RETIE PEREIRA,200124,None,200124-2026
145337,2026-05-27,2026-05-26,192858,17563,GEOVANNY ANDRES QUIMBAYO FLORIDO,MEDIOS DE TRANSPORTE (T) PRO,29000.0,1.0,-29000.0,2100003.0,2100373,F,None,None,PurchOrd,NaN,RETIE PEREIRA,200124,None,200124-2026
145340,2026-05-22,NaT,192775,11247,NaN,SERVICIOS FREELANCE,191625.0,1.0,-191625.0,NaN,2097048,G,None,None,PurchOrd,NaN,RETIE CONVENIOS,200732,None,NaN


In [144]:
costo=  (
    c.groupby('lote', as_index=False)['Valor']
     .sum()
)
costo

,lote,Valor
0,100003,-1170000.0
1,100009,-200000.0
2,100017,-157000.0
3,100030,-672268.5
4,100042,-589000.0
...,...,...
15613,99948,-450000.0
15614,99950,-247000.0
15615,99972,-219000.0
15616,99973,-159000.0


## Proformas (ordenes de venta)
Ordenes de venta con fecha mayor a 2024-01-01

In [145]:
p="""select t.tranDate,t.enddate as dueDate, t.tranId,t.entity,   s.entityid,
i.fullName as Servicio, l.rate as Precio, l.quantity, g.amount*(-1) as Valor, t.custbody_ks_sm_proforma, custbody_sm_factura_reemplazo, createdfrom, t.id, t.status, t.type,
i.custitem_sm_lineacomision, k.name as cc, b.name as lote,t.custbody_sm_reemplaza_a, t.memo

    from transaction as t
        inner join transactionLine as l  on t.id=l.transaction
        left join item as i on l.item=i.id
        left join TransactionAccountingLine as u on  u.transactionline = l.id and u.transaction=l.transaction
        inner join account as a on u.account=a.id
        left join classification as c on l.class=c.id
        left join employee as s on t.employee = s.id 
        left join department as d on l.department=d.id  
            
        left join CUSTOMRECORD_CSEG1 as k on l.cseg1=k.id
        left join CUSTOMRECORD_CSEG2 as b on l.cseg2=b.id        
        inner join TransactionAccountingLine as g on l.id=g.transactionline and l.transaction=g.transaction
    where type in ('SalesOrd') and i.itemType= 'Service' and i.custitem_sm_linea_serv not in (18) and t.custbody_ks_sm_proforma ='T' AND t.tranDate >= TO_DATE('2021-01-01', 'YYYY-MM-DD')
 """
 
 #, NextTransactionLink.nextdoc as siguiente_doc ,  left join NextTransactionLink  on t.id=NextTransactionLink.nextdoc     
 

In [146]:
cursor.execute(p)
nombres_columnas = [column[0] for column in cursor.description]
rows = cursor.fetchall()
resultados_tuplas = [tuple(row) for row in rows]

In [147]:
df = pd.DataFrame(resultados_tuplas, columns=nombres_columnas)
df['createdfrom'] = df['createdfrom'].fillna(0).astype(int)
p=df.copy()
p

,tranDate,dueDate,tranId,entity,entityid,Servicio,Precio,quantity,Valor,custbody_ks_sm_proforma,custbody_sm_factura_reemplazo,createdfrom,id,status,type,custitem_sm_lineacomision,cc,lote,custbody_sm_reemplaza_a,memo
0,2021-01-15,2021-03-16,PR-0553,3281,CARLOS FABIAN GONZALEZ GARRIDO,CALIBRACION MACROMEDIDOR SITIO,688150.0,-1.0,688150.0,T,None,30783,61111,G,SalesOrd,1.0,CALIBRACION SITIO,96277,None,CAL-LMAM-96277-202O OC: 4400576197; ACEPTACION...
1,2021-01-15,2021-03-16,PR-0553,3281,CARLOS FABIAN GONZALEZ GARRIDO,CALIBRACION MACROMEDIDOR SITIO,688150.0,-1.0,688150.0,T,None,30783,61111,G,SalesOrd,1.0,CALIBRACION SITIO,96277,None,CAL-LMAM-96277-202O OC: 4400576197; ACEPTACION...
2,2021-01-15,2021-03-16,PR-0553,3281,CARLOS FABIAN GONZALEZ GARRIDO,CALIBRACION MACROMEDIDOR SITIO,688150.0,-1.0,688150.0,T,None,30783,61111,G,SalesOrd,1.0,CALIBRACION SITIO,96277,None,CAL-LMAM-96277-202O OC: 4400576197; ACEPTACION...
3,2021-01-15,2021-03-16,PR-0553,3281,CARLOS FABIAN GONZALEZ GARRIDO,CALIBRACION MACROMEDIDOR SITIO,688150.0,-1.0,688150.0,T,None,30783,61111,G,SalesOrd,1.0,CALIBRACION SITIO,96277,None,CAL-LMAM-96277-202O OC: 4400576197; ACEPTACION...
4,2021-01-15,2021-03-16,PR-0553,3281,CARLOS FABIAN GONZALEZ GARRIDO,CALIBRACION MACROMEDIDOR SITIO,688150.0,-1.0,688150.0,T,None,30783,61111,G,SalesOrd,1.0,CALIBRACION SITIO,96277,None,CAL-LMAM-96277-202O OC: 4400576197; ACEPTACION...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11112,2026-05-26,NaT,PR-9437,78681,JULY JAZMIN CARO ROMERO,AUDITORIA DE SEGUIMIENTO DE PRODUCTO ELECTRICO...,5272614.0,-1.0,5272614.0,T,None,2071320,2098945,G,SalesOrd,8.0,RETIE CERTIFICACION,199842,None,CER-PRO-199842-2026 /// OC-26PC00519
11113,2026-05-26,NaT,PR-9438,78681,JULY JAZMIN CARO ROMERO,AUDITORIA DE SEGUIMIENTO DE PRODUCTO ELECTRICO...,5272614.0,-1.0,5272614.0,T,None,2071695,2098950,G,SalesOrd,8.0,RETIE CERTIFICACION,199872,None,CER-PRO-199872-2026 /// OC-26PC00520
11114,2025-09-30,NaT,PR-9150,205101,JAIME ANDRES ZAMUDIO CACERES,AUDITORIA DE SEGUIMIENTO DE PRODUCTO VAJILLAS,9500000.0,-1.0,9500000.0,T,None,1925960,1937484,H,SalesOrd,8.0,VAJILLAS CERTIFICACION,189637,None,CER-PRO-189637-2025
11115,2025-09-30,2025-10-10,PR-9146,205101,JAIME ANDRES ZAMUDIO CACERES,AUDITORIA DE SEGUIMIENTO DE PRODUCTO VAJILLAS,9500000.0,-1.0,9500000.0,T,None,1925960,1937101,H,SalesOrd,8.0,VAJILLAS CERTIFICACION,189637,None,CER-PRO-189637-2025


## Notas Crédito
Notas crédito con fecha mayor a 2024-01-01

In [148]:
nc="""select t.tranDate,t.dueDate, t.tranId, t.entity,  s.entityid,
i.fullName as Servicio, l.rate as Precio, l.quantity, g.amount*(-1) as Valor, createdfrom, t.id, t.status, custbody_sm_factura_reemplazo, t.custbody_ks_sm_proforma, t.type,
i.custitem_sm_lineacomision, k.name as cc, b.name as lote, t.custbody_sm_reemplaza_a, t.memo 
    from transaction as t
        inner join transactionLine as l  on t.id=l.transaction
        left join item as i on l.item=i.id
        left join TransactionAccountingLine as u on  u.transactionline = l.id and u.transaction=l.transaction
        inner join account as a on u.account=a.id
        left join classification as c on l.class=c.id
        left join employee as s on t.employee = s.id 
        left join department as d on l.department=d.id    
        left join CUSTOMRECORD_CSEG1 as k on l.cseg1=k.id
        left join CUSTOMRECORD_CSEG2 as b on l.cseg2=b.id
        inner join TransactionAccountingLine as g on l.id=g.transactionline and l.transaction=g.transaction
    where type in ('CustCred') and i.itemType= 'Service' and i.custitem_sm_linea_serv not in (18) AND t.tranDate >= TO_DATE('2021-01-01', 'YYYY-MM-DD')
 """

In [149]:
cursor.execute(nc)
nombres_columnas = [column[0] for column in cursor.description]
rows = cursor.fetchall()
resultados_tuplas = [tuple(row) for row in rows]

In [150]:
df = pd.DataFrame(resultados_tuplas, columns=nombres_columnas)
df['createdfrom'] = df['createdfrom'].fillna(0).astype(int)
nc=df.copy()
nc

,tranDate,dueDate,tranId,entity,entityid,Servicio,Precio,quantity,Valor,createdfrom,id,status,custbody_sm_factura_reemplazo,custbody_ks_sm_proforma,type,custitem_sm_lineacomision,cc,lote,custbody_sm_reemplaza_a,memo
0,2022-07-15,None,NC1285,24062,CARLOS ANDRES MONTOYA VALENCIA,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,13950000.0,0.5,-6975000.0,593570,622635,B,622631,F,CustCred,15.0,RITEL PEREIRA,115138,None,ANULA FACTURA SM25811/EL CLIENTE NO LA CONTAB...
1,2024-06-20,None,NC3013,5079,ANDREA PAOLA GOMEZ PAEZ,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,33800000.0,26.0,-11700000.0,1396488,1436492,B,NaN,F,CustCred,15.0,RITEL IBAGUE,134559,None,CTA DE OBRA No. 1 CONTRATO No. 30800025- Codig...
2,2024-07-08,None,NC3060,3299,NATALY ORREGO GAVIRIA,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,7700000.0,1.0,-7700000.0,1455085,1458672,B,1458670,F,CustCred,15.0,RITEL MEDELLIN,148471,None,INSP-RIT-148471-2023 RITEL Origen Monserrate T...
3,2024-09-25,None,NC3438,103266,CLARA ISABEL MUÑOZ,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,10500000.0,1.0,-10500000.0,1584421,1594478,B,NaN,F,CustCred,15.0,RITEL CALI,155796,None,NUMERO DEL LOTE-155789 RUTA.DOCUMENTOS \\bogsr...
4,2025-07-03,None,NC4286,2997,NATALY ORREGO GAVIRIA,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,3025210.0,1.0,-3025210.0,1858922,1874522,B,NaN,F,CustCred,15.0,RITEL MEDELLIN,183745,None,ANULA FACTURA SM59717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8022,2024-09-10,None,NC3368,35355,CARLOS FABIAN GONZALEZ GARRIDO,Servicios de calibración con laboratorios subc...,1200000.0,2.0,-2400000.0,1497423,1569106,B,1569089,F,CustCred,NaN,SUBCONTRATADOS LVV,163159,None,COT: 163159 Orden de compra: No PO2535 Documen...
8023,2024-09-10,None,NC3368,35355,CARLOS FABIAN GONZALEZ GARRIDO,Servicios de calibración con laboratorios subc...,600000.0,1.0,-600000.0,1497423,1569106,B,1569089,F,CustCred,NaN,SUBCONTRATADOS LVV,163159,None,COT: 163159 Orden de compra: No PO2535 Documen...
8024,2024-09-10,None,NC3368,35355,CARLOS FABIAN GONZALEZ GARRIDO,Servicios de calibración con laboratorios subc...,410000.0,7.0,-2870000.0,1497423,1569106,B,1569089,F,CustCred,NaN,SUBCONTRATADOS LVV,163159,None,COT: 163159 Orden de compra: No PO2535 Documen...
8025,2025-05-12,None,NC4174,69607,LAURA ELENA RAMIREZ ROJAS,Servicios de calibración con laboratorios subc...,406000.0,2.0,-812000.0,1783300,1833801,B,NaN,F,CustCred,NaN,SUBCONTRATADOS LVV,176669,None,"COT 176669, OC 1348390 entrada almacen OV7126636"


## Notas Credito proformas

In [151]:
ncp="""select t.tranDate, t.tranId,  t.memo, g.amount as Valor
    from transaction as t
        inner join transactionLine as l  on t.id=l.transaction
        inner join TransactionAccountingLine as g on l.id=g.transactionline and l.transaction=g.transaction
    where type in ('Journal') AND t.tranDate >= TO_DATE('2021-01-01', 'YYYY-MM-DD') and custbody_ks_sm_tipoje=6
 """

In [152]:
cursor.execute(ncp)
nombres_columnas = [column[0] for column in cursor.description]
rows = cursor.fetchall()
resultados_tuplas = [tuple(row) for row in rows]

In [153]:
df = pd.DataFrame(resultados_tuplas, columns=nombres_columnas)
df=df[df['Valor']<=0]
#df['createdfrom'] = df['createdfrom'].fillna(0).astype(int)
ncp=df.copy()
ncp

,tranDate,tranId,memo,Valor
0,2021-02-28,NC-000018,ANULACION PR-0516,-4760000.00
2,2021-02-28,NC-000020,ANULACION R1-14321,-4165000.00
4,2021-02-28,NC-000014,ANULACION R1-14195,-2570400.00
6,2021-02-28,NC-000021,ANULACION R1-14097,-5046238.05
8,2021-02-28,NC-000019,ANULACION PR-0529,-2356200.00
...,...,...,...,...
1645,2025-10-15,NC-001014,SE CIERRA PR-9146 LA CERRO EL COMERCIAL POR ERROR,-12084450.00
1647,2025-09-29,NC-001006,SE CIERRA PR-9000 SE REEMPLAZA CON SM61911,-1338750.00
1649,2025-10-17,NC-001020,SE CIERRA PR-8878 SIN REEMPLAZO POR CONCILIACI...,-520000.00
1650,2025-10-17,NC-001020,SE CIERRA PR-8878 SIN REEMPLAZO POR CONCILIACI...,-2574000.00


In [154]:
def dividir_texto(texto):
    if pd.isna(texto):
        return pd.Series([None, None])
    match = re.search(r'(reemp\w*|factura)', texto, flags=re.IGNORECASE)
    if match:
        idx = match.start()
        return pd.Series([texto[:idx].strip(), texto[idx:].strip()])
    return pd.Series([texto.strip(), None])

# Aplicar y crear nuevas columnas
ncp[['antes_texto', 'desde_texto']] = ncp['memo'].apply(dividir_texto)

In [155]:
def extraer_documentos(texto):
    if not isinstance(texto, str):
        return []

    texto = texto.lower()

    # Buscar patrones tipo PR con guion, con espacio o con espacio y guion intermedios
    patrones_pr = re.findall(r'\b(?:pr[\s-]*|proforma[\s-]*)(\d{3,6})\b', texto)

    # Buscar patrones tipo R1 (respetando guiones o espacios)
    patrones_r1 = re.findall(r'\b(R1[\s-]?\d{3,6})\b', texto, flags=re.IGNORECASE)

    # Normalizar resultados
    pr_normalizados = [f'PR-{num}' for num in patrones_pr]
    r1_normalizados = [re.sub(r'\s', '-', r.upper()) for r in patrones_r1]

    return pr_normalizados + r1_normalizados

# Aplicar la función y explotar
ncp['documentos_detectados'] = ncp['antes_texto'].apply(extraer_documentos)
ncp = ncp.explode('documentos_detectados').reset_index(drop=True)

In [156]:
ncp

,tranDate,tranId,memo,Valor,antes_texto,desde_texto,documentos_detectados
0,2021-02-28,NC-000018,ANULACION PR-0516,-4760000.00,ANULACION PR-0516,NaN,PR-0516
1,2021-02-28,NC-000020,ANULACION R1-14321,-4165000.00,ANULACION R1-14321,NaN,R1-14321
2,2021-02-28,NC-000014,ANULACION R1-14195,-2570400.00,ANULACION R1-14195,NaN,R1-14195
3,2021-02-28,NC-000021,ANULACION R1-14097,-5046238.05,ANULACION R1-14097,NaN,R1-14097
4,2021-02-28,NC-000019,ANULACION PR-0529,-2356200.00,ANULACION PR-0529,NaN,PR-0529
...,...,...,...,...,...,...,...
864,2025-10-15,NC-001014,SE CIERRA PR-9146 LA CERRO EL COMERCIAL POR ERROR,-12084450.00,SE CIERRA PR-9146 LA CERRO EL COMERCIAL POR ERROR,NaN,PR-9146
865,2025-09-29,NC-001006,SE CIERRA PR-9000 SE REEMPLAZA CON SM61911,-1338750.00,SE CIERRA PR-9000 SE,REEMPLAZA CON SM61911,PR-9000
866,2025-10-17,NC-001020,SE CIERRA PR-8878 SIN REEMPLAZO POR CONCILIACI...,-520000.00,SE CIERRA PR-8878 SIN,REEMPLAZO POR CONCILIACION DE CARTERA,PR-8878
867,2025-10-17,NC-001020,SE CIERRA PR-8878 SIN REEMPLAZO POR CONCILIACI...,-2574000.00,SE CIERRA PR-8878 SIN,REEMPLAZO POR CONCILIACION DE CARTERA,PR-8878


In [157]:
ncp=ncp[ncp['documentos_detectados'].notnull() & (ncp['documentos_detectados'] != '')]
ncp

,tranDate,tranId,memo,Valor,antes_texto,desde_texto,documentos_detectados
0,2021-02-28,NC-000018,ANULACION PR-0516,-4760000.00,ANULACION PR-0516,NaN,PR-0516
1,2021-02-28,NC-000020,ANULACION R1-14321,-4165000.00,ANULACION R1-14321,NaN,R1-14321
2,2021-02-28,NC-000014,ANULACION R1-14195,-2570400.00,ANULACION R1-14195,NaN,R1-14195
3,2021-02-28,NC-000021,ANULACION R1-14097,-5046238.05,ANULACION R1-14097,NaN,R1-14097
4,2021-02-28,NC-000019,ANULACION PR-0529,-2356200.00,ANULACION PR-0529,NaN,PR-0529
...,...,...,...,...,...,...,...
864,2025-10-15,NC-001014,SE CIERRA PR-9146 LA CERRO EL COMERCIAL POR ERROR,-12084450.00,SE CIERRA PR-9146 LA CERRO EL COMERCIAL POR ERROR,NaN,PR-9146
865,2025-09-29,NC-001006,SE CIERRA PR-9000 SE REEMPLAZA CON SM61911,-1338750.00,SE CIERRA PR-9000 SE,REEMPLAZA CON SM61911,PR-9000
866,2025-10-17,NC-001020,SE CIERRA PR-8878 SIN REEMPLAZO POR CONCILIACI...,-520000.00,SE CIERRA PR-8878 SIN,REEMPLAZO POR CONCILIACION DE CARTERA,PR-8878
867,2025-10-17,NC-001020,SE CIERRA PR-8878 SIN REEMPLAZO POR CONCILIACI...,-2574000.00,SE CIERRA PR-8878 SIN,REEMPLAZO POR CONCILIACION DE CARTERA,PR-8878


## Cartera prof

In [158]:
df=pd.read_excel("Reporte de Proformas_2026_02_17.xlsx")
#"D:\OneDrive\OneDrive - Servimeters\Proyectos\2025\comisiones\Reporte de Proformas_2026_02_17.xls"

In [159]:
df

,Numero de Proforma,Tercero,Estatus,Centro de Costo,Lote,Fecha de Emision,Ejecutivo comercial (Rep de Ventas),Fecha de Vencimiento,Valor Total Proformas,Valor pagado,Saldo Pendiente,Fecha Real Pago,Fecha de Contabilizacion de Pago,Pagado Con,Freelance
0,PR-0001,3689 O&G INGENIERIA SAS,Facturada,RETIE MEDELLIN,90080,12/11/2020,MONICA MERCEDES SAÑUDO HERRERA,22/11/2020,1.785000e+06,1.785000e+06,0.0,28/09/2020,30/11/2020,",Pago #215,",- None -
1,PR-0150,1207 INGENIEROS ESPECIALISTAS EN ENERGÍA ELÉCT...,Facturada,RETIE MEDELLIN,90525,02/10/2020,MONICA MERCEDES SAÑUDO HERRERA,01/12/2020,1.309000e+06,1.309000e+06,0.0,23/11/2020,26/11/2020,",Pago #109,",- None -
2,PR-0002,1136 INGENIERIA M B A S.A.S,Facturada,RETIE CARTAGENA,93013,11/11/2020,LORENA ROCIO VASQUEZ FORERO,21/11/2020,1.199520e+07,1.199520e+07,0.0,06/11/2020,30/11/2020,",Diario #368,",- None -
3,PR-0005,939 GRUPO QUALITAS SAS,Facturada,RETIE BARRANQUILLA,90951,27/10/2020,NATALIA DEL PILAR OSPINA PULIDO,22/11/2020,1.309000e+06,1.309000e+06,0.0,NaN,18/12/2020,", ,Diario #1015",- None -
4,PR-0003,939 GRUPO QUALITAS SAS,Facturada,RETIE BARRANQUILLA,90952,27/10/2020,NATALIA DEL PILAR OSPINA PULIDO,22/11/2020,1.309000e+06,1.309000e+06,0.0,NaN,18/12/2020,",Diario #1013,",- None -
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9481,PR-9481,12378 TRANSFORMADORES GVR ENERGY S.A.S,Facturación pendiente,RETIE IBAGUE,201742,01/06/2026,ANDREA PAOLA GOMEZ PAEZ,01/07/2026,4.500000e+06,0.000000e+00,4500000.0,NaN,NaN,", ,",- None -
9482,PR-9482,24719 CARIBEMAR DE LA COSTA SAS ESP,Facturada,"ENERGIA USADOS,ENERGIA NUEVOS",128398,02/06/2026,LINDA YESENIA SABIO BARRERO,02/07/2026,1.428567e+08,1.428567e+08,0.0,NaN,11/06/2026,",Diario #35490,",- None -
9483,PR-9483,1003 ERCO ENERGIA S.A.S,Facturación pendiente,RETIE MEDELLIN,196527,03/06/2026,LEIDY JOHANA TORRES MORALES,03/07/2026,7.882500e+06,0.000000e+00,7882500.0,NaN,NaN,", ,",- None -
9484,PR-9484,3352 CONSTRUCTORA CONCONCRETO SA,Facturación pendiente,RETIE MEDELLIN,155285,05/06/2026,YESICA PAOLA BUSTAMANTE CAÑAS,05/07/2026,3.855600e+06,0.000000e+00,3855600.0,NaN,NaN,", ,",- None -


In [160]:
cpf=df.copy()
cpf

,Numero de Proforma,Tercero,Estatus,Centro de Costo,Lote,Fecha de Emision,Ejecutivo comercial (Rep de Ventas),Fecha de Vencimiento,Valor Total Proformas,Valor pagado,Saldo Pendiente,Fecha Real Pago,Fecha de Contabilizacion de Pago,Pagado Con,Freelance
0,PR-0001,3689 O&G INGENIERIA SAS,Facturada,RETIE MEDELLIN,90080,12/11/2020,MONICA MERCEDES SAÑUDO HERRERA,22/11/2020,1.785000e+06,1.785000e+06,0.0,28/09/2020,30/11/2020,",Pago #215,",- None -
1,PR-0150,1207 INGENIEROS ESPECIALISTAS EN ENERGÍA ELÉCT...,Facturada,RETIE MEDELLIN,90525,02/10/2020,MONICA MERCEDES SAÑUDO HERRERA,01/12/2020,1.309000e+06,1.309000e+06,0.0,23/11/2020,26/11/2020,",Pago #109,",- None -
2,PR-0002,1136 INGENIERIA M B A S.A.S,Facturada,RETIE CARTAGENA,93013,11/11/2020,LORENA ROCIO VASQUEZ FORERO,21/11/2020,1.199520e+07,1.199520e+07,0.0,06/11/2020,30/11/2020,",Diario #368,",- None -
3,PR-0005,939 GRUPO QUALITAS SAS,Facturada,RETIE BARRANQUILLA,90951,27/10/2020,NATALIA DEL PILAR OSPINA PULIDO,22/11/2020,1.309000e+06,1.309000e+06,0.0,NaN,18/12/2020,", ,Diario #1015",- None -
4,PR-0003,939 GRUPO QUALITAS SAS,Facturada,RETIE BARRANQUILLA,90952,27/10/2020,NATALIA DEL PILAR OSPINA PULIDO,22/11/2020,1.309000e+06,1.309000e+06,0.0,NaN,18/12/2020,",Diario #1013,",- None -
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9481,PR-9481,12378 TRANSFORMADORES GVR ENERGY S.A.S,Facturación pendiente,RETIE IBAGUE,201742,01/06/2026,ANDREA PAOLA GOMEZ PAEZ,01/07/2026,4.500000e+06,0.000000e+00,4500000.0,NaN,NaN,", ,",- None -
9482,PR-9482,24719 CARIBEMAR DE LA COSTA SAS ESP,Facturada,"ENERGIA USADOS,ENERGIA NUEVOS",128398,02/06/2026,LINDA YESENIA SABIO BARRERO,02/07/2026,1.428567e+08,1.428567e+08,0.0,NaN,11/06/2026,",Diario #35490,",- None -
9483,PR-9483,1003 ERCO ENERGIA S.A.S,Facturación pendiente,RETIE MEDELLIN,196527,03/06/2026,LEIDY JOHANA TORRES MORALES,03/07/2026,7.882500e+06,0.000000e+00,7882500.0,NaN,NaN,", ,",- None -
9484,PR-9484,3352 CONSTRUCTORA CONCONCRETO SA,Facturación pendiente,RETIE MEDELLIN,155285,05/06/2026,YESICA PAOLA BUSTAMANTE CAÑAS,05/07/2026,3.855600e+06,0.000000e+00,3855600.0,NaN,NaN,", ,",- None -


## Cartera Facturas

In [161]:
df=pd.read_excel("ResultadosKSREPORTECARTERAROBOT285.xlsx")

In [162]:
df

,Ver,Factura,Fecha Factura,Fecha Vencimiento,Nombre,Nit,Representante de Ventas,Valor Factura,Dias Vencido,Centro de Costo,Lote(s),Pagado Con (Tipo),Fecha Contabilizado Pago,Fecha Real Pago,Pagado Con:,Pagado,Saldo,Freelance,Condicion General,Observacion General,Ultima Modificacion
0,"<a href=""/app/accounting/transactions/custinvc...",ND0028,2024-05-22,2024-05-22,22683 SERVICIO NACIONAL DE APRENDIZAJE,899999034,MARIA JOSE PEREZ CERVANTES,1.000000e-02,751.0,CALIBRACION SITIO,150367,Nota de crédito,2024-03-26,NaT,Nota de crédito #NC2750,0,0,NaN,NaN,NaN,NaT
1,"<a href=""/app/accounting/transactions/custinvc...",ND75,2025-10-28,2025-10-28,58428 RAMA JUDICIAL DIRECCION SECCIONAL DE ADM...,- None -,ANGIE ALISON BERMUDEZ DIAZ,1.500000e-01,227.0,ESCALERAS Y ASCENSORES BOGOTA,173370,Diario,2025-12-31,NaT,Diario #33528,0,0,NaN,NaN,NaN,NaT
2,"<a href=""/app/accounting/transactions/custinvc...",ND0001,2020-11-07,2020-11-22,2357 INGENIERIA Y SERVICIO SAS INCER SAS,800083329,ADRIANA MARITZA QUITIAN ROZO,7.190000e+00,2028.0,CANALETAS PARSHALL Y AGUAS RESIDUALES,- None -,Nota de crédito,2020-11-07,NaT,Nota de crédito #NC0001,7,0,NaN,NaN,NaN,NaT
3,"<a href=""/app/accounting/transactions/custinvc...",SM10001,2020-11-07,2020-11-22,2357 INGENIERIA Y SERVICIO SAS INCER SAS,800083329,ADRIANA MARITZA QUITIAN ROZO,1.198000e+01,2028.0,CANALETAS PARSHALL Y AGUAS RESIDUALES,- None -,"Nota de crédito,Nota de crédito",2020-11-07,NaT,"Nota de crédito #NC0001,Nota de crédito #NC0001",12,0,NaN,NaN,NaN,NaT
4,"<a href=""/app/accounting/transactions/custinvc...",SM45770,2024-04-02,NaT,34165 G-MAS CONSTRUCTORES S.A.S.,900967098,GUSTAVO ADOLFO CARDENAS MORENO,1.000000e+02,NaN,RETIE CONVENIOS,152104,"Nota de crédito,Nota de crédito",2024-04-02,NaT,"Nota de crédito #NC2794,Nota de crédito #NC2794",100,0,NaN,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61535,"<a href=""/app/accounting/transactions/custinvc...",SM69496,2026-04-29,2026-04-29,49881 LARGACHA CONSULTORES S.A.S.,900437761,GUSTAVO ADOLFO CARDENAS MORENO,5.678788e+09,44.0,SISTEMAS DE GESTION COLOMBIA,199907,Nota de crédito,2026-04-30,NaT,Nota de crédito #NC5035,5678788290,0,NaN,NaN,NaN,NaT
61536,"<a href=""/app/accounting/transactions/custinvc...",SM60042,2025-06-19,2025-06-19,49881 LARGACHA CONSULTORES S.A.S.,900437761,GUSTAVO ADOLFO CARDENAS MORENO,9.701642e+09,358.0,SISTEMAS DE GESTION PANAMA,185174,"Nota de crédito,Nota de crédito",2025-06-20,NaT,"Nota de crédito #NC4258,Nota de crédito #NC4258",9701641763,0,NaN,NaN,NaN,NaT
61537,"<a href=""/app/accounting/transactions/custinvc...",SM60041,2025-06-19,2025-06-19,49881 LARGACHA CONSULTORES S.A.S.,900437761,GUSTAVO ADOLFO CARDENAS MORENO,1.112794e+10,358.0,SISTEMAS DE GESTION PANAMA,185173,"Nota de crédito,Nota de crédito",2025-06-20,NaT,"Nota de crédito #NC4257,Nota de crédito #NC4257",11127943179,0,NaN,NaN,NaN,NaT
61538,"<a href=""/app/accounting/transactions/custinvc...",ND0050,2024-11-19,2024-11-19,902 PRICESMART COLOMBIA SAS,900319753,JAVIER ALEJANDRO ACEVEDO TORRES,3.551885e+10,570.0,RETIE CALI,154071,"Nota de crédito,Nota de crédito",2024-11-19,NaT,"Nota de crédito #NC3633,Nota de crédito #NC3633",35518848655,0,NaN,NaN,NaN,NaT


In [163]:
df['Columna1']=df['Pagado Con:'].str.split(',')

In [164]:
df_expanded = df.explode('Columna1', ignore_index=True)

In [165]:
df_expanded[['Tipo', 'Número']] = df_expanded['Columna1'].str.extract(r'(.+?) #(.+)', expand=True)

In [166]:
df_expanded['Columna1'] = df_expanded['Columna1'].astype(str).str.strip()
df_expanded= df_expanded[df_expanded['Columna1'] != '']
df = df.dropna(subset=['Columna1'])

In [167]:
cf=df.copy()

In [168]:
c="""
SELECT
    b.name AS lote,
    k.name AS cc,
    SUM(l.foreignAmount) AS valor
FROM
    transaction t
    INNER JOIN transactionline l
        ON l.transaction = t.id
    LEFT JOIN CUSTOMRECORD_CSEG1 k
        ON l.cseg1 = k.id
    LEFT JOIN CUSTOMRECORD_CSEG2 b
        ON l.cseg2 = b.id
WHERE
    t.type = 'PurchOrd'
    AND NVL(t.voided, 'F') = 'F'
    AND NVL(t.void, 'F') = 'F'
    AND NVL(l.isClosed, 'F') = 'F'
    AND NVL(l.mainLine, 'F') = 'F'
    AND NVL(l.taxLine, 'F') = 'F'
    AND NVL(l.doNotDisplayLine, 'F') = 'F'
    AND l.cseg2 IS NOT NULL
    AND l.cseg1 IS NOT NULL
    AND NVL(l.foreignAmount, 0) <> 0
GROUP BY
    b.name,
    k.name
ORDER BY
    b.name,
    k.name
    """

In [169]:
cursor.execute(c)
nombres_columnas = [column[0] for column in cursor.description]
rows = cursor.fetchall()
resultados_tuplas = [tuple(row) for row in rows]
df = pd.DataFrame(resultados_tuplas, columns=nombres_columnas)
#df['createdfrom'] = df['createdfrom'].fillna(0).astype(int)
c=df.copy()
c

,lote,cc,valor
0,100003,MATERIAL VOLUMETRICO,1170000.0
1,100009,MATERIAL VOLUMETRICO,200000.0
2,100030,RETIE CONVENIOS,672268.5
3,100049,RETILAP BOGOTA,45000.0
4,100097,SUBCONTRATADOS LMAM,2030000.0
...,...,...,...
15235,99919,RETIE CONVENIOS,160000.0
15236,99923,RETIE CONVENIOS,160000.0
15237,99936,MATERIAL VOLUMETRICO,240000.0
15238,99948,RETIE CONVENIOS,450000.0


## Validacion de cargue

In [170]:

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

required_dfs = {
    "f": "Facturas",
    "p": "Proformas",
    "nc": "Notas crédito",
    "ncp": "NC / diarios de anulación de proformas",
    "cpf": "Cartera proformas",
    "pisos": "Pisos y metas",
    "cf": "Cartera / pagos",
}

missing = [name for name in required_dfs if name not in globals()]
if missing:
    raise NameError(f"Faltan dataframes en memoria: {missing}")

facturas_raw = f.copy()
proformas_raw = p.copy()
nc_raw = nc.copy()
ncp_raw = ncp.copy()
cpf_raw = cpf.copy()
pisos_raw = pisos.copy()
cartera_raw = cf.copy()

print("Dataframes base cargados:")
for name, label in required_dfs.items():
    df_ = globals()[name]
    print(f"- {label}: {df_.shape[0]} filas x {df_.shape[1]} columnas")


Dataframes base cargados:
- Facturas: 80315 filas x 20 columnas
- Proformas: 11117 filas x 20 columnas
- Notas crédito: 8027 filas x 20 columnas
- NC / diarios de anulación de proformas: 812 filas x 7 columnas
- Cartera proformas: 9486 filas x 15 columnas
- Pisos y metas: 2172 filas x 14 columnas
- Cartera / pagos: 61539 filas x 22 columnas


# Normalización de datos

In [171]:
# =========================================================
# CELDA 2 - NORMALIZACION BASE
# =========================================================


def norm_text(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip()

def norm_upper(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip().str.upper()

def norm_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def norm_date(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce")

def norm_id(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    return s.fillna("0")

# Rehacer copias limpias desde los dataframes originales
facturas_raw = f.copy()
proformas_raw = p.copy()
nc_raw = nc.copy()
ncp_raw = ncp.copy()
cpf_raw = cpf.copy()
pisos_raw = pisos.copy()
cartera_raw = cf.copy()

# =========================
# Normalización ventas / NC
# =========================
for df_name, df_ in [("facturas_raw", facturas_raw), ("proformas_raw", proformas_raw), ("nc_raw", nc_raw)]:
    for col in ["tranId", "type", "status", "entityid", "Servicio", "cc", "memo"]:
        if col in df_.columns:
            df_[col] = norm_text(df_[col])

    for col in ["tranDate", "dueDate"]:
        if col in df_.columns:
            df_[col] = norm_date(df_[col])

    for col in ["Valor", "Precio", "quantity"]:
        if col in df_.columns:
            df_[col] = norm_num(df_[col])

    for col in ["id", "createdfrom", "custitem_sm_lineacomision", "custbody_sm_reemplaza_a", "custbody_sm_factura_reemplazo"]:
        if col in df_.columns:
            df_[col] = norm_id(df_[col])

    if "lote" in df_.columns:
        df_["lote"] = norm_text(df_["lote"]).fillna("")

    if "custbody_ks_sm_proforma" in df_.columns:
        df_["custbody_ks_sm_proforma"] = df_["custbody_ks_sm_proforma"].fillna(False).astype(bool)

# =========================
# Normalización NCP
# =========================
for col in ["tranId", "memo"]:
    if col in ncp_raw.columns:
        ncp_raw[col] = norm_text(ncp_raw[col])

if "tranDate" in ncp_raw.columns:
    ncp_raw["tranDate"] = norm_date(ncp_raw["tranDate"])

# =========================
# Normalización CPF / cartera proformas
# =========================
for col in cpf_raw.columns:
    if cpf_raw[col].dtype == "object":
        cpf_raw[col] = norm_text(cpf_raw[col])

for col in ["Fecha Real Pago", "Fecha Contabilizado Pago", "Fecha de Contabilizacion de Pago", "Fecha de Emision", "Fecha de Vencimiento"]:
    if col in cpf_raw.columns:
        cpf_raw[col] = norm_date(cpf_raw[col])

# =========================
# Normalización cartera / pagos
# =========================
for col in cartera_raw.columns:
    if cartera_raw[col].dtype == "object":
        cartera_raw[col] = norm_text(cartera_raw[col])

for col in ["Fecha Real Pago", "Fecha Contabilizado Pago", "Fecha de pago"]:
    if col in cartera_raw.columns:
        cartera_raw[col] = norm_date(cartera_raw[col])

for col in ["Saldo", "Pagado"]:
    if col in cartera_raw.columns:
        cartera_raw[col] = norm_num(cartera_raw[col])

# =========================
# Normalización pisos
# =========================
for col in pisos_raw.columns:
    if pisos_raw[col].dtype == "object":
        pisos_raw[col] = norm_text(pisos_raw[col])

for col in ["meta", "piso", "comision_ejecutivo", "comision_jefe", "comision_gerente"]:
    if col in pisos_raw.columns:
        pisos_raw[col] = norm_num(pisos_raw[col])

print("Normalización base completada.")
print("Facturas:", facturas_raw[["tranId", "id", "createdfrom", "custbody_sm_factura_reemplazo", "tranDate", "Valor"]].head(2).to_dict("records"))
print("Proformas:", proformas_raw[["tranId", "id", "createdfrom", "custbody_sm_factura_reemplazo", "tranDate", "Valor"]].head(2).to_dict("records"))
print("NC:", nc_raw[["tranId", "id", "createdfrom", "custbody_sm_factura_reemplazo", "tranDate", "Valor"]].head(2).to_dict("records"))

Normalización base completada.
Facturas: [{'tranId': 'SM24994', 'id': '560420', 'createdfrom': '0', 'custbody_sm_factura_reemplazo': '0', 'tranDate': Timestamp('2022-05-20 00:00:00'), 'Valor': 2447200.0}, {'tranId': 'SM27260', 'id': '643570', 'createdfrom': '640051', 'custbody_sm_factura_reemplazo': '0', 'tranDate': Timestamp('2022-08-03 00:00:00'), 'Valor': 8702660.0}]
Proformas: [{'tranId': 'PR-0553', 'id': '61111', 'createdfrom': '30783', 'custbody_sm_factura_reemplazo': '0', 'tranDate': Timestamp('2021-01-15 00:00:00'), 'Valor': 688150.0}, {'tranId': 'PR-0553', 'id': '61111', 'createdfrom': '30783', 'custbody_sm_factura_reemplazo': '0', 'tranDate': Timestamp('2021-01-15 00:00:00'), 'Valor': 688150.0}]
NC: [{'tranId': 'NC1285', 'id': '622635', 'createdfrom': '593570', 'custbody_sm_factura_reemplazo': '622631', 'tranDate': Timestamp('2022-07-15 00:00:00'), 'Valor': -6975000.0}, {'tranId': 'NC3013', 'id': '1436492', 'createdfrom': '1396488', 'custbody_sm_factura_reemplazo': '0', 'tran

C:\Users\gloria.cabrera\AppData\Local\Temp\ipykernel_42192\2240628880.py:16: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(s, errors="coerce")
C:\Users\gloria.cabrera\AppData\Local\Temp\ipykernel_42192\2240628880.py:16: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(s, errors="coerce")
C:\Users\gloria.cabrera\AppData\Local\Temp\ipykernel_42192\2240628880.py:16: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(s, errors="coerce")


In [172]:
# =========================================================
# CELDA 3 - CONSTRUCCION DE VENTAS_BASE
# =========================================================

facturas_base = facturas_raw.copy()
proformas_base = proformas_raw.copy()

facturas_base["origen_documento"] = "factura"
proformas_base["origen_documento"] = "proforma"

ventas_base = pd.concat(
    [facturas_base, proformas_base],
    ignore_index=True,
    sort=False
)

ventas_base["tranId"] = norm_text(ventas_base["tranId"])
ventas_base["type"] = norm_text(ventas_base["type"])
ventas_base["status"] = norm_text(ventas_base["status"])
ventas_base["entityid"] = norm_text(ventas_base["entityid"])
ventas_base["Servicio"] = norm_text(ventas_base["Servicio"])
ventas_base["cc"] = norm_text(ventas_base["cc"])
ventas_base["memo"] = norm_text(ventas_base["memo"])

for col in ["id", "createdfrom", "custitem_sm_lineacomision", "custbody_sm_reemplaza_a", "custbody_sm_factura_reemplazo"]:
    if col in ventas_base.columns:
        ventas_base[col] = norm_id(ventas_base[col])

ventas_base["tranDate"] = norm_date(ventas_base["tranDate"])
ventas_base["dueDate"] = norm_date(ventas_base["dueDate"])
ventas_base["Valor"] = norm_num(ventas_base["Valor"])
ventas_base["Precio"] = norm_num(ventas_base["Precio"])
ventas_base["quantity"] = norm_num(ventas_base["quantity"])

if "lote" in ventas_base.columns:
    ventas_base["lote"] = norm_text(ventas_base["lote"]).fillna("")

# Este campo lo conservamos, pero no lo usamos como criterio principal
if "custbody_ks_sm_proforma" in ventas_base.columns:
    ventas_base["custbody_ks_sm_proforma"] = ventas_base["custbody_ks_sm_proforma"].fillna(False).astype(bool)
else:
    ventas_base["custbody_ks_sm_proforma"] = False

# Regla robusta: como ventas_base viene de dos fuentes separadas,
# usamos origen_documento como verdad principal
ventas_base["es_factura"] = ventas_base["origen_documento"].eq("factura")
ventas_base["es_proforma"] = ventas_base["origen_documento"].eq("proforma")

ventas_base["periodo_factura"] = ventas_base["tranDate"].dt.to_period("M")

ventas_base["Servicio_key"] = norm_upper(ventas_base["Servicio"])
ventas_base["lote_key"] = norm_upper(ventas_base["lote"])
ventas_base["valor_abs"] = ventas_base["Valor"].abs().round(2)

ventas_base["createdfrom_key"] = norm_id(ventas_base["createdfrom"])
ventas_base["id_key"] = norm_id(ventas_base["id"])
ventas_base["reemplaza_a_key"] = norm_id(ventas_base["custbody_sm_reemplaza_a"])
ventas_base["factura_reemplazo_key"] = norm_id(ventas_base["custbody_sm_factura_reemplazo"])

cols_preview = [
    "tranId", "type", "origen_documento", "es_factura", "es_proforma",
    "id_key", "createdfrom_key", "reemplaza_a_key", "factura_reemplazo_key",
    "Servicio_key", "lote_key", "valor_abs", "periodo_factura"
]

cols_preview = [c for c in cols_preview if c in ventas_base.columns]

print("ventas_base creada.")
print("Filas:", len(ventas_base))
print("\nResumen origen_documento vs banderas:")
display(
    ventas_base.groupby(["origen_documento", "es_factura", "es_proforma"], dropna=False)
    .size()
    .reset_index(name="filas")
)

display(ventas_base[cols_preview].head(10))



ventas_base creada.
Filas: 91432

Resumen origen_documento vs banderas:


,origen_documento,es_factura,es_proforma,filas
0,factura,True,False,80315
1,proforma,False,True,11117


,tranId,type,origen_documento,es_factura,es_proforma,id_key,createdfrom_key,reemplaza_a_key,factura_reemplazo_key,Servicio_key,lote_key,valor_abs,periodo_factura
0,SM24994,CustInvc,factura,True,False,560420,0,0,0,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,114427,2447200.0,2022-05
1,SM27260,CustInvc,factura,True,False,643570,640051,0,0,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,110651,8702660.0,2022-08
2,SM27887,CustInvc,factura,True,False,664951,0,0,0,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,111412,5100020.0,2022-08
3,SM28758,CustInvc,factura,True,False,695890,693459,0,0,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,123008,5600000.0,2022-09
4,SM32030,CustInvc,factura,True,False,799023,639914,0,0,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,122878,2700000.0,2022-12
5,SM32337,CustInvc,factura,True,False,814499,0,0,0,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,114369,3864000.0,2023-01
6,SM33455,CustInvc,factura,True,False,855856,855296,0,0,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,136496,10660000.0,2023-02
7,SM35975,CustInvc,factura,True,False,946391,872719,0,0,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,131780,5600000.0,2023-05
8,SM35451,CustInvc,factura,True,False,929429,426616,0,0,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,112329,2200000.0,2023-04
9,SM36972,CustInvc,factura,True,False,985674,780870,0,0,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,133422,13600000.0,2023-06


In [173]:
# =========================================================
# CELDA 4 - CONSTRUCCION DE VENTAS_OUT
# =========================================================

def join_unique(series: pd.Series):
    vals = []
    for v in series.dropna().astype(str):
        v = v.strip()
        if v and v not in {"0", "nan", "<NA>"}:
            vals.append(v)
    vals = sorted(set(vals))
    return " + ".join(vals) if vals else pd.NA

ventas_work = ventas_base.copy()
ventas_work["__rowid"] = np.arange(len(ventas_work))

nc_work = nc_raw.copy()
nc_work["createdfrom_key"] = norm_id(nc_work["createdfrom"])
nc_work["Servicio_key"] = norm_upper(nc_work["Servicio"])

if "lote" in nc_work.columns:
    nc_work["lote_key"] = norm_upper(nc_work["lote"]).fillna("")
else:
    nc_work["lote_key"] = ""

nc_work["factura_reemplazo_key_nc"] = norm_id(nc_work["custbody_sm_factura_reemplazo"])
nc_work.loc[nc_work["factura_reemplazo_key_nc"].eq("0"), "factura_reemplazo_key_nc"] = pd.NA

id_factura_a_tranid = (
    ventas_work.loc[ventas_work["es_factura"], ["id_key", "tranId"]]
    .dropna(subset=["id_key", "tranId"])
    .drop_duplicates(subset=["id_key"], keep="first")
    .set_index("id_key")["tranId"]
    .to_dict()
)

nc_work["factura_reemplazo_tranId_nc"] = nc_work["factura_reemplazo_key_nc"].map(id_factura_a_tranid)

ventas_fact = ventas_work.loc[ventas_work["es_factura"]].copy()

# =========================
# 1) Match base por factura + servicio
# =========================
nc_counts = (
    nc_work.groupby(["createdfrom_key", "Servicio_key"], as_index=False)
    .agg(nc_count=("tranId", "nunique"))
)

nc_join = nc_work[
    [
        "createdfrom_key",
        "Servicio_key",
        "lote_key",
        "tranId",
        "tranDate",
        "Valor",
        "factura_reemplazo_key_nc",
        "factura_reemplazo_tranId_nc",
    ]
].rename(
    columns={
        "tranId": "tranId_nc",
        "tranDate": "tranDate_nc",
        "Valor": "Valor_nc",
    }
)

j = (
    ventas_fact[
        [
            "__rowid",
            "id_key",
            "Servicio_key",
            "lote_key",
            "Valor",
            "valor_abs",
            "tranId",
        ]
    ]
    .rename(columns={"lote_key": "lote_venta_key"})
    .merge(
        nc_join,
        left_on=["id_key", "Servicio_key"],
        right_on=["createdfrom_key", "Servicio_key"],
        how="left",
    )
    .merge(
        nc_counts,
        left_on=["id_key", "Servicio_key"],
        right_on=["createdfrom_key", "Servicio_key"],
        how="left",
    )
)

mask_unica = (j["nc_count"] == 1) & j["Valor_nc"].notna()
mask_multiple_valor = (
    (j["nc_count"] > 1) &
    j["Valor_nc"].notna() &
    (j["valor_abs"].round(2) == j["Valor_nc"].abs().round(2))
)

j["match_valido"] = mask_unica | mask_multiple_valor
j["tipo_match_nc"] = np.select(
    [
        mask_unica,
        mask_multiple_valor,
    ],
    [
        "servicio_unica",
        "servicio_valor",
    ],
    default=pd.NA,
)

j_valid = (
    j.loc[j["match_valido"]]
    .sort_values(["__rowid", "tranDate_nc"])
    .drop_duplicates("__rowid", keep="last")
)

# =========================
# 2) Fallback por factura + servicio + lote
# =========================
asignados = set(j_valid["__rowid"])

pend = ventas_fact[~ventas_fact["__rowid"].isin(asignados)].copy()
pend = pend.rename(columns={"lote_key": "lote_venta_key"})

nc_agg = (
    nc_work.loc[nc_work["lote_key"].fillna("").ne("")]
    .groupby(["createdfrom_key", "Servicio_key", "lote_key"], as_index=False)
    .agg(
        Valor_nc=("Valor", "sum"),
        tranId_nc=("tranId", join_unique),
        tranDate_nc=("tranDate", "max"),
        factura_reemplazo_key_nc=("factura_reemplazo_key_nc", join_unique),
        factura_reemplazo_tranId_nc=("factura_reemplazo_tranId_nc", join_unique),
    )
    .rename(columns={"lote_key": "lote_nc_key"})
)

cand = pend[
    ["__rowid", "id_key", "Servicio_key", "lote_venta_key", "valor_abs"]
].merge(
    nc_agg,
    left_on=["id_key", "Servicio_key", "lote_venta_key"],
    right_on=["createdfrom_key", "Servicio_key", "lote_nc_key"],
    how="inner",
)

cand = cand[cand["valor_abs"].round(2) >= cand["Valor_nc"].abs().round(2)].copy()
cand["abs_val_venta"] = cand["valor_abs"]

pick = (
    cand.sort_values(
        ["id_key", "Servicio_key", "lote_venta_key", "abs_val_venta", "tranDate_nc"],
        ascending=[True, True, True, False, True]
    )
    .drop_duplicates(["id_key", "Servicio_key", "lote_venta_key"], keep="first")
)

asig_lote = pick[
    [
        "__rowid",
        "Valor_nc",
        "tranId_nc",
        "tranDate_nc",
        "factura_reemplazo_key_nc",
        "factura_reemplazo_tranId_nc",
    ]
].copy()

asig_lote["tipo_match_nc"] = "servicio_lote_agregado"

# =========================
# 3) Consolidar asignaciones
# =========================
asign_cols = [
    "__rowid",
    "Valor_nc",
    "tranId_nc",
    "tranDate_nc",
    "factura_reemplazo_key_nc",
    "factura_reemplazo_tranId_nc",
    "tipo_match_nc",
]

j_final = (
    pd.concat(
        [j_valid[asign_cols], asig_lote[asign_cols]],
        ignore_index=True,
    )
    .drop_duplicates("__rowid", keep="first")
)

# =========================
# 4) Llevar a ventas
# =========================
ventas_out = ventas_work.merge(
    j_final,
    on="__rowid",
    how="left",
    validate="1:1",
)

hay_nc_del_serv = (
    j.groupby("__rowid")["tranId_nc"]
    .apply(lambda s: s.notna().any())
    .reset_index()
    .rename(columns={"tranId_nc": "hay_nc_del_serv"})
)

nc_count_por_linea = (
    j[["__rowid", "nc_count"]]
    .drop_duplicates(subset=["__rowid"])
)

ventas_out = ventas_out.merge(hay_nc_del_serv, on="__rowid", how="left", validate="1:1")
ventas_out = ventas_out.merge(nc_count_por_linea, on="__rowid", how="left", validate="1:1")

ventas_out["hay_nc_del_serv"] = ventas_out["hay_nc_del_serv"].fillna(False)
ventas_out["nc_count"] = ventas_out["nc_count"].fillna(0).astype("int64")

ventas_out["alarma_nc_servicio"] = (
    ventas_out["es_factura"] &
    ventas_out["hay_nc_del_serv"] &
    ventas_out["Valor_nc"].isna()
)

ventas_out["motivo_alarma_nc"] = np.select(
    [
        ventas_out["alarma_nc_servicio"] & (ventas_out["nc_count"] > 1),
        ventas_out["alarma_nc_servicio"] & (ventas_out["nc_count"] == 1),
    ],
    [
        "multiples_nc_sin_match_por_valor_o_lote",
        "unica_nc_sin_asignar",
    ],
    default="",
)

print("ventas_out creada.")
print(f"Filas totales: {len(ventas_out)}")
print(f"Filas factura con NC asignada: {ventas_out.loc[ventas_out['es_factura'] & ventas_out['tranId_nc'].notna()].shape[0]}")
print(f"Filas con alarma NC servicio: {int(ventas_out['alarma_nc_servicio'].sum())}")

cols_preview = [
    "tranId",
    "id_key",
    "Servicio_key",
    "lote_key",
    "Valor",
    "tranId_nc",
    "tranDate_nc",
    "Valor_nc",
    "factura_reemplazo_key_nc",
    "factura_reemplazo_tranId_nc",
    "tipo_match_nc",
    "hay_nc_del_serv",
    "nc_count",
    "alarma_nc_servicio",
    "motivo_alarma_nc",
]

cols_preview = [c for c in cols_preview if c in ventas_out.columns]

display(
    ventas_out.loc[
        ventas_out["es_factura"] & (ventas_out["tranId_nc"].notna() | ventas_out["alarma_nc_servicio"]),
        cols_preview
    ].head(15)
)


ventas_out creada.
Filas totales: 91432
Filas factura con NC asignada: 7709
Filas con alarma NC servicio: 7


,tranId,id_key,Servicio_key,lote_key,Valor,tranId_nc,tranDate_nc,Valor_nc,factura_reemplazo_key_nc,factura_reemplazo_tranId_nc,tipo_match_nc,hay_nc_del_serv,nc_count,alarma_nc_servicio,motivo_alarma_nc
12,SM39879,1094563,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,142518,5674860.0,NC2316,2023-09-18,-5674860.0,<NA>,NaN,servicio_unica,True,1,False,
13,SM40216,1107170,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,148160,5600000.0,NC2341,2023-09-29,-5600000.0,1107148,SM40214,servicio_unica,True,1,False,
17,SM45250,1311086,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,126184,15180000.0,NC2859,2024-04-29,-15180000.0,<NA>,NaN,servicio_unica,True,1,False,
18,SM46805,1377796,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,135696,5200000.0,NC2999,2024-06-17,-5200000.0,<NA>,NaN,servicio_unica,True,1,False,
24,SM48850,1455085,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,148471,7700000.0,NC3060,2024-07-08,-7700000.0,1458670,SM48900,servicio_unica,True,1,False,
25,SM48568,1443837,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,153871,9504000.0,NC3174,2024-07-25,-9504000.0,1489322,SM49675,servicio_unica,True,1,False,
26,SM48681,1447901,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,159538,10000000.0,NC3240,2024-08-13,-10000000.0,<NA>,NaN,servicio_unica,True,1,False,
39,SM57719,1806610,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,176027,6500000.0,NC4950,2026-03-25,-6500000.0,<NA>,NaN,servicio_unica,True,1,False,
45,SM59717,1858922,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,183745,3025210.0,NC4286,2025-07-03,-3025210.0,<NA>,NaN,servicio_unica,True,1,False,
52,SM65390,1985395,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,182651,4000000.0,NC4729,2025-12-12,-4000000.0,<NA>,NaN,servicio_unica,True,1,False,


In [174]:
# =========================================================
# CELDA 5 - ORIGEN Y REEMPLAZOS
# =========================================================

ventas_origen = ventas_out.copy()

# =========================
# 1) Mapas básicos de documentos
# =========================
docs_base = (
    ventas_origen[
        [
            "id_key",
            "tranId",
            "tranDate",
            "es_factura",
            "es_proforma",
            "Servicio_key",
            "lote_key",
            "valor_abs",
        ]
    ]
    .dropna(subset=["id_key", "tranId"])
    .sort_values(["id_key", "tranDate", "tranId"])
    .drop_duplicates(subset=["id_key"], keep="first")
    .copy()
)

id_to_tranid = docs_base.set_index("id_key")["tranId"].to_dict()
id_to_trandate = docs_base.set_index("id_key")["tranDate"].to_dict()

proforma_ids = set(docs_base.loc[docs_base["es_proforma"], "id_key"])
factura_ids = set(docs_base.loc[docs_base["es_factura"], "id_key"])

# =========================
# 2) Inicializar columnas de salida
# =========================
ventas_origen["documento_origen"] = pd.NA
ventas_origen["fecha_origen"] = pd.NaT
ventas_origen["documento_reemplazado"] = pd.NA
ventas_origen["tipo_reemplazo"] = "ninguno"
ventas_origen["reemplazo"] = False
ventas_origen["observacion_reemplazo"] = ""

# =========================
# 3) Reemplazo de factura por campo custbody_sm_reemplaza_a
# =========================
mask_reemplaza_factura = (
    ventas_origen["es_factura"] &
    ventas_origen["reemplaza_a_key"].notna() &
    ventas_origen["reemplaza_a_key"].ne("0") &
    ventas_origen["reemplaza_a_key"].isin(factura_ids)
)

ventas_origen.loc[mask_reemplaza_factura, "documento_origen"] = (
    ventas_origen.loc[mask_reemplaza_factura, "reemplaza_a_key"].map(id_to_tranid)
)
ventas_origen.loc[mask_reemplaza_factura, "fecha_origen"] = pd.to_datetime(
    ventas_origen.loc[mask_reemplaza_factura, "reemplaza_a_key"].map(id_to_trandate),
    errors="coerce",
)
ventas_origen.loc[mask_reemplaza_factura, "documento_reemplazado"] = (
    ventas_origen.loc[mask_reemplaza_factura, "reemplaza_a_key"].map(id_to_tranid)
)
ventas_origen.loc[mask_reemplaza_factura, "tipo_reemplazo"] = "reemplazo_factura_campo"
ventas_origen.loc[mask_reemplaza_factura, "reemplazo"] = True

# =========================
# 4) Reemplazo de factura indicado en la NC
#    La NC dice cuál es la factura reemplazo
# =========================
nc_rep_pairs = (
    nc_work.loc[
        nc_work["factura_reemplazo_key_nc"].notna() &
        nc_work["factura_reemplazo_key_nc"].ne("0"),
        ["factura_reemplazo_key_nc", "createdfrom_key", "factura_reemplazo_tranId_nc"]
    ]
    .drop_duplicates()
    .copy()
)

map_nc_rep_id_to_origin_id = (
    nc_rep_pairs
    .drop_duplicates(subset=["factura_reemplazo_key_nc"], keep="first")
    .set_index("factura_reemplazo_key_nc")["createdfrom_key"]
    .to_dict()
)

map_nc_rep_tranid_to_origin_id = (
    nc_rep_pairs.loc[
        nc_rep_pairs["factura_reemplazo_tranId_nc"].notna()
    ]
    .drop_duplicates(subset=["factura_reemplazo_tranId_nc"], keep="first")
    .set_index("factura_reemplazo_tranId_nc")["createdfrom_key"]
    .to_dict()
)

mask_reemplazo_por_nc_id = (
    ventas_origen["es_factura"] &
    ventas_origen["id_key"].isin(map_nc_rep_id_to_origin_id.keys()) &
    ~ventas_origen["reemplazo"]
)

origin_id_from_nc_id = ventas_origen.loc[mask_reemplazo_por_nc_id, "id_key"].map(map_nc_rep_id_to_origin_id)

ventas_origen.loc[mask_reemplazo_por_nc_id, "documento_origen"] = origin_id_from_nc_id.map(id_to_tranid)
ventas_origen.loc[mask_reemplazo_por_nc_id, "fecha_origen"] = pd.to_datetime(
    origin_id_from_nc_id.map(id_to_trandate),
    errors="coerce",
)
ventas_origen.loc[mask_reemplazo_por_nc_id, "documento_reemplazado"] = origin_id_from_nc_id.map(id_to_tranid)
ventas_origen.loc[mask_reemplazo_por_nc_id, "tipo_reemplazo"] = "reemplazo_factura_nc"
ventas_origen.loc[mask_reemplazo_por_nc_id, "reemplazo"] = True

mask_reemplazo_por_nc_tranid = (
    ventas_origen["es_factura"] &
    ventas_origen["tranId"].isin(map_nc_rep_tranid_to_origin_id.keys()) &
    ~ventas_origen["reemplazo"]
)

origin_id_from_nc_tranid = ventas_origen.loc[mask_reemplazo_por_nc_tranid, "tranId"].map(map_nc_rep_tranid_to_origin_id)

ventas_origen.loc[mask_reemplazo_por_nc_tranid, "documento_origen"] = origin_id_from_nc_tranid.map(id_to_tranid)
ventas_origen.loc[mask_reemplazo_por_nc_tranid, "fecha_origen"] = pd.to_datetime(
    origin_id_from_nc_tranid.map(id_to_trandate),
    errors="coerce",
)
ventas_origen.loc[mask_reemplazo_por_nc_tranid, "documento_reemplazado"] = origin_id_from_nc_tranid.map(id_to_tranid)
ventas_origen.loc[mask_reemplazo_por_nc_tranid, "tipo_reemplazo"] = "reemplazo_factura_nc"
ventas_origen.loc[mask_reemplazo_por_nc_tranid, "reemplazo"] = True

# =========================
# 5) Reemplazo de factura por createdfrom cuando apunta a factura
# =========================
mask_reemplazo_factura_createdfrom = (
    ventas_origen["es_factura"] &
    ventas_origen["createdfrom_key"].notna() &
    ventas_origen["createdfrom_key"].ne("0") &
    ventas_origen["createdfrom_key"].isin(factura_ids) &
    ~ventas_origen["reemplazo"]
)

ventas_origen.loc[mask_reemplazo_factura_createdfrom, "documento_origen"] = (
    ventas_origen.loc[mask_reemplazo_factura_createdfrom, "createdfrom_key"].map(id_to_tranid)
)
ventas_origen.loc[mask_reemplazo_factura_createdfrom, "fecha_origen"] = pd.to_datetime(
    ventas_origen.loc[mask_reemplazo_factura_createdfrom, "createdfrom_key"].map(id_to_trandate),
    errors="coerce",
)
ventas_origen.loc[mask_reemplazo_factura_createdfrom, "documento_reemplazado"] = (
    ventas_origen.loc[mask_reemplazo_factura_createdfrom, "createdfrom_key"].map(id_to_tranid)
)
ventas_origen.loc[mask_reemplazo_factura_createdfrom, "tipo_reemplazo"] = "reemplazo_factura_createdfrom"
ventas_origen.loc[mask_reemplazo_factura_createdfrom, "reemplazo"] = True

# =========================
# 6) Factura que proviene de proforma por createdfrom
# =========================
mask_reemplazo_proforma = (
    ventas_origen["es_factura"] &
    ventas_origen["createdfrom_key"].notna() &
    ventas_origen["createdfrom_key"].ne("0") &
    ventas_origen["createdfrom_key"].isin(proforma_ids) &
    ~ventas_origen["reemplazo"]
)

ventas_origen.loc[mask_reemplazo_proforma, "documento_origen"] = (
    ventas_origen.loc[mask_reemplazo_proforma, "createdfrom_key"].map(id_to_tranid)
)
ventas_origen.loc[mask_reemplazo_proforma, "fecha_origen"] = pd.to_datetime(
    ventas_origen.loc[mask_reemplazo_proforma, "createdfrom_key"].map(id_to_trandate),
    errors="coerce",
)
ventas_origen.loc[mask_reemplazo_proforma, "documento_reemplazado"] = (
    ventas_origen.loc[mask_reemplazo_proforma, "createdfrom_key"].map(id_to_tranid)
)
ventas_origen.loc[mask_reemplazo_proforma, "tipo_reemplazo"] = "reemplazo_proforma_createdfrom"
ventas_origen.loc[mask_reemplazo_proforma, "reemplazo"] = True

# =========================
# 7) Fallback: mismo servicio + lote + valor_abs
#    Solo entre documentos distintos
# =========================
fallback_base = ventas_origen.loc[
    (~ventas_origen["reemplazo"]) &
    ventas_origen["es_factura"],
    [
        "id_key",
        "tranId",
        "tranDate",
        "Servicio_key",
        "lote_key",
        "valor_abs",
    ]
].copy()

fallback_base = fallback_base.loc[
    fallback_base["Servicio_key"].notna() &
    fallback_base["lote_key"].notna() &
    fallback_base["valor_abs"].notna()
].copy()

# Muy importante:
# quitar duplicados a nivel documento dentro de la misma combinación,
# para que varias líneas de la misma factura no se marquen entre sí.
fallback_docs = (
    fallback_base
    .drop_duplicates(
        subset=["id_key", "tranId", "Servicio_key", "lote_key", "valor_abs"],
        keep="first",
    )
    .sort_values(["Servicio_key", "lote_key", "valor_abs", "tranDate", "tranId"])
    .copy()
)

fallback_docs["rank_fallback"] = (
    fallback_docs.groupby(["Servicio_key", "lote_key", "valor_abs"])
    .cumcount() + 1
)

fallback_original = (
    fallback_docs.loc[
        fallback_docs["rank_fallback"] == 1,
        ["Servicio_key", "lote_key", "valor_abs", "id_key", "tranId", "tranDate"]
    ]
    .rename(columns={
        "id_key": "id_key_original",
        "tranId": "tranId_original",
        "tranDate": "fecha_original",
    })
)

fallback_rep = (
    fallback_docs.loc[fallback_docs["rank_fallback"] > 1]
    .merge(
        fallback_original,
        on=["Servicio_key", "lote_key", "valor_abs"],
        how="left",
        validate="m:1",
    )
)

# Blindaje extra: nunca permitir autorreferencia
fallback_rep = fallback_rep.loc[
    fallback_rep["id_key"] != fallback_rep["id_key_original"]
].copy()

fallback_rep_ids = set(fallback_rep["id_key"])

mask_fallback = ventas_origen["id_key"].isin(fallback_rep_ids) & ~ventas_origen["reemplazo"]

fallback_map_doc = (
    fallback_rep
    .drop_duplicates(subset=["id_key"], keep="first")
    .set_index("id_key")["tranId_original"]
    .to_dict()
)

fallback_map_fecha = (
    fallback_rep
    .drop_duplicates(subset=["id_key"], keep="first")
    .set_index("id_key")["fecha_original"]
    .to_dict()
)

ventas_origen.loc[mask_fallback, "documento_origen"] = (
    ventas_origen.loc[mask_fallback, "id_key"].map(fallback_map_doc)
)
ventas_origen.loc[mask_fallback, "fecha_origen"] = pd.to_datetime(
    ventas_origen.loc[mask_fallback, "id_key"].map(fallback_map_fecha),
    errors="coerce",
)
ventas_origen.loc[mask_fallback, "documento_reemplazado"] = (
    ventas_origen.loc[mask_fallback, "id_key"].map(fallback_map_doc)
)
ventas_origen.loc[mask_fallback, "tipo_reemplazo"] = "reemplazo_fallback"
ventas_origen.loc[mask_fallback, "reemplazo"] = True

# Blindaje final general: si por cualquier ruta quedó autoreferencia, se limpia
mask_self_replace = (
    ventas_origen["reemplazo"].fillna(False)
    & ventas_origen["documento_origen"].notna()
    & ventas_origen["tranId"].astype("string").str.strip().eq(
        ventas_origen["documento_origen"].astype("string").str.strip()
    )
)

ventas_origen.loc[mask_self_replace, "documento_origen"] = pd.NA
ventas_origen.loc[mask_self_replace, "fecha_origen"] = pd.NaT
ventas_origen.loc[mask_self_replace, "documento_reemplazado"] = pd.NA
ventas_origen.loc[mask_self_replace, "tipo_reemplazo"] = "ninguno"
ventas_origen.loc[mask_self_replace, "reemplazo"] = False

# =========================
# 8) Observacion reemplazo
# =========================
ventas_origen["observacion_reemplazo"] = np.where(
    ventas_origen["reemplazo"],
    "No suma por ser reemplazo de " + ventas_origen["documento_origen"].astype("string"),
    ""
)

print("Origen y reemplazos resueltos.")
display(
    ventas_origen["tipo_reemplazo"]
    .value_counts(dropna=False)
    .rename_axis("tipo_reemplazo")
    .reset_index(name="filas")
)

cols_preview = [
    "tranId",
    "type",
    "id_key",
    "createdfrom_key",
    "reemplaza_a_key",
    "factura_reemplazo_key_nc",
    "factura_reemplazo_tranId_nc",
    "documento_origen",
    "fecha_origen",
    "tipo_reemplazo",
    "reemplazo",
    "observacion_reemplazo",
]

cols_preview = [c for c in cols_preview if c in ventas_origen.columns]

display(
    ventas_origen.loc[
        ventas_origen["reemplazo"],
        cols_preview
    ].head(25)
)


Origen y reemplazos resueltos.


,tipo_reemplazo,filas
0,ninguno,71014
1,reemplazo_proforma_createdfrom,9757
2,reemplazo_fallback,6357
3,reemplazo_factura_nc,4288
4,reemplazo_factura_campo,16


,tranId,type,id_key,createdfrom_key,reemplaza_a_key,factura_reemplazo_key_nc,factura_reemplazo_tranId_nc,documento_origen,fecha_origen,tipo_reemplazo,reemplazo,observacion_reemplazo
0,SM24994,CustInvc,560420,0,0,<NA>,NaN,SM24025,2022-04-18,reemplazo_fallback,True,No suma por ser reemplazo de SM24025
4,SM32030,CustInvc,799023,639914,0,<NA>,NaN,PR-5378,2022-10-19,reemplazo_proforma_createdfrom,True,No suma por ser reemplazo de PR-5378
5,SM32337,CustInvc,814499,0,0,<NA>,NaN,SM27742,2022-08-17,reemplazo_fallback,True,No suma por ser reemplazo de SM27742
7,SM35975,CustInvc,946391,872719,0,<NA>,NaN,SM33123,2023-02-06,reemplazo_fallback,True,No suma por ser reemplazo de SM33123
8,SM35451,CustInvc,929429,426616,0,<NA>,NaN,PR-3456,2022-02-03,reemplazo_proforma_createdfrom,True,No suma por ser reemplazo de PR-3456
9,SM36972,CustInvc,985674,780870,0,<NA>,NaN,PR-5671,2022-11-30,reemplazo_proforma_createdfrom,True,No suma por ser reemplazo de PR-5671
13,SM40216,CustInvc,1107170,1099548,0,1107148,SM40214,SM40211,2023-09-27,reemplazo_fallback,True,No suma por ser reemplazo de SM40211
18,SM46805,CustInvc,1377796,855298,0,<NA>,NaN,PR-6037,2023-02-17,reemplazo_proforma_createdfrom,True,No suma por ser reemplazo de PR-6037
19,SM46798,CustInvc,1377774,1294282,0,<NA>,NaN,PR-7702,2024-03-01,reemplazo_proforma_createdfrom,True,No suma por ser reemplazo de PR-7702
21,SM46804,CustInvc,1377793,855882,0,<NA>,NaN,PR-6033,2023-02-26,reemplazo_proforma_createdfrom,True,No suma por ser reemplazo de PR-6033


In [175]:
# =========================================================
# CELDA 6 - AFECTACION NC Y ANULACIONES
# =========================================================

ventas_afect = ventas_origen.copy()

# =========================
# 1) Base fechas / columnas
# =========================
ventas_afect["tranDate"] = norm_date(ventas_afect["tranDate"])
ventas_afect["tranDate_nc"] = norm_date(ventas_afect["tranDate_nc"])
ventas_afect["Valor_nc"] = norm_num(ventas_afect["Valor_nc"])

ventas_afect["periodo_factura"] = ventas_afect["tranDate"].dt.to_period("M")
ventas_afect["periodo_nc"] = ventas_afect["tranDate_nc"].dt.to_period("M")

ventas_afect["documento_anulado"] = False
ventas_afect["tipo_afectacion_nc"] = "ninguna"
ventas_afect["requiere_recalculo_piso"] = False
ventas_afect["periodo_recalculo"] = pd.NA
ventas_afect["observacion_nc"] = ""

# =========================
# 2) Facturas con NC
# =========================
mask_factura_con_nc = (
    ventas_afect["es_factura"] &
    ventas_afect["tranId_nc"].notna()
)

mask_factura_nc_mismo_mes = (
    mask_factura_con_nc &
    ventas_afect["periodo_factura"].eq(ventas_afect["periodo_nc"])
)

mask_factura_nc_otro_mes = (
    mask_factura_con_nc &
    ~ventas_afect["periodo_factura"].eq(ventas_afect["periodo_nc"])
)

mask_factura_nc_otro_mes_con_reemplazo = (
    mask_factura_nc_otro_mes &
    ventas_afect["factura_reemplazo_tranId_nc"].notna()
)

mask_factura_nc_otro_mes_sin_reemplazo = (
    mask_factura_nc_otro_mes &
    ventas_afect["factura_reemplazo_tranId_nc"].isna()
)

ventas_afect.loc[mask_factura_nc_mismo_mes, "tipo_afectacion_nc"] = "factura_nc_mismo_mes"

ventas_afect.loc[mask_factura_nc_otro_mes_con_reemplazo, "tipo_afectacion_nc"] = "factura_nc_otro_mes_con_reemplazo"
ventas_afect.loc[mask_factura_nc_otro_mes_con_reemplazo, "observacion_nc"] = (
    "Suma en su mes original; NC " +
    ventas_afect.loc[mask_factura_nc_otro_mes_con_reemplazo, "tranId_nc"].astype("string") +
    " reemplazada por " +
    ventas_afect.loc[mask_factura_nc_otro_mes_con_reemplazo, "factura_reemplazo_tranId_nc"].astype("string")
)

ventas_afect.loc[mask_factura_nc_otro_mes_sin_reemplazo, "tipo_afectacion_nc"] = "factura_nc_otro_mes_sin_reemplazo"
ventas_afect.loc[mask_factura_nc_otro_mes_sin_reemplazo, "requiere_recalculo_piso"] = True
ventas_afect.loc[mask_factura_nc_otro_mes_sin_reemplazo, "periodo_recalculo"] = (
    ventas_afect.loc[mask_factura_nc_otro_mes_sin_reemplazo, "periodo_factura"].astype("string")
)
ventas_afect.loc[mask_factura_nc_otro_mes_sin_reemplazo, "observacion_nc"] = (
    "Suma en su mes original; recalcular piso de " +
    ventas_afect.loc[mask_factura_nc_otro_mes_sin_reemplazo, "periodo_factura"].astype("string") +
    " por " +
    ventas_afect.loc[mask_factura_nc_otro_mes_sin_reemplazo, "tranId_nc"].astype("string") +
    " sin reemplazo"
)

# =========================
# 3) Preparar anulaciones de proformas desde NCP
# =========================
ncp_work = ncp_raw.copy()

for col in ["tranId", "memo"]:
    if col in ncp_work.columns:
        ncp_work[col] = norm_text(ncp_work[col])

if "tranDate" in ncp_work.columns:
    ncp_work["tranDate"] = norm_date(ncp_work["tranDate"])

# Intentar detectar documento afectado
if "documentos_detectados" in ncp_work.columns:
    ncp_work["doc_afectado"] = norm_upper(ncp_work["documentos_detectados"])
else:
    memo_src = ncp_work["memo"] if "memo" in ncp_work.columns else pd.Series("", index=ncp_work.index, dtype="string")
    ncp_work["doc_afectado"] = memo_src.astype("string").str.extract(r"(PR-\d+|SM\d+)", expand=False)
    ncp_work["doc_afectado"] = norm_upper(ncp_work["doc_afectado"])

ncp_work["tranId"] = norm_text(ncp_work["tranId"]) if "tranId" in ncp_work.columns else pd.Series(pd.NA, index=ncp_work.index, dtype="string")

ncp_docs = (
    ncp_work.loc[
        ncp_work["doc_afectado"].notna(),
        ["doc_afectado", "tranId", "tranDate"]
    ]
    .sort_values(["doc_afectado", "tranDate", "tranId"])
    .drop_duplicates(subset=["doc_afectado"], keep="first")
    .rename(columns={
        "doc_afectado": "tranId_key",
        "tranId": "tranId_nc_ncp",
        "tranDate": "tranDate_nc_ncp",
    })
)

# =========================
# 4) Preparar anulaciones de proformas desde CPF
# =========================
cpf_work = cpf_raw.copy()

col_doc = next((c for c in ["Numero de Proforma", "Factura"] if c in cpf_work.columns), None)
col_pagado_con = next((c for c in ["Pagado Con (Tipo)", "Pagado Con"] if c in cpf_work.columns), None)
col_fecha = next((c for c in ["Fecha Contabilizado Pago", "Fecha de Contabilizacion de Pago"] if c in cpf_work.columns), None)

if col_doc and col_pagado_con:
    cpf_work[col_doc] = norm_upper(cpf_work[col_doc])
    cpf_work[col_pagado_con] = norm_text(cpf_work[col_pagado_con])

    if col_fecha:
        cpf_work[col_fecha] = norm_date(cpf_work[col_fecha])

    mask_nc_cpf = cpf_work[col_pagado_con].str.contains(r"(?i)NC-\d+", na=False)

    cpf_nc = cpf_work.loc[mask_nc_cpf, [col_doc, col_pagado_con] + ([col_fecha] if col_fecha else [])].copy()
    cpf_nc["tranId_nc_cpf"] = cpf_nc[col_pagado_con].astype("string").str.extract(r"(?i)(NC-\d+)", expand=False)

    if col_fecha:
        cpf_nc["tranDate_nc_cpf"] = cpf_nc[col_fecha]
    else:
        cpf_nc["tranDate_nc_cpf"] = pd.NaT

    cpf_docs = (
        cpf_nc.sort_values([col_doc, "tranDate_nc_cpf", "tranId_nc_cpf"])
        .drop_duplicates(subset=[col_doc], keep="first")
        .rename(columns={col_doc: "tranId_key"})
        [["tranId_key", "tranId_nc_cpf", "tranDate_nc_cpf"]]
    )
else:
    cpf_docs = pd.DataFrame(columns=["tranId_key", "tranId_nc_cpf", "tranDate_nc_cpf"])

# =========================
# 5) Aplicar anulaciones a proformas
# =========================
ventas_afect["tranId_key"] = norm_upper(ventas_afect["tranId"])

ventas_afect = ventas_afect.merge(
    ncp_docs,
    on="tranId_key",
    how="left",
    validate="m:1",
)

ventas_afect = ventas_afect.merge(
    cpf_docs,
    on="tranId_key",
    how="left",
    validate="m:1",
)

mask_proforma_anulada_ncp = (
    ventas_afect["es_proforma"] &
    ventas_afect["tranId_nc_ncp"].notna()
)

mask_proforma_anulada_cpf = (
    ventas_afect["es_proforma"] &
    ventas_afect["tranId_nc_cpf"].notna() &
    ~mask_proforma_anulada_ncp
)

# Completar columnas estándar de NC en proformas anuladas
ventas_afect.loc[mask_proforma_anulada_ncp, "tranId_nc"] = ventas_afect.loc[mask_proforma_anulada_ncp, "tranId_nc_ncp"]
ventas_afect.loc[mask_proforma_anulada_ncp, "tranDate_nc"] = ventas_afect.loc[mask_proforma_anulada_ncp, "tranDate_nc_ncp"]

ventas_afect.loc[mask_proforma_anulada_cpf, "tranId_nc"] = ventas_afect.loc[mask_proforma_anulada_cpf, "tranId_nc_cpf"]
ventas_afect.loc[mask_proforma_anulada_cpf, "tranDate_nc"] = ventas_afect.loc[mask_proforma_anulada_cpf, "tranDate_nc_cpf"]

mask_proforma_anulada = mask_proforma_anulada_ncp | mask_proforma_anulada_cpf

ventas_afect.loc[mask_proforma_anulada, "documento_anulado"] = True
ventas_afect.loc[mask_proforma_anulada, "Valor_nc"] = -ventas_afect.loc[mask_proforma_anulada, "Valor"].abs()
ventas_afect.loc[mask_proforma_anulada, "periodo_nc"] = ventas_afect.loc[mask_proforma_anulada, "tranDate_nc"].dt.to_period("M")

mask_proforma_mismo_mes = (
    mask_proforma_anulada &
    ventas_afect["periodo_factura"].eq(ventas_afect["periodo_nc"])
)

mask_proforma_otro_mes = (
    mask_proforma_anulada &
    ~ventas_afect["periodo_factura"].eq(ventas_afect["periodo_nc"])
)

ventas_afect.loc[mask_proforma_mismo_mes, "tipo_afectacion_nc"] = "proforma_anulada_mismo_mes"

ventas_afect.loc[mask_proforma_otro_mes, "tipo_afectacion_nc"] = "proforma_anulada_otro_mes"
ventas_afect.loc[mask_proforma_otro_mes, "requiere_recalculo_piso"] = True
ventas_afect.loc[mask_proforma_otro_mes, "periodo_recalculo"] = (
    ventas_afect.loc[mask_proforma_otro_mes, "periodo_factura"].astype("string")
)
ventas_afect.loc[mask_proforma_otro_mes, "observacion_nc"] = (
    "Suma en su mes original; recalcular piso de " +
    ventas_afect.loc[mask_proforma_otro_mes, "periodo_factura"].astype("string") +
    " por anulación " +
    ventas_afect.loc[mask_proforma_otro_mes, "tranId_nc"].astype("string")
)

print("Afectacion NC y anulaciones resueltas.")

display(
    ventas_afect["tipo_afectacion_nc"]
    .value_counts(dropna=False)
    .rename_axis("tipo_afectacion_nc")
    .reset_index(name="filas")
)

cols_preview = [
    "tranId",
    "es_factura",
    "es_proforma",
    "reemplazo",
    "tranId_nc",
    "tranDate_nc",
    "Valor_nc",
    "factura_reemplazo_tranId_nc",
    "documento_anulado",
    "tipo_afectacion_nc",
    "requiere_recalculo_piso",
    "periodo_recalculo",
    "observacion_nc",
]

cols_preview = [c for c in cols_preview if c in ventas_afect.columns]

display(
    ventas_afect.loc[
        (ventas_afect["tipo_afectacion_nc"] != "ninguna") |
        (ventas_afect["documento_anulado"]),
        cols_preview
    ].head(25)
)


Afectacion NC y anulaciones resueltas.


,tipo_afectacion_nc,filas
0,ninguna,82594
1,factura_nc_otro_mes_con_reemplazo,3185
2,factura_nc_mismo_mes,2386
3,factura_nc_otro_mes_sin_reemplazo,2138
4,proforma_anulada_otro_mes,937
5,proforma_anulada_mismo_mes,192


,tranId,es_factura,es_proforma,reemplazo,tranId_nc,tranDate_nc,Valor_nc,factura_reemplazo_tranId_nc,documento_anulado,tipo_afectacion_nc,requiere_recalculo_piso,periodo_recalculo,observacion_nc
12,SM39879,True,False,False,NC2316,2023-09-18,-5674860.00,NaN,False,factura_nc_mismo_mes,False,<NA>,
13,SM40216,True,False,True,NC2341,2023-09-29,-5600000.00,SM40214,False,factura_nc_mismo_mes,False,<NA>,
17,SM45250,True,False,False,NC2859,2024-04-29,-15180000.00,NaN,False,factura_nc_otro_mes_sin_reemplazo,True,2024-03,Suma en su mes original; recalcular piso de 20...
18,SM46805,True,False,True,NC2999,2024-06-17,-5200000.00,NaN,False,factura_nc_otro_mes_sin_reemplazo,True,2024-05,Suma en su mes original; recalcular piso de 20...
24,SM48850,True,False,False,NC3060,2024-07-08,-7700000.00,SM48900,False,factura_nc_mismo_mes,False,<NA>,
25,SM48568,True,False,True,NC3174,2024-07-25,-9504000.00,SM49675,False,factura_nc_otro_mes_con_reemplazo,False,<NA>,Suma en su mes original; NC NC3174 reemplazada...
26,SM48681,True,False,False,NC3240,2024-08-13,-10000000.00,NaN,False,factura_nc_otro_mes_sin_reemplazo,True,2024-06,Suma en su mes original; recalcular piso de 20...
39,SM57719,True,False,False,NC4950,2026-03-25,-6500000.00,NaN,False,factura_nc_otro_mes_sin_reemplazo,True,2025-04,Suma en su mes original; recalcular piso de 20...
45,SM59717,True,False,False,NC4286,2025-07-03,-3025210.00,NaN,False,factura_nc_otro_mes_sin_reemplazo,True,2025-06,Suma en su mes original; recalcular piso de 20...
52,SM65390,True,False,False,NC4729,2025-12-12,-4000000.00,NaN,False,factura_nc_mismo_mes,False,<NA>,


In [176]:
# =========================================================
# CELDA 7 - ESTADO PISO
# =========================================================

ventas_piso = ventas_afect.copy()

ventas_piso["Valor"] = norm_num(ventas_piso["Valor"])
ventas_piso["TotalParaPiso_base"] = ventas_piso["Valor"]
ventas_piso["estado_piso"] = "suma"
ventas_piso["TotalParaPiso_final"] = ventas_piso["TotalParaPiso_base"]
ventas_piso["observacion_piso"] = ""

# =========================
# 1) Condiciones
# =========================
cond_reemplazo = ventas_piso["reemplazo"].fillna(False)

cond_nc_mismo_mes = ventas_piso["tipo_afectacion_nc"].isin([
    "factura_nc_mismo_mes",
    "proforma_anulada_mismo_mes",
])

cond_revision = ventas_piso["alarma_nc_servicio"].fillna(False)

cond_otro_mes = ventas_piso["tipo_afectacion_nc"].isin([
    "factura_nc_otro_mes_con_reemplazo",
    "factura_nc_otro_mes_sin_reemplazo",
    "proforma_anulada_otro_mes",
])

# =========================
# 2) Estado piso con prioridad
# =========================
ventas_piso.loc[cond_reemplazo, "estado_piso"] = "no suma"
ventas_piso.loc[~cond_reemplazo & cond_nc_mismo_mes, "estado_piso"] = "no suma"
ventas_piso.loc[~cond_reemplazo & ~cond_nc_mismo_mes & cond_revision, "estado_piso"] = "para revision"
ventas_piso.loc[ventas_piso["estado_piso"].eq("no suma"), "TotalParaPiso_final"] = 0
ventas_piso.loc[ventas_piso["estado_piso"].eq("para revision"), "TotalParaPiso_final"] = np.nan

# =========================
# 3) Observaciones
# =========================
mask_rep = cond_reemplazo
ventas_piso.loc[mask_rep, "observacion_piso"] = ventas_piso.loc[mask_rep, "observacion_reemplazo"]

mask_fact_nc_mismo = (
    ~cond_reemplazo &
    ventas_piso["tipo_afectacion_nc"].eq("factura_nc_mismo_mes")
)
ventas_piso.loc[mask_fact_nc_mismo, "observacion_piso"] = (
    "No suma por nota crédito " +
    ventas_piso.loc[mask_fact_nc_mismo, "tranId_nc"].astype("string") +
    " del mismo periodo"
)

mask_prof_mismo = (
    ~cond_reemplazo &
    ventas_piso["tipo_afectacion_nc"].eq("proforma_anulada_mismo_mes")
)
ventas_piso.loc[mask_prof_mismo, "observacion_piso"] = (
    "No suma por anulación " +
    ventas_piso.loc[mask_prof_mismo, "tranId_nc"].astype("string") +
    " del mismo periodo"
)

mask_revision = (
    ~cond_reemplazo &
    ~cond_nc_mismo_mes &
    cond_revision
)
ventas_piso.loc[mask_revision, "observacion_piso"] = (
    "Para revisión: " +
    ventas_piso.loc[mask_revision, "motivo_alarma_nc"].astype("string")
)

mask_otro_mes = (
    ventas_piso["estado_piso"].eq("suma") &
    cond_otro_mes
)
ventas_piso.loc[mask_otro_mes, "observacion_piso"] = (
    ventas_piso.loc[mask_otro_mes, "observacion_nc"]
)

mask_default = ventas_piso["observacion_piso"].astype("string").fillna("").eq("")
ventas_piso.loc[mask_default, "observacion_piso"] = "Suma para piso"

print("Estado piso calculado.")

display(
    ventas_piso["estado_piso"]
    .value_counts(dropna=False)
    .rename_axis("estado_piso")
    .reset_index(name="filas")
)

cols_preview = [
    "tranId",
    "es_factura",
    "es_proforma",
    "documento_origen",
    "reemplazo",
    "tipo_reemplazo",
    "tranId_nc",
    "tipo_afectacion_nc",
    "requiere_recalculo_piso",
    "periodo_recalculo",
    "estado_piso",
    "TotalParaPiso_base",
    "TotalParaPiso_final",
    "observacion_piso",
]

cols_preview = [c for c in cols_preview if c in ventas_piso.columns]

display(
    ventas_piso.loc[
        ventas_piso["estado_piso"].ne("suma") | ventas_piso["tipo_afectacion_nc"].ne("ninguna"),
        cols_preview
    ].head(30)
)


Estado piso calculado.


,estado_piso,filas
0,suma,69246
1,no suma,22179
2,para revision,7


,tranId,es_factura,es_proforma,documento_origen,reemplazo,tipo_reemplazo,tranId_nc,tipo_afectacion_nc,requiere_recalculo_piso,periodo_recalculo,estado_piso,TotalParaPiso_base,TotalParaPiso_final,observacion_piso
0,SM24994,True,False,SM24025,True,reemplazo_fallback,<NA>,ninguna,False,<NA>,no suma,2447200.00,0.0,No suma por ser reemplazo de SM24025
4,SM32030,True,False,PR-5378,True,reemplazo_proforma_createdfrom,<NA>,ninguna,False,<NA>,no suma,2700000.00,0.0,No suma por ser reemplazo de PR-5378
5,SM32337,True,False,SM27742,True,reemplazo_fallback,<NA>,ninguna,False,<NA>,no suma,3864000.00,0.0,No suma por ser reemplazo de SM27742
7,SM35975,True,False,SM33123,True,reemplazo_fallback,<NA>,ninguna,False,<NA>,no suma,5600000.00,0.0,No suma por ser reemplazo de SM33123
8,SM35451,True,False,PR-3456,True,reemplazo_proforma_createdfrom,<NA>,ninguna,False,<NA>,no suma,2200000.00,0.0,No suma por ser reemplazo de PR-3456
9,SM36972,True,False,PR-5671,True,reemplazo_proforma_createdfrom,<NA>,ninguna,False,<NA>,no suma,13600000.00,0.0,No suma por ser reemplazo de PR-5671
12,SM39879,True,False,<NA>,False,ninguno,NC2316,factura_nc_mismo_mes,False,<NA>,no suma,5674860.00,0.0,No suma por nota crédito NC2316 del mismo periodo
13,SM40216,True,False,SM40211,True,reemplazo_fallback,NC2341,factura_nc_mismo_mes,False,<NA>,no suma,5600000.00,0.0,No suma por ser reemplazo de SM40211
17,SM45250,True,False,<NA>,False,ninguno,NC2859,factura_nc_otro_mes_sin_reemplazo,True,2024-03,suma,15180000.00,15180000.0,Suma en su mes original; recalcular piso de 20...
18,SM46805,True,False,PR-6037,True,reemplazo_proforma_createdfrom,NC2999,factura_nc_otro_mes_sin_reemplazo,True,2024-05,no suma,5200000.00,0.0,No suma por ser reemplazo de PR-6037


In [177]:
# =========================================================
# CELDA 8 - REGLAS DE CUMPLIMIENTO DE PISOS
# costo queda solo informativo; no entra en cálculos
# =========================================================

if "ventas_piso" not in globals():
    raise NameError("No existe 'ventas_piso'. Ejecuta primero la CELDA 7 - ESTADO PISO.")

if "pisos_raw" not in globals():
    raise NameError("No existe 'pisos_raw'. Ejecuta primero la CELDA 2 - NORMALIZACION BASE.")

if "costo" not in globals():
    raise NameError("No existe el DataFrame 'costo'. Cárgalo antes de ejecutar esta celda.")


def _to_period_month(s: pd.Series) -> pd.Series:
    if str(s.dtype).startswith("period"):
        return s.astype("period[M]")
    return pd.to_datetime(s, errors="coerce").dt.to_period("M")


# =========================
# 0) Mapa de líneas
# =========================
relation_data = {
    "ID INTERNO": [14, 13, 7, 8, 18, 9, 17, 16, 2, 1, 3, 6, 5, 19, 12, 10, 11, 15, 4],
    "NOMBRE": [
        "ADMON", "CALIBRACION", "CAPACITACION", "CERTIFICACION", "CSG",
        "ESCAS", "FRONTERAS", "INTER", "LMA", "LMAM", "LME", "LVM",
        "LVV", "NA", "PRUEBAS", "RETIE/LAP", "RIESGO", "RITEL",
        "VARIABLES ELECTRICAS"
    ],
}

df_relation = pd.DataFrame(relation_data)
df_relation["id_linea_key"] = norm_id(df_relation["ID INTERNO"])
df_relation["linea_comision_key"] = norm_upper(df_relation["NOMBRE"])


# =========================
# 1) Base de ventas para piso
# =========================
cumpl_base = ventas_piso.copy()

if "periodo_factura" not in cumpl_base.columns:
    cumpl_base["periodo_factura"] = norm_date(cumpl_base["tranDate"]).dt.to_period("M")

cumpl_base["periodo_piso"] = _to_period_month(cumpl_base["periodo_factura"])
cumpl_base["ejecutivo_key"] = norm_upper(cumpl_base["entityid"])
cumpl_base["custitem_sm_lineacomision_key"] = norm_id(cumpl_base["custitem_sm_lineacomision"])
cumpl_base["TotalParaPiso_final"] = norm_num(cumpl_base["TotalParaPiso_final"])
cumpl_base["lote_key"] = norm_text(cumpl_base["lote"]).fillna("")

cumpl_base = cumpl_base.merge(
    df_relation[["id_linea_key", "linea_comision_key"]].rename(
        columns={"id_linea_key": "custitem_sm_lineacomision_key"}
    ),
    on="custitem_sm_lineacomision_key",
    how="left",
    validate="m:1",
)

cumpl_base["tiene_revision_origen"] = (
    cumpl_base["estado_piso"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("para revision")
)


# =========================
# 1A) Cruce de costo por lote
# costo solo informativo
# =========================
costos_work = costo.copy()
costos_work.columns = [str(c).strip() for c in costos_work.columns]

if "lote" not in costos_work.columns:
    raise KeyError(f"No existe la columna 'lote' en costo. Columnas: {costos_work.columns.tolist()}")

if "Valor" not in costos_work.columns:
    raise KeyError(f"No existe la columna 'Valor' en costo. Columnas: {costos_work.columns.tolist()}")

costos_work["lote_key"] = norm_text(costos_work["lote"]).fillna("")
costos_work["costo"] = norm_num(costos_work["Valor"]).abs().fillna(0)

costos_work = (
    costos_work.groupby("lote_key", as_index=False)
    .agg(costo=("costo", "sum"))
)


# =========================
# 2) Base de pisos
# =========================
pisos_work = pisos_raw.copy()

pisos_work["periodo_piso"] = norm_date(pisos_work["perido"]).dt.to_period("M")
pisos_work["ejecutivo_key"] = norm_upper(pisos_work["ejecutivo"])
pisos_work["linea_comision_key"] = norm_upper(pisos_work["linea_comision"])
pisos_work["meta"] = norm_num(pisos_work["meta"])
pisos_work["piso"] = norm_num(pisos_work["piso"])


# =========================
# 3) Validación de duplicados en pisos
# =========================
dup_pisos = (
    pisos_work[
        ["periodo_piso", "ejecutivo_key", "linea_comision_key"]
    ]
    .dropna()
    .groupby(["periodo_piso", "ejecutivo_key", "linea_comision_key"], as_index=False)
    .size()
    .rename(columns={"size": "n"})
)

dup_pisos = dup_pisos[dup_pisos["n"] > 1].copy()

if not dup_pisos.empty:
    raise ValueError(
        "Hay duplicados en pisos por periodo+ejecutivo+linea.\n"
        + dup_pisos.head(20).to_string(index=False)
    )


# =========================
# 4) Agregar ventas por periodo + ejecutivo + línea
# costo se suma una sola vez por lote dentro del grupo,
# pero NO se usa en las reglas; queda solo indicativo
# =========================
periodos_pisos = set(pisos_work["periodo_piso"].dropna().tolist())

cumpl_base["periodo_tiene_config"] = cumpl_base["periodo_piso"].isin(periodos_pisos)
cumpl_base["linea_tiene_mapeo"] = cumpl_base["linea_comision_key"].notna()

group_keys = [
    "periodo_piso",
    "entityid",
    "ejecutivo_key",
    "linea_comision_key",
    "periodo_tiene_config",
    "linea_tiene_mapeo",
]

ventas_sum = (
    cumpl_base.groupby(
        group_keys,
        dropna=False,
        as_index=False,
    )
    .agg(
        TotalParaPiso=("TotalParaPiso_final", "sum"),
        tiene_revision_origen=("tiene_revision_origen", "max"),
    )
)

costos_sum = (
    cumpl_base[group_keys + ["lote_key"]]
    .drop_duplicates(subset=group_keys + ["lote_key"], keep="first")
    .merge(
        costos_work,
        on="lote_key",
        how="left",
        validate="m:1",
    )
    .groupby(group_keys, dropna=False, as_index=False)
    .agg(costo=("costo", "sum"))
)

v_pivot = ventas_sum.merge(
    costos_sum,
    on=group_keys,
    how="left",
    validate="1:1",
)

v_pivot["costo"] = norm_num(v_pivot["costo"]).fillna(0)


# =========================
# 5) Cruce con pisos
# =========================
df_merged = v_pivot.merge(
    pisos_work[
        [
            "periodo_piso",
            "ejecutivo_key",
            "linea_comision_key",
            "meta",
            "piso",
        ]
    ],
    on=["periodo_piso", "ejecutivo_key", "linea_comision_key"],
    how="left",
    validate="m:1",
)

for c in ["TotalParaPiso", "meta", "piso", "costo"]:
    df_merged[c] = norm_num(df_merged[c])

df_merged["costo"] = df_merged["costo"].fillna(0)

df_merged["cumple"] = False
df_merged["observacion_cumple"] = "No cumple piso ni meta"
df_merged["requiere_revision_cumplimiento"] = False
df_merged["motivo_revision_cumplimiento"] = ""


# =========================
# 6) Marcar casos para revisión
# =========================
mask_periodo_sin_config = ~df_merged["periodo_tiene_config"].fillna(False)
mask_linea_sin_mapeo = ~df_merged["linea_tiene_mapeo"].fillna(False)
mask_revision_origen = df_merged["tiene_revision_origen"].fillna(False)

df_merged.loc[mask_periodo_sin_config, "requiere_revision_cumplimiento"] = True
df_merged.loc[mask_periodo_sin_config, "motivo_revision_cumplimiento"] = (
    "Sin configuración de piso para el período"
)
df_merged.loc[mask_periodo_sin_config, "observacion_cumple"] = (
    "Para revisión: sin configuración de piso para el período"
)

df_merged.loc[
    ~mask_periodo_sin_config & mask_linea_sin_mapeo,
    "requiere_revision_cumplimiento"
] = True
df_merged.loc[
    ~mask_periodo_sin_config & mask_linea_sin_mapeo,
    "motivo_revision_cumplimiento"
] = "Sin mapeo de línea de comisión"
df_merged.loc[
    ~mask_periodo_sin_config & mask_linea_sin_mapeo,
    "observacion_cumple"
] = "Para revisión: sin mapeo de línea de comisión"

df_merged.loc[
    ~mask_periodo_sin_config & ~mask_linea_sin_mapeo & mask_revision_origen,
    "requiere_revision_cumplimiento"
] = True
df_merged.loc[
    ~mask_periodo_sin_config & ~mask_linea_sin_mapeo & mask_revision_origen,
    "motivo_revision_cumplimiento"
] = "Hay documentos con estado_piso='para revision' dentro de la línea/periodo"
df_merged.loc[
    ~mask_periodo_sin_config & ~mask_linea_sin_mapeo & mask_revision_origen,
    "observacion_cumple"
] = "Para revisión: hay documentos en revisión dentro de la línea/periodo"


# =========================
# 7) Reglas de cumplimiento
# costo NO entra aquí
# =========================
mask_evaluable = ~df_merged["requiere_revision_cumplimiento"].fillna(False)

df_merged["piso_real_max"] = (
    df_merged.groupby(["periodo_piso", "ejecutivo_key"], dropna=False)["piso"]
    .transform(lambda s: s.dropna().max() if s.notna().any() else np.nan)
)

mask_con_piso = mask_evaluable & df_merged["piso"].notna()

cond_meta = mask_con_piso & (
    df_merged["TotalParaPiso"] >= df_merged["meta"]
).fillna(False)

cond_piso = (
    mask_con_piso
    & df_merged["meta"].notna()
    & (df_merged["TotalParaPiso"] < df_merged["meta"])
    & (df_merged["TotalParaPiso"] >= df_merged["piso"])
).fillna(False)

cond_sin_meta = (
    mask_evaluable
    & df_merged["meta"].isna()
    & df_merged["piso"].notna()
    & (df_merged["TotalParaPiso"] >= df_merged["piso"])
).fillna(False)

df_merged.loc[cond_meta | cond_piso | cond_sin_meta, "cumple"] = True
df_merged.loc[cond_meta, "observacion_cumple"] = "Cumple meta"
df_merged.loc[cond_piso | cond_sin_meta, "observacion_cumple"] = "Cumple piso"


# =========================
# 8) Filas sin piso configurado dentro de períodos válidos
# =========================
mask_sin_piso = mask_evaluable & df_merged["piso"].isna()
mask_sin_piso_sin_ref = mask_sin_piso & df_merged["piso_real_max"].isna()
mask_sin_piso_con_ref = mask_sin_piso & df_merged["piso_real_max"].notna()

df_merged.loc[mask_sin_piso_sin_ref, "piso"] = 0
df_merged.loc[mask_sin_piso_sin_ref, "cumple"] = (
    df_merged.loc[mask_sin_piso_sin_ref, "TotalParaPiso"] >= 0
)
df_merged.loc[
    mask_sin_piso_sin_ref,
    "observacion_cumple"
] = "Cumple piso (sin piso configurado para el ejecutivo en ese mes; se asume 0)"

ref_max = (
    df_merged[
        mask_evaluable
        & df_merged["piso"].notna()
        & df_merged["piso"].eq(df_merged["piso_real_max"])
    ]
    .groupby(["periodo_piso", "ejecutivo_key"], as_index=False)["cumple"]
    .max()
    .rename(columns={"cumple": "cumple_piso_max"})
)

df_merged = df_merged.merge(
    ref_max,
    on=["periodo_piso", "ejecutivo_key"],
    how="left",
    validate="m:1",
)

df_merged.loc[mask_sin_piso_con_ref, "cumple"] = (
    df_merged.loc[mask_sin_piso_con_ref, "cumple_piso_max"].fillna(False)
)

df_merged.loc[
    mask_sin_piso_con_ref & df_merged["cumple_piso_max"].fillna(False),
    "observacion_cumple"
] = "Cumple por arrastre del mayor piso del ejecutivo en el mes"

df_merged.loc[
    mask_sin_piso_con_ref & ~df_merged["cumple_piso_max"].fillna(False),
    "observacion_cumple"
] = "No cumple: no alcanzó el mayor piso configurado del ejecutivo en el mes"


# =========================
# 9) Regla especial RETIE/LAP
# =========================
retie_gate = (
    df_merged[
        mask_evaluable
        & df_merged["linea_comision_key"].eq("RETIE/LAP")
    ][["periodo_piso", "ejecutivo_key", "meta", "cumple"]]
    .copy()
)

retie_gate["tiene_meta_retie"] = retie_gate["meta"].notna()

retie_gate = (
    retie_gate.groupby(["periodo_piso", "ejecutivo_key"], as_index=False)
    .agg(
        tiene_meta_retie=("tiene_meta_retie", "max"),
        cumple_retie=("cumple", "max"),
    )
)

df_merged = df_merged.merge(
    retie_gate,
    on=["periodo_piso", "ejecutivo_key"],
    how="left",
    validate="m:1",
)

mask_linea_configurada = df_merged["meta"].notna() | df_merged["piso"].notna()

mask_arrastre_retie = (
    mask_evaluable
    & df_merged["tiene_meta_retie"].fillna(False)
    & df_merged["cumple_retie"].fillna(False)
    & df_merged["linea_comision_key"].ne("RETIE/LAP")
    & mask_linea_configurada
)

cumple_previo = df_merged["cumple"].fillna(False).copy()

df_merged.loc[mask_arrastre_retie, "cumple"] = True

mask_arrastre_nuevo = mask_arrastre_retie & ~cumple_previo

df_merged.loc[
    mask_arrastre_nuevo,
    "observacion_cumple"
] = "Cumple por regla RETIE/LAP: al cumplir RETIE/LAP se pagan las demás líneas configuradas del ejecutivo en el mes"


# =========================
# 10) Salida final
# =========================
df_filtrado = df_merged[
    df_merged["entityid"].notna()
].copy()

cumplimiento_final = df_filtrado[
    [
        "periodo_piso",
        "entityid",
        "linea_comision_key",
        "TotalParaPiso",
        "costo",
        "piso",
        "meta",
        "cumple",
        "observacion_cumple",
        "requiere_revision_cumplimiento",
        "motivo_revision_cumplimiento",
    ]
].copy()

cumplimiento_final = cumplimiento_final.rename(
    columns={
        "periodo_piso": "periodo_mes",
        "linea_comision_key": "linea_comision",
    }
)

cumplimiento_final = cumplimiento_final.sort_values(
    ["periodo_mes", "entityid", "linea_comision"],
    na_position="last",
).reset_index(drop=True)


# =========================
# 11) Diagnóstico final
# =========================
print("Filas con costo informativo:")
print(int(df_merged["costo"].fillna(0).gt(0).sum()))

print("\nMuestra cumplimiento_final:")
display(cumplimiento_final.head(20))

Filas con costo informativo:
3708

Muestra cumplimiento_final:


,periodo_mes,entityid,linea_comision,TotalParaPiso,costo,piso,meta,cumple,observacion_cumple,requiere_revision_cumplimiento,motivo_revision_cumplimiento
0,2021-01,ADRIANA MARITZA QUITIAN ROZO,RETIE/LAP,96544500.0,2031683.0,NaN,NaN,False,Para revisión: sin configuración de piso para ...,True,Sin configuración de piso para el período
1,2021-01,ALEJANDRO HERNANDEZ HOYOS,NA,170000.0,0.0,NaN,NaN,False,Para revisión: sin configuración de piso para ...,True,Sin configuración de piso para el período
2,2021-01,ALEJANDRO HERNANDEZ HOYOS,RETIE/LAP,19050000.0,199000.0,NaN,NaN,False,Para revisión: sin configuración de piso para ...,True,Sin configuración de piso para el período
3,2021-01,AYLIN MARCELA MENDOZA RUIZ,ESCAS,10300000.0,0.0,NaN,NaN,False,Para revisión: sin configuración de piso para ...,True,Sin configuración de piso para el período
4,2021-01,CAMILO ANDRES QUIJANO GOMEZ,ESCAS,10690800.0,0.0,NaN,NaN,False,Para revisión: sin configuración de piso para ...,True,Sin configuración de piso para el período
5,2021-01,CARLOS ANDRES MONTOYA VALENCIA,RETIE/LAP,42934365.0,0.0,NaN,NaN,False,Para revisión: sin configuración de piso para ...,True,Sin configuración de piso para el período
6,2021-01,CARLOS EDUARDO BELLO GUZMAN,ESCAS,9185000.0,345000.0,NaN,NaN,False,Para revisión: sin configuración de piso para ...,True,Sin configuración de piso para el período
7,2021-01,CARLOS EDUARDO BELLO GUZMAN,NA,410000.0,345000.0,NaN,NaN,False,Para revisión: sin configuración de piso para ...,True,Sin configuración de piso para el período
8,2021-01,CARLOS FABIAN GONZALEZ GARRIDO,LMAM,34521750.0,354000.0,NaN,NaN,False,Para revisión: sin configuración de piso para ...,True,Sin configuración de piso para el período
9,2021-01,CARLOS FABIAN GONZALEZ GARRIDO,LME,2166800.0,0.0,NaN,NaN,False,Para revisión: sin configuración de piso para ...,True,Sin configuración de piso para el período


In [178]:
# =========================================================
# CELDA 9 - REGLAS DE PAGO Y CORTE CONTABLE
# =========================================================

if "ventas_piso" not in globals():
    raise NameError("No existe 'ventas_piso'. Ejecuta primero la CELDA 7 - ESTADO PISO.")

if "cumplimiento_final" not in globals():
    raise NameError("No existe 'cumplimiento_final'. Ejecuta primero la CELDA 8 - REGLAS DE CUMPLIMIENTO DE PISOS.")

if "cartera_raw" not in globals():
    raise NameError("No existe 'cartera_raw'. Ejecuta primero la CELDA 2 - NORMALIZACION BASE.")

if "pisos_raw" not in globals():
    raise NameError("No existe 'pisos_raw'. Ejecuta primero la CELDA 2 - NORMALIZACION BASE.")


def _to_period_month(s: pd.Series) -> pd.Series:
    if str(s.dtype).startswith("period"):
        return s.astype("period[M]")
    return pd.to_datetime(s, errors="coerce").dt.to_period("M")


def _join_unique(s: pd.Series):
    vals = [str(x).strip() for x in s.dropna().astype(str) if str(x).strip() != ""]
    if not vals:
        return pd.NA
    return " | ".join(dict.fromkeys(vals))


# =========================
# 0) Mapa de líneas
# =========================
relation_data = {
    "ID INTERNO": [14, 13, 7, 8, 18, 9, 17, 16, 2, 1, 3, 6, 5, 19, 12, 10, 11, 15, 4],
    "NOMBRE": [
        "ADMON", "CALIBRACION", "CAPACITACION", "CERTIFICACION", "CSG",
        "ESCAS", "FRONTERAS", "INTER", "LMA", "LMAM", "LME", "LVM",
        "LVV", "NA", "PRUEBAS", "RETIE/LAP", "RIESGO", "RITEL",
        "VARIABLES ELECTRICAS"
    ],
}

df_relation = pd.DataFrame(relation_data)
df_relation["id_linea_key"] = norm_id(df_relation["ID INTERNO"])
df_relation["linea_comision_key"] = norm_upper(df_relation["NOMBRE"])


# =========================
# 1) Base de documentos para pago
# =========================
para_pago_base = ventas_piso.copy()

para_pago_base["tranId"] = norm_text(para_pago_base["tranId"])
para_pago_base["tranId_key"] = norm_upper(para_pago_base["tranId"])
para_pago_base["tranDate"] = norm_date(para_pago_base["tranDate"])
para_pago_base["dueDate"] = norm_date(para_pago_base["dueDate"])

if "periodo_factura" not in para_pago_base.columns:
    para_pago_base["periodo_factura"] = para_pago_base["tranDate"].dt.to_period("M")

para_pago_base["periodo_mes"] = _to_period_month(para_pago_base["periodo_factura"])
para_pago_base["ejecutivo_key"] = norm_upper(para_pago_base["entityid"])
para_pago_base["custitem_sm_lineacomision_key"] = norm_id(para_pago_base["custitem_sm_lineacomision"])
para_pago_base["Valor"] = norm_num(para_pago_base["Valor"])
para_pago_base["TotalParaPiso_final"] = norm_num(para_pago_base["TotalParaPiso_final"])

para_pago_base = para_pago_base.merge(
    df_relation[["id_linea_key", "linea_comision_key"]].rename(
        columns={"id_linea_key": "custitem_sm_lineacomision_key"}
    ),
    on="custitem_sm_lineacomision_key",
    how="left",
    validate="m:1",
)

# Documento base para medir pago:
# solo en reemplazos se usa el documento origen; en los demás casos se usa la factura actual.
para_pago_base["documento_origen"] = norm_text(para_pago_base["documento_origen"])
para_pago_base["doc_base_comision"] = np.where(
    para_pago_base["reemplazo"].fillna(False) & para_pago_base["documento_origen"].notna(),
    para_pago_base["documento_origen"],
    para_pago_base["tranId"],
)
para_pago_base["doc_base_comision"] = norm_text(pd.Series(para_pago_base["doc_base_comision"], index=para_pago_base.index))
para_pago_base["doc_base_comision_key"] = norm_upper(para_pago_base["doc_base_comision"])

lookup_due = (
    para_pago_base[["tranId_key", "dueDate"]]
    .dropna(subset=["tranId_key"])
    .sort_values(["tranId_key", "dueDate"], na_position="last")
    .drop_duplicates(subset=["tranId_key"], keep="first")
    .set_index("tranId_key")["dueDate"]
)

para_pago_base["dueDate_base_pago"] = para_pago_base["doc_base_comision_key"].map(lookup_due)
para_pago_base["dueDate_base_pago"] = para_pago_base["dueDate_base_pago"].combine_first(para_pago_base["dueDate"])


# =========================
# 2) Agregar cartera / pagos por factura
# =========================
cartera_pago = cartera_raw.copy()

if "Factura" not in cartera_pago.columns:
    raise KeyError("No existe la columna 'Factura' en cartera_raw.")

cartera_pago["Factura"] = norm_text(cartera_pago["Factura"])
cartera_pago["Factura_key"] = norm_upper(cartera_pago["Factura"])

for c in ["Fecha Real Pago", "Fecha Contabilizado Pago", "Fecha de pago"]:
    if c in cartera_pago.columns:
        cartera_pago[c] = norm_date(cartera_pago[c])

for c in ["Pagado", "Saldo"]:
    if c in cartera_pago.columns:
        cartera_pago[c] = norm_num(cartera_pago[c])

agg_map = {}

if "Fecha Real Pago" in cartera_pago.columns:
    agg_map["Fecha Real Pago"] = "max"
if "Fecha Contabilizado Pago" in cartera_pago.columns:
    agg_map["Fecha Contabilizado Pago"] = "max"
if "Pagado" in cartera_pago.columns:
    agg_map["Pagado"] = "max"
if "Saldo" in cartera_pago.columns:
    agg_map["Saldo"] = "min"
if "Pagado Con (Tipo)" in cartera_pago.columns:
    agg_map["Pagado Con (Tipo)"] = _join_unique
if "Pagado Con:" in cartera_pago.columns:
    agg_map["Pagado Con:"] = _join_unique

pagos_factura = (
    cartera_pago
    .groupby("Factura_key", as_index=False)
    .agg(agg_map)
)

if "Fecha Real Pago" not in pagos_factura.columns:
    pagos_factura["Fecha Real Pago"] = pd.NaT
if "Fecha Contabilizado Pago" not in pagos_factura.columns:
    pagos_factura["Fecha Contabilizado Pago"] = pd.NaT
if "Pagado" not in pagos_factura.columns:
    pagos_factura["Pagado"] = np.nan
if "Saldo" not in pagos_factura.columns:
    pagos_factura["Saldo"] = np.nan

pagos_factura["Fecha de pago"] = pagos_factura["Fecha Real Pago"].combine_first(
    pagos_factura["Fecha Contabilizado Pago"]
)

para_pago_full = para_pago_base.merge(
    pagos_factura,
    left_on="tranId_key",
    right_on="Factura_key",
    how="left",
    validate="m:1",
)


# =========================
# 3) Cruce con cumplimiento de pisos
# =========================
cumpl_key = cumplimiento_final.copy()

cumpl_key["periodo_mes"] = cumpl_key["periodo_mes"].astype("period[M]")
cumpl_key["ejecutivo_key"] = norm_upper(cumpl_key["entityid"])
cumpl_key["linea_comision_key"] = norm_upper(cumpl_key["linea_comision"])

cumpl_key = cumpl_key[
    [
        "periodo_mes",
        "ejecutivo_key",
        "linea_comision_key",
        "TotalParaPiso",
        "costo",
        "meta",
        "piso",
        "cumple",
        "observacion_cumple",
        "requiere_revision_cumplimiento",
        "motivo_revision_cumplimiento",
    ]
].drop_duplicates(
    subset=["periodo_mes", "ejecutivo_key", "linea_comision_key"],
    keep="last",
)

para_pago_full = para_pago_full.merge(
    cumpl_key,
    on=["periodo_mes", "ejecutivo_key", "linea_comision_key"],
    how="left",
    validate="m:1",
)

para_pago_full["sin_resultado_cumplimiento"] = (
    para_pago_full["cumple"].isna()
    & para_pago_full["observacion_cumple"].isna()
    & para_pago_full["requiere_revision_cumplimiento"].isna()
)


# =========================
# 4) Cruce con configuración de comisiones
# =========================
pisos_cfg = pisos_raw.copy()

pisos_cfg["periodo_mes"] = norm_date(pisos_cfg["perido"]).dt.to_period("M")
pisos_cfg["ejecutivo_key"] = norm_upper(pisos_cfg["ejecutivo"])
pisos_cfg["linea_comision_key"] = norm_upper(pisos_cfg["linea_comision"])

for c in ["comision_ejecutivo", "comision_jefe", "comision_gerente"]:
    if c in pisos_cfg.columns:
        pisos_cfg[c] = norm_num(pisos_cfg[c])

dup_cfg = (
    pisos_cfg[
        ["periodo_mes", "ejecutivo_key", "linea_comision_key"]
    ]
    .dropna()
    .groupby(["periodo_mes", "ejecutivo_key", "linea_comision_key"], as_index=False)
    .size()
    .rename(columns={"size": "n"})
)
dup_cfg = dup_cfg[dup_cfg["n"] > 1].copy()

if not dup_cfg.empty:
    raise ValueError(
        "Hay duplicados en configuración de comisiones por periodo+ejecutivo+linea.\n"
        + dup_cfg.head(20).to_string(index=False)
    )

cfg_cols = [
    c for c in [
        "periodo_mes",
        "ejecutivo_key",
        "linea_comision_key",
        "jefe",
        "comision_ejecutivo",
        "comision_jefe",
        "gerente",
        "comision_gerente",
    ]
    if c in pisos_cfg.columns
]

para_pago_full = para_pago_full.merge(
    pisos_cfg[cfg_cols].drop_duplicates(
        subset=["periodo_mes", "ejecutivo_key", "linea_comision_key"]
    ),
    on=["periodo_mes", "ejecutivo_key", "linea_comision_key"],
    how="left",
    validate="m:1",
)


# =========================
# 5) Tabla real de cortes
# =========================
cortes = pd.DataFrame([
    {"inicio_corte": "2024-12-20", "fin_corte": "2025-01-22", "periodo_corte": "2025-01"},
    {"inicio_corte": "2025-01-23", "fin_corte": "2025-02-21", "periodo_corte": "2025-02"},
    {"inicio_corte": "2025-02-22", "fin_corte": "2025-03-20", "periodo_corte": "2025-03"},
    {"inicio_corte": "2025-03-21", "fin_corte": "2025-04-21", "periodo_corte": "2025-04"},
    {"inicio_corte": "2025-04-22", "fin_corte": "2025-05-20", "periodo_corte": "2025-05"},
    {"inicio_corte": "2025-05-21", "fin_corte": "2025-06-20", "periodo_corte": "2025-06"},
    {"inicio_corte": "2025-06-21", "fin_corte": "2025-07-21", "periodo_corte": "2025-07"},
    {"inicio_corte": "2025-07-22", "fin_corte": "2025-08-21", "periodo_corte": "2025-08"},
    {"inicio_corte": "2025-08-22", "fin_corte": "2025-09-22", "periodo_corte": "2025-09"},
    {"inicio_corte": "2025-09-23", "fin_corte": "2025-10-20", "periodo_corte": "2025-10"},
    {"inicio_corte": "2025-10-21", "fin_corte": "2025-11-20", "periodo_corte": "2025-11"},
    {"inicio_corte": "2025-11-21", "fin_corte": "2025-12-19", "periodo_corte": "2025-12"},
    {"inicio_corte": "2025-12-20", "fin_corte": "2026-01-22", "periodo_corte": "2026-01"},
    {"inicio_corte": "2026-01-23", "fin_corte": "2026-02-20", "periodo_corte": "2026-02"},
    {"inicio_corte": "2026-02-21", "fin_corte": "2026-03-20", "periodo_corte": "2026-03"},
    {"inicio_corte": "2026-03-21", "fin_corte": "2026-04-21", "periodo_corte": "2026-04"},
    {"inicio_corte": "2026-04-22", "fin_corte": "2026-05-21", "periodo_corte": "2026-05"},
    {"inicio_corte": "2026-05-22", "fin_corte": "2026-06-22", "periodo_corte": "2026-06"},
    {"inicio_corte": "2026-06-23", "fin_corte": "2026-07-22", "periodo_corte": "2026-07"},
    {"inicio_corte": "2026-07-23", "fin_corte": "2026-08-21", "periodo_corte": "2026-08"},
    {"inicio_corte": "2026-08-22", "fin_corte": "2026-09-22", "periodo_corte": "2026-09"},
    {"inicio_corte": "2026-09-23", "fin_corte": "2026-10-22", "periodo_corte": "2026-10"},
    {"inicio_corte": "2026-10-23", "fin_corte": "2026-11-23", "periodo_corte": "2026-11"},
    {"inicio_corte": "2026-11-24", "fin_corte": "2026-12-21", "periodo_corte": "2026-12"},
])

cortes["inicio_corte"] = pd.to_datetime(cortes["inicio_corte"], errors="coerce")
cortes["fin_corte"] = pd.to_datetime(cortes["fin_corte"], errors="coerce")
cortes["periodo_corte"] = cortes["periodo_corte"].astype("string")

para_pago_full["Fecha Real Pago"] = norm_date(para_pago_full["Fecha Real Pago"])
para_pago_full["Fecha Contabilizado Pago"] = norm_date(para_pago_full["Fecha Contabilizado Pago"])
para_pago_full["Fecha de pago"] = norm_date(para_pago_full["Fecha de pago"])
para_pago_full["Pagado"] = norm_num(para_pago_full["Pagado"])
para_pago_full["Saldo"] = norm_num(para_pago_full["Saldo"])

para_pago_full["Inicio Corte Contable"] = pd.NaT
para_pago_full["Fin Corte Contable"] = pd.NaT
para_pago_full["Periodo Corte"] = pd.Series(pd.NA, index=para_pago_full.index, dtype="string")

for _, corte in cortes.iterrows():
    mask_corte = para_pago_full["Fecha Contabilizado Pago"].between(
        corte["inicio_corte"],
        corte["fin_corte"],
        inclusive="both",
    )

    para_pago_full.loc[mask_corte, "Inicio Corte Contable"] = corte["inicio_corte"]
    para_pago_full.loc[mask_corte, "Fin Corte Contable"] = corte["fin_corte"]
    para_pago_full.loc[mask_corte, "Periodo Corte"] = corte["periodo_corte"]

para_pago_full["Cumple Corte Contable"] = para_pago_full["Periodo Corte"].notna()


# =========================
# 6) Días y factor de pago
#    El análisis se hace con Fecha Real Pago vs dueDate_base_pago
# =========================
para_pago_full["dias_pago"] = (
    para_pago_full["Fecha Real Pago"] - para_pago_full["dueDate_base_pago"]
).dt.days

para_pago_full["factor"] = np.select(
    [
        para_pago_full["dias_pago"] <= 30,
        (para_pago_full["dias_pago"] > 30) & (para_pago_full["dias_pago"] <= 60),
        para_pago_full["dias_pago"] > 60,
    ],
    [1.0, 0.5, 0.0],
    default=np.nan,
)

para_pago_full["nota_pago"] = np.select(
    [
        para_pago_full["dias_pago"] <= 30,
        (para_pago_full["dias_pago"] > 30) & (para_pago_full["dias_pago"] <= 60),
        para_pago_full["dias_pago"] > 60,
    ],
    [
        "Pago <= 30 días desde dueDate",
        "Pago entre 31 y 60 días desde dueDate",
        "Pago > 60 días desde dueDate",
    ],
    default="Sin cálculo de días de pago",
)


# =========================
# 7) Decisión de pago
#    Prioridad: revisión > no paga > paga
# =========================
def _mask_bool(s):
    return pd.Series(s, index=para_pago_full.index).fillna(False).astype(bool).to_numpy()


estado_piso_key = (
    para_pago_full["estado_piso"]
    .astype("string")
    .str.strip()
    .str.lower()
)

cumple_piso_s = pd.Series(
    para_pago_full["cumple"].fillna(False).astype(bool),
    index=para_pago_full.index,
)

saldo_cero_s = pd.Series(
    norm_num(para_pago_full["Saldo"]).eq(0),
    index=para_pago_full.index,
).fillna(False).astype(bool)

es_proforma_s = pd.Series(
    para_pago_full["es_proforma"].fillna(False).astype(bool),
    index=para_pago_full.index,
)

mask_revision_cumpl = _mask_bool(para_pago_full["requiere_revision_cumplimiento"])
mask_linea_sin_mapeo = _mask_bool(para_pago_full["linea_comision_key"].isna())
mask_sin_cumpl = _mask_bool(para_pago_full["sin_resultado_cumplimiento"])
mask_estado_revision = _mask_bool(estado_piso_key.eq("para revision"))
mask_sin_due_base = _mask_bool(
    para_pago_full["dueDate_base_pago"].isna()
    & (para_pago_full["Fecha Real Pago"].notna() | para_pago_full["Fecha Contabilizado Pago"].notna())
)
mask_sin_fecha_real_con_contab = _mask_bool(
    para_pago_full["Fecha Real Pago"].isna()
    & para_pago_full["Fecha Contabilizado Pago"].notna()
)
mask_sin_fecha_contab_con_real = _mask_bool(
    para_pago_full["Fecha Contabilizado Pago"].isna()
    & para_pago_full["Fecha Real Pago"].notna()
)
mask_fecha_contab_fuera_tabla = _mask_bool(
    para_pago_full["Fecha Contabilizado Pago"].notna()
    & para_pago_full["Periodo Corte"].isna()
)
mask_saldo_cero_sin_fechas = _mask_bool(
    saldo_cero_s
    & para_pago_full["Fecha Real Pago"].isna()
    & para_pago_full["Fecha Contabilizado Pago"].isna()
)
mask_no_suma = _mask_bool(estado_piso_key.eq("no suma"))
mask_no_cumple = _mask_bool(~cumple_piso_s)
mask_sin_pago = _mask_bool(
    para_pago_full["Fecha Real Pago"].isna()
    & para_pago_full["Fecha Contabilizado Pago"].isna()
    & ~saldo_cero_s
)
mask_saldo_abierto = _mask_bool(
    para_pago_full["Saldo"].notna()
    & para_pago_full["Saldo"].ne(0)
)
mask_factor_100 = _mask_bool(para_pago_full["factor"].eq(1.0))
mask_factor_50 = _mask_bool(para_pago_full["factor"].eq(0.5))
mask_factor_0 = _mask_bool(para_pago_full["factor"].eq(0.0))
mask_es_proforma = _mask_bool(es_proforma_s)

para_pago_full["decision_pago"] = np.select(
    [
        mask_revision_cumpl,
        mask_linea_sin_mapeo,
        mask_sin_cumpl,
        mask_estado_revision,
        mask_sin_due_base,
        mask_sin_fecha_real_con_contab,
        mask_sin_fecha_contab_con_real,
        mask_fecha_contab_fuera_tabla,
        mask_saldo_cero_sin_fechas,
        mask_es_proforma,
        mask_no_suma,
        mask_no_cumple,
        mask_sin_pago,
        mask_saldo_abierto,
        mask_factor_100,
        mask_factor_50,
        mask_factor_0,
    ],
    [
        "PARA REVISAR",
        "PARA REVISAR",
        "PARA REVISAR",
        "PARA REVISAR",
        "PARA REVISAR",
        "PARA REVISAR",
        "PARA REVISAR",
        "PARA REVISAR",
        "PARA REVISAR",
        "NO PAGA",
        "NO PAGA",
        "NO PAGA",
        "NO PAGA",
        "NO PAGA",
        "PAGA 100%",
        "PAGA 50%",
        "NO PAGA",
    ],
    default="PARA REVISAR",
)

para_pago_full["motivo_pago"] = np.select(
    [
        mask_revision_cumpl,
        mask_linea_sin_mapeo,
        mask_sin_cumpl,
        mask_estado_revision,
        mask_sin_due_base,
        mask_sin_fecha_real_con_contab,
        mask_sin_fecha_contab_con_real,
        mask_fecha_contab_fuera_tabla,
        mask_saldo_cero_sin_fechas,
        mask_es_proforma,
        mask_no_suma,
        mask_no_cumple,
        mask_sin_pago,
        mask_saldo_abierto,
        mask_factor_100,
        mask_factor_50,
        mask_factor_0,
    ],
    [
        para_pago_full["motivo_revision_cumplimiento"].fillna(
            para_pago_full["observacion_cumple"].fillna("Revisión de cumplimiento")
        ),
        "Sin mapeo de línea de comisión",
        "Sin resultado de cumplimiento para periodo + ejecutivo + línea",
        para_pago_full["observacion_piso"].fillna("Estado piso en revisión"),
        "Sin dueDate base para medir días de pago",
        "Hay Fecha Contabilizado Pago pero falta Fecha Real Pago",
        "Hay Fecha Real Pago pero falta Fecha Contabilizado Pago",
        "La Fecha Contabilizado Pago quedó fuera de la tabla de cortes",
        "Saldo 0 pero sin fechas de pago; validar soporte",
        "Es proforma; no paga comisión",
        para_pago_full["observacion_piso"].fillna("No suma para piso"),
        para_pago_full["observacion_cumple"].fillna("No cumplió piso del mes"),
        "Sin pago registrado",
        "La factura no está totalmente pagada",
        "Cumple pago <= 30 días desde dueDate",
        "Cumple pago entre 31 y 60 días desde dueDate",
        "Pago después de 60 días desde dueDate",
    ],
    default="Validar caso manualmente",
)

para_pago_full["requiere_revision_pago"] = para_pago_full["decision_pago"].eq("PARA REVISAR")


# =========================
# 8) Factor final y valores de comisión
#    Asumo Valor del documento como base de comisión.
# =========================
para_pago_full["factor_comision_final"] = np.select(
    [
        para_pago_full["decision_pago"].eq("PAGA 100%"),
        para_pago_full["decision_pago"].eq("PAGA 50%"),
        para_pago_full["decision_pago"].eq("NO PAGA"),
    ],
    [1.0, 0.5, 0.0],
    default=np.nan,
)

for c in ["comision_ejecutivo", "comision_jefe", "comision_gerente"]:
    if c in para_pago_full.columns:
        para_pago_full[c] = norm_num(para_pago_full[c])

para_pago_full["valor_comision_ejecutivo"] = (
    para_pago_full["Valor"] *
    para_pago_full["comision_ejecutivo"] *
    para_pago_full["factor_comision_final"]
)

if "comision_jefe" in para_pago_full.columns:
    para_pago_full["valor_comision_jefe"] = (
        para_pago_full["Valor"] *
        para_pago_full["comision_jefe"] *
        para_pago_full["factor_comision_final"]
    )

if "comision_gerente" in para_pago_full.columns:
    para_pago_full["valor_comision_gerente"] = (
        para_pago_full["Valor"] *
        para_pago_full["comision_gerente"] *
        para_pago_full["factor_comision_final"]
    )

para_pago_full["porcentaje_comision_ejecutivo"] = (
    para_pago_full["comision_ejecutivo"] *
    para_pago_full["factor_comision_final"] * 100
)

if "comision_jefe" in para_pago_full.columns:
    para_pago_full["porcentaje_comision_jefe"] = (
        para_pago_full["comision_jefe"] *
        para_pago_full["factor_comision_final"] * 100
    )

if "comision_gerente" in para_pago_full.columns:
    para_pago_full["porcentaje_comision_gerente"] = (
        para_pago_full["comision_gerente"] *
        para_pago_full["factor_comision_final"] * 100
    )

para_pago_full["anio_factura"] = para_pago_full["tranDate"].dt.year.astype("Int64")


# =========================
# 9) Diagnóstico
# =========================
print("Resumen decisión de pago:")
display(
    para_pago_full["decision_pago"]
    .value_counts(dropna=False)
    .rename_axis("decision_pago")
    .reset_index(name="filas")
)

print("\nResumen por periodo de corte:")
display(
    para_pago_full["Periodo Corte"]
    .value_counts(dropna=False)
    .rename_axis("Periodo Corte")
    .reset_index(name="filas")
    .head(20)
)

cols_preview = [
    "tranId",
    "doc_base_comision",
    "entityid",
    "linea_comision_key",
    "tranDate",
    "dueDate",
    "dueDate_base_pago",
    "Fecha Real Pago",
    "Fecha Contabilizado Pago",
    "Periodo Corte",
    "Cumple Corte Contable",
    "dias_pago",
    "factor",
    "estado_piso",
    "cumple",
    "observacion_cumple",
    "decision_pago",
    "motivo_pago",
    "Valor",
    "valor_comision_ejecutivo",
]

cols_preview = [c for c in cols_preview if c in para_pago_full.columns]

print("\nMuestra para_pago_full:")
display(para_pago_full[cols_preview].head(20))

Resumen decisión de pago:


,decision_pago,filas
0,PARA REVISAR,71446
1,NO PAGA,10046
2,PAGA 100%,9360
3,PAGA 50%,580



Resumen por periodo de corte:


,Periodo Corte,filas
0,<NA>,70063
1,2026-05,1526
2,2025-06,1507
3,2025-09,1345
4,2025-01,1338
5,2025-08,1324
6,2026-01,1302
7,2025-11,1264
8,2025-05,1262
9,2025-02,1259



Muestra para_pago_full:


,tranId,doc_base_comision,entityid,linea_comision_key,tranDate,dueDate,dueDate_base_pago,Fecha Real Pago,Fecha Contabilizado Pago,Periodo Corte,Cumple Corte Contable,dias_pago,factor,estado_piso,cumple,observacion_cumple,decision_pago,motivo_pago,Valor,valor_comision_ejecutivo
0,SM24994,SM24025,JESUS DANIEL TORREGROSA EBRATT,RITEL,2022-05-20,2022-06-19,2022-05-18,2022-07-13,2022-07-14,<NA>,False,56.0,0.5,no suma,False,Para revisión: sin configuración de piso para ...,PARA REVISAR,Sin configuración de piso para el período,2447200.0,NaN
1,SM27260,SM27260,CLARA ISABEL MUÑOZ,RITEL,2022-08-03,2022-09-02,2022-09-02,2022-09-13,2022-09-17,<NA>,False,11.0,1.0,suma,False,Para revisión: sin configuración de piso para ...,PARA REVISAR,Sin configuración de piso para el período,8702660.0,NaN
2,SM27887,SM27887,ADRIANA MARITZA QUITIAN ROZO,RITEL,2022-08-22,2022-09-21,2022-09-21,2024-11-01,2024-11-08,<NA>,False,772.0,0.0,suma,False,Para revisión: sin configuración de piso para ...,PARA REVISAR,Sin configuración de piso para el período,5100020.0,NaN
3,SM28758,SM28758,CLARA ISABEL MUÑOZ,RITEL,2022-09-16,2022-09-26,2022-09-26,2022-09-30,2022-09-30,<NA>,False,4.0,1.0,suma,False,Para revisión: sin configuración de piso para ...,PARA REVISAR,Sin configuración de piso para el período,5600000.0,NaN
4,SM32030,PR-5378,LUISA FERNANDA CAMACHO CUADRADO,RITEL,2022-12-18,2022-12-28,2022-10-29,NaT,2022-12-18,<NA>,False,NaN,NaN,no suma,False,Para revisión: sin configuración de piso para ...,PARA REVISAR,Sin configuración de piso para el período,2700000.0,NaN
5,SM32337,SM27742,JESUS DANIEL TORREGROSA EBRATT,RITEL,2023-01-05,2023-02-04,2022-09-16,2023-02-10,2023-02-13,<NA>,False,147.0,0.0,no suma,False,Para revisión: sin configuración de piso para ...,PARA REVISAR,Sin configuración de piso para el período,3864000.0,NaN
6,SM33455,SM33455,ADRIANA MARITZA QUITIAN ROZO,RITEL,2023-02-14,2023-03-16,2023-03-16,2023-03-10,2023-03-22,<NA>,False,-6.0,1.0,suma,False,Para revisión: sin configuración de piso para ...,PARA REVISAR,Sin configuración de piso para el período,10660000.0,NaN
7,SM35975,SM33123,NATALIA DEL PILAR OSPINA PULIDO,RITEL,2023-05-08,2023-06-07,2023-03-08,2023-07-07,2023-07-11,<NA>,False,121.0,0.0,no suma,False,Para revisión: sin configuración de piso para ...,PARA REVISAR,Sin configuración de piso para el período,5600000.0,NaN
8,SM35451,PR-3456,MARIA VICTORIA CEBALLOS LONDOÑO,RITEL,2023-04-21,2023-04-30,2022-02-13,NaT,2023-04-21,<NA>,False,NaN,NaN,no suma,False,Para revisión: sin configuración de piso para ...,PARA REVISAR,Sin configuración de piso para el período,2200000.0,NaN
9,SM36972,PR-5671,ADRIANA MARITZA QUITIAN ROZO,RITEL,2023-06-09,2023-06-09,2022-12-10,2023-06-09,2023-06-13,<NA>,False,181.0,0.0,no suma,False,Para revisión: sin configuración de piso para ...,PARA REVISAR,Sin configuración de piso para el período,13600000.0,NaN


In [ ]:
# =========================================================
# CELDA 10 - REPORTES Y EXPORTES LIMPIOS
# =========================================================
# Asume que ya existen:
# norm_date, norm_num, norm_text, norm_id,
# stringify_period_cols, sort_cols, limpiar_para_excel, safe_name

def dejar_columnas_exportables(df, cols_negocio):
    df = df.copy()
    df = df.loc[:, ~df.columns.duplicated()].copy()
    cols_finales = [c for c in cols_negocio if c in df.columns]
    return df[cols_finales].copy()


=========================
# Compatibilidad mapa de líneas
# =========================
if "df_relation" not in globals():
    relation_data = {
        "ID INTERNO": [14, 13, 7, 8, 18, 9, 17, 16, 2, 1, 3, 6, 5, 19, 12, 10, 11, 15, 4],
        "NOMBRE": [
            "ADMON", "CALIBRACION", "CAPACITACION", "CERTIFICACION", "CSG",
            "ESCAS", "FRONTERAS", "INTER", "LMA", "LMAM", "LME", "LVM",
            "LVV", "NA", "PRUEBAS", "RETIE/LAP", "RIESGO", "RITEL",
            "VARIABLES ELECTRICAS"
        ],
    }
    df_relation = pd.DataFrame(relation_data)

if "id_linea_key" not in df_relation.columns and "ID INTERNO" in df_relation.columns:
    df_relation["id_linea_key"] = norm_id(df_relation["ID INTERNO"])

if "linea_comision_key" not in df_relation.columns and "NOMBRE" in df_relation.columns:
    df_relation["linea_comision_key"] = norm_upper(df_relation["NOMBRE"])

if "linea_codigo_key" not in df_relation.columns:
    if "id_linea_key" in df_relation.columns:
        df_relation["linea_codigo_key"] = df_relation["id_linea_key"]
    else:
        raise KeyError(
            "df_relation no tiene ni 'linea_codigo_key' ni 'id_linea_key'. "
            f"Columnas actuales: {df_relation.columns.tolist()}"
        )


# =========================
# 2) detalle_reporte desde ventas_piso
# =========================
detalle_base = ventas_piso.copy()

detalle_base["tranDate"] = norm_date(detalle_base["tranDate"])
detalle_base["dueDate"] = norm_date(detalle_base["dueDate"])
detalle_base["Valor"] = norm_num(detalle_base["Valor"])
detalle_base["TotalParaPiso_base"] = norm_num(detalle_base["TotalParaPiso_base"])
detalle_base["TotalParaPiso_final"] = norm_num(detalle_base["TotalParaPiso_final"])

if "periodo_factura" not in detalle_base.columns:
    detalle_base["periodo_factura"] = detalle_base["tranDate"].dt.to_period("M")

detalle_base["Periodo Factura"] = detalle_base["periodo_factura"].astype(str)
detalle_base["Anio Factura"] = detalle_base["tranDate"].dt.year.astype("Int64")
detalle_base["Linea Comision Codigo"] = norm_id(detalle_base["custitem_sm_lineacomision"])
detalle_base["lote_key"] = norm_text(detalle_base["lote"]).fillna("")

detalle_base = detalle_base.merge(
    df_relation[["linea_codigo_key", "linea_comision_key"]].rename(
        columns={"linea_codigo_key": "Linea Comision Codigo"}
    ),
    on="Linea Comision Codigo",
    how="left",
    validate="m:1",
)

if not costos_work.empty:
    detalle_base = detalle_base.merge(
        costos_work.rename(columns={"costo": "Costo Informativo"}),
        on="lote_key",
        how="left",
        validate="m:1",
    )
else:
    detalle_base["Costo Informativo"] = np.nan

detalle_base["Costo Informativo"] = norm_num(detalle_base["Costo Informativo"]).fillna(0)

detalle_reporte = detalle_base.rename(columns={
    "tranId": "Factura",
    "tranDate": "Fecha Factura",
    "dueDate": "Fecha Vencimiento",
    "entity": "ID Cliente",
    "entityid": "Ejecutivo",
    "Servicio": "Servicio",
    "cc": "Centro de Costo",
    "lote": "Lote",
    "linea_comision_key": "Linea Comision",
    "Valor": "Valor Factura",
    "TotalParaPiso_base": "Total Para Piso Base",
    "TotalParaPiso_final": "Total Para Piso Final",
    "estado_piso": "Estado Piso",
    "observacion_piso": "Observacion Piso",
    "documento_origen": "Documento Origen",
    "reemplazo": "Es Reemplazo",
    "tipo_reemplazo": "Tipo Reemplazo",
    "tranId_nc": "Nota Credito",
    "tranDate_nc": "Fecha Nota Credito",
    "Valor_nc": "Valor Nota Credito",
    "tipo_afectacion_nc": "Tipo Afectacion NC",
    "requiere_recalculo_piso": "Requiere Recalculo Piso",
    "periodo_recalculo": "Periodo Recalculo",
})

detalle_reporte = stringify_period_cols(detalle_reporte)

cols_detalle = [
    "Factura",
    "Fecha Factura",
    "Fecha Vencimiento",
    "Anio Factura",
    "Periodo Factura",
    "Ejecutivo",
    "ID Cliente",
    "Servicio",
    "Centro de Costo",
    "Linea Comision Codigo",
    "Linea Comision",
    "Lote",
    "Valor Factura",
    "Total Para Piso Base",
    "Total Para Piso Final",
    "Costo Informativo",
    "Estado Piso",
    "Observacion Piso",
    "Documento Origen",
    "Es Reemplazo",
    "Tipo Reemplazo",
    "Nota Credito",
    "Fecha Nota Credito",
    "Valor Nota Credito",
    "Tipo Afectacion NC",
    "Requiere Recalculo Piso",
    "Periodo Recalculo",
]

detalle_reporte = sort_cols(detalle_reporte, cols_detalle)
detalle_reporte = dejar_columnas_exportables(detalle_reporte, cols_detalle)


# =========================
# 3) cumpl_reporte desde cumplimiento_final
# =========================
cumpl_base = cumplimiento_final.copy()
cumpl_base = stringify_period_cols(cumpl_base)

cumpl_base["Anio"] = pd.to_numeric(
    cumpl_base["periodo_mes"].astype("string").str[:4],
    errors="coerce"
).astype("Int64")

cumpl_reporte = cumpl_base.rename(columns={
    "periodo_mes": "Periodo",
    "entityid": "Ejecutivo",
    "linea_comision": "Linea Comision",
    "TotalParaPiso": "Total Para Piso",
    "costo": "Costo Informativo",
    "meta": "Meta",
    "piso": "Piso",
    "cumple": "Cumple",
    "observacion_cumple": "Observacion Cumple",
    "requiere_revision_cumplimiento": "Requiere Revision Cumplimiento",
    "motivo_revision_cumplimiento": "Motivo Revision Cumplimiento",
})

cols_cumpl = [
    "Anio",
    "Periodo",
    "Ejecutivo",
    "Linea Comision",
    "Total Para Piso",
    "Costo Informativo",
    "Meta",
    "Piso",
    "Cumple",
    "Observacion Cumple",
    "Requiere Revision Cumplimiento",
    "Motivo Revision Cumplimiento",
]

cumpl_reporte = sort_cols(cumpl_reporte, cols_cumpl)
cumpl_reporte = dejar_columnas_exportables(cumpl_reporte, cols_cumpl)


# =========================
# 4) para_pagar_full_reporte desde para_pago_full
# =========================
pago_base = para_pago_full.copy()

pago_base["tranDate"] = norm_date(pago_base["tranDate"])
pago_base["dueDate"] = norm_date(pago_base["dueDate"])
pago_base["dueDate_base_pago"] = norm_date(pago_base["dueDate_base_pago"])
pago_base["Fecha Real Pago"] = norm_date(pago_base["Fecha Real Pago"])
pago_base["Fecha Contabilizado Pago"] = norm_date(pago_base["Fecha Contabilizado Pago"])
pago_base["Inicio Corte Contable"] = norm_date(pago_base["Inicio Corte Contable"])
pago_base["Fin Corte Contable"] = norm_date(pago_base["Fin Corte Contable"])

for c in [
    "Valor", "Pagado", "Saldo", "factor", "factor_comision_final",
    "comision_ejecutivo", "valor_comision_ejecutivo",
    "comision_jefe", "valor_comision_jefe",
    "comision_gerente", "valor_comision_gerente",
    "TotalParaPiso", "costo", "meta", "piso",
]:
    if c in pago_base.columns:
        pago_base[c] = norm_num(pago_base[c])

if "linea_comision_key" not in pago_base.columns and "custitem_sm_lineacomision" in pago_base.columns:
    pago_base["Linea Comision Codigo"] = norm_id(pago_base["custitem_sm_lineacomision"])
    pago_base = pago_base.merge(
        df_relation[["linea_codigo_key", "linea_comision_key"]].rename(
            columns={"linea_codigo_key": "Linea Comision Codigo"}
        ),
        on="Linea Comision Codigo",
        how="left",
        validate="m:1",
    )
else:
    pago_base["Linea Comision Codigo"] = (
        norm_id(pago_base["custitem_sm_lineacomision"])
        if "custitem_sm_lineacomision" in pago_base.columns
        else pd.NA
    )

if "costo" not in pago_base.columns:
    pago_base["lote_key"] = norm_text(pago_base["lote"]).fillna("")
    if not costos_work.empty:
        pago_base = pago_base.merge(
            costos_work,
            on="lote_key",
            how="left",
            validate="m:1",
        )
    else:
        pago_base["costo"] = np.nan

pago_base["Periodo Factura"] = (
    pago_base["periodo_mes"].astype(str)
    if "periodo_mes" in pago_base.columns
    else pago_base["tranDate"].dt.to_period("M").astype(str)
)
pago_base["Anio Factura"] = pago_base["tranDate"].dt.year.astype("Int64")

para_pagar_full_reporte = pago_base.rename(columns={
    "doc_base_comision": "Documento Base Comision",
    "tranId": "Factura",
    "documento_origen": "Documento Origen",
    "tranDate": "Fecha Factura",
    "dueDate": "Fecha Vencimiento",
    "dueDate_base_pago": "Fecha Vencimiento Base Pago",
    "entity": "ID Cliente",
    "entityid": "Ejecutivo",
    "Servicio": "Servicio",
    "cc": "Centro de Costo",
    "linea_comision_key": "Linea Comision",
    "lote": "Lote",
    "Valor": "Valor Factura",
    "Pagado": "Valor Pagado",
    "Saldo": "Saldo Factura",
    "Fecha Contabilizado Pago": "Fecha Contabilizacion Pago",
    "factor": "Factor Pago",
    "factor_comision_final": "Factor Comision Final",
    "nota_pago": "Observacion Pago",
    "estado_piso": "Estado Piso",
    "observacion_piso": "Observacion Piso",
    "cumple": "Cumple",
    "observacion_cumple": "Observacion Cumple",
    "decision_pago": "Decision Pago",
    "motivo_pago": "Motivo Pago",
    "costo": "Costo Informativo",
    "TotalParaPiso": "Total Para Piso",
    "meta": "Meta",
    "piso": "Piso",
    "comision_ejecutivo": "Porcentaje Comision Ejecutivo",
    "valor_comision_ejecutivo": "Valor Comision Ejecutivo",
    "jefe": "Jefe",
    "comision_jefe": "Porcentaje Comision Jefe",
    "valor_comision_jefe": "Valor Comision Jefe",
    "gerente": "Gerente",
    "comision_gerente": "Porcentaje Comision Gerente",
    "valor_comision_gerente": "Valor Comision Gerente",
    "dias_pago": "Dias Pago",
    "es_proforma": "Es Proforma",
    "reemplazo": "Es Reemplazo",
    "requiere_revision_cumplimiento": "Requiere Revision Cumplimiento",
    "motivo_revision_cumplimiento": "Motivo Revision Cumplimiento",
    "requiere_revision_pago": "Requiere Revision Pago",
})

para_pagar_full_reporte = stringify_period_cols(para_pagar_full_reporte)

cols_pago = [
    "Documento Base Comision",
    "Factura",
    "Documento Origen",
    "Fecha Factura",
    "Fecha Vencimiento",
    "Fecha Vencimiento Base Pago",
    "Anio Factura",
    "Periodo Factura",
    "Ejecutivo",
    "Jefe",
    "Gerente",
    "ID Cliente",
    "Servicio",
    "Centro de Costo",
    "Linea Comision Codigo",
    "Linea Comision",
    "Lote",
    "Valor Factura",
    "Total Para Piso",
    "Costo Informativo",
    "Meta",
    "Piso",
    "Fecha Real Pago",
    "Fecha Contabilizacion Pago",
    "Inicio Corte Contable",
    "Fin Corte Contable",
    "Periodo Corte",
    "Cumple Corte Contable",
    "Dias Pago",
    "Factor Pago",
    "Factor Comision Final",
    "Observacion Pago",
    "Cumple",
    "Observacion Cumple",
    "Decision Pago",
    "Motivo Pago",
    "Valor Pagado",
    "Saldo Factura",
    "Porcentaje Comision Ejecutivo",
    "Valor Comision Ejecutivo",
    "Porcentaje Comision Jefe",
    "Valor Comision Jefe",
    "Porcentaje Comision Gerente",
    "Valor Comision Gerente",
    "Estado Piso",
    "Observacion Piso",
    "Es Reemplazo",
    "Es Proforma",
    "Requiere Revision Cumplimiento",
    "Motivo Revision Cumplimiento",
    "Requiere Revision Pago",
]

para_pagar_full_reporte = sort_cols(para_pagar_full_reporte, cols_pago)
para_pagar_full_reporte = dejar_columnas_exportables(para_pagar_full_reporte, cols_pago)


# =========================
# 5) para_revision
# =========================
if "Decision Pago" in para_pagar_full_reporte.columns:
    para_revision_reporte = para_pagar_full_reporte[
        para_pagar_full_reporte["Decision Pago"].astype("string").str.strip().eq("PARA REVISAR")
    ].copy()
else:
    para_revision_reporte = detalle_reporte[
        detalle_reporte["Estado Piso"].astype("string").str.strip().str.lower().eq("para revision")
    ].copy()

if set(cols_pago).intersection(para_revision_reporte.columns):
    para_revision_reporte = dejar_columnas_exportables(para_revision_reporte, cols_pago)
else:
    para_revision_reporte = dejar_columnas_exportables(para_revision_reporte, cols_detalle)


# =========================
# 6) Limpieza para Excel
# =========================
detalle_reporte_xlsx = limpiar_para_excel(detalle_reporte)
cumpl_reporte_xlsx = limpiar_para_excel(cumpl_reporte)
para_pagar_full_reporte_xlsx = limpiar_para_excel(para_pagar_full_reporte)
para_revision_reporte_xlsx = limpiar_para_excel(para_revision_reporte)


# =========================
# 7) Exportes
# =========================
base_export = Path.cwd() / "exports"
base_export.mkdir(parents=True, exist_ok=True)

archivo_general = base_export / "general.xlsx"
with pd.ExcelWriter(archivo_general, engine="openpyxl", mode="w") as writer:
    detalle_reporte_xlsx.to_excel(writer, sheet_name="detalle_estado", index=False)
    cumpl_reporte_xlsx.to_excel(writer, sheet_name="cumplimiento", index=False)
    para_pagar_full_reporte_xlsx.to_excel(writer, sheet_name="para_pagar", index=False)
    para_revision_reporte_xlsx.to_excel(writer, sheet_name="para_revision", index=False)

print(f"Generado: {archivo_general}")

# 1) Estado por anio
anios_detalle = (
    sorted(detalle_reporte_xlsx["Anio Factura"].dropna().astype(int).unique().tolist())
    if "Anio Factura" in detalle_reporte_xlsx.columns else []
)
for anio in anios_detalle:
    ruta = base_export / f"estado_por_anio_{anio}.xlsx"
    df_anio = detalle_reporte_xlsx[detalle_reporte_xlsx["Anio Factura"].eq(anio)].copy()

    with pd.ExcelWriter(ruta, engine="openpyxl", mode="w") as writer:
        df_anio[df_anio["Estado Piso"].astype("string").str.strip().str.lower().eq("suma")].to_excel(
            writer, sheet_name="suma", index=False
        )
        df_anio[df_anio["Estado Piso"].astype("string").str.strip().str.lower().eq("no suma")].to_excel(
            writer, sheet_name="no_suma", index=False
        )
        df_anio[df_anio["Estado Piso"].astype("string").str.strip().str.lower().eq("para revision")].to_excel(
            writer, sheet_name="para_revision", index=False
        )

    print(f"Generado: {ruta}")

# 2) Cumplimiento por anio
anios_cumpl = (
    sorted(cumpl_reporte_xlsx["Anio"].dropna().astype(int).unique().tolist())
    if "Anio" in cumpl_reporte_xlsx.columns else []
)
for anio in anios_cumpl:
    ruta = base_export / f"cumplimiento_por_anio_{anio}.xlsx"
    df_anio = cumpl_reporte_xlsx[cumpl_reporte_xlsx["Anio"].eq(anio)].copy()

    with pd.ExcelWriter(ruta, engine="openpyxl", mode="w") as writer:
        df_anio.to_excel(writer, sheet_name="cumplimiento", index=False)

    print(f"Generado: {ruta}")

# 3) Para pagar por anio
anios_pago = (
    sorted(para_pagar_full_reporte_xlsx["Anio Factura"].dropna().astype(int).unique().tolist())
    if "Anio Factura" in para_pagar_full_reporte_xlsx.columns else []
)
for anio in anios_pago:
    ruta = base_export / f"para_pagar_por_anio_{anio}.xlsx"
    df_anio = para_pagar_full_reporte_xlsx[
        para_pagar_full_reporte_xlsx["Anio Factura"].eq(anio)
    ].copy()

    with pd.ExcelWriter(ruta, engine="openpyxl", mode="w") as writer:
        df_anio.to_excel(writer, sheet_name="para_pagar", index=False)

    print(f"Generado: {ruta}")

# 4) Por ejecutivo y anio
base_ej = base_export / "por_ejecutivo"
base_ej.mkdir(parents=True, exist_ok=True)

for anio in anios_pago:
    carpeta_anio = base_ej / str(anio)
    carpeta_anio.mkdir(parents=True, exist_ok=True)

    df_anio = para_pagar_full_reporte_xlsx[
        para_pagar_full_reporte_xlsx["Anio Factura"].eq(anio)
    ].copy()

    if "Ejecutivo" not in df_anio.columns:
        continue

    for ejecutivo, df_ej in df_anio.groupby("Ejecutivo", dropna=True):
        ruta = carpeta_anio / f"{safe_name(ejecutivo)}.xlsx"
        with pd.ExcelWriter(ruta, engine="openpyxl", mode="w") as writer:
            df_ej.sort_values(
                [c for c in ["Fecha Factura", "Factura", "Servicio", "Lote"] if c in df_ej.columns],
                na_position="last",
            ).to_excel(writer, sheet_name="para_pagar", index=False)

        print(f"Generado: {ruta}")

print("Exportes finalizados.")

KeyError: "['linea_codigo_key'] not in index"

In [ ]:
# =========================================================
# CELDA 11 - RESUMEN PARA PAGO POR CORTE
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np


periodo_corte_resumen = "2026-05"


def normalizar_periodo_yyyy_mm(s: pd.Series) -> pd.Series:
    s2 = pd.Series(s, index=s.index)
    dt = pd.to_datetime(s2, errors="coerce")
    out = s2.astype("string").str.strip()

    mask_dt = dt.notna()
    if mask_dt.any():
        out.loc[mask_dt] = dt.loc[mask_dt].dt.to_period("M").astype(str)

    out = out.str.replace(r"^(\d{4}-\d{2}).*$", r"\1", regex=True)
    return out


def unique_preserve_order(values):
    out = []
    seen = set()
    for v in values:
        if pd.isna(v):
            continue
        v = str(v).strip()
        if not v or v in seen:
            continue
        seen.add(v)
        out.append(v)
    return out


if "pisos_raw" not in globals():
    raise NameError("No existe 'pisos_raw'. Ejecuta primero la celda de normalización base.")

if "para_pagar_full_reporte" in globals():
    pago_resumen_base = para_pagar_full_reporte.copy()
else:
    if "para_pago_full" not in globals():
        raise NameError("No existe 'para_pago_full' ni 'para_pagar_full_reporte'. Ejecuta primero la celda 9 o la 10.")

    pago_resumen_base = para_pago_full.copy().rename(columns={
        "entityid": "Ejecutivo",
        "decision_pago": "Decision Pago",
        "jefe": "Jefe",
        "gerente": "Gerente",
        "valor_comision_ejecutivo": "Valor Comision Ejecutivo",
        "valor_comision_jefe": "Valor Comision Jefe",
        "valor_comision_gerente": "Valor Comision Gerente",
    })


# =========================
# 1) Maestra fija desde pisos_raw
# =========================
pisos_cfg = pisos_raw.copy()

col_periodo_pisos = None
for c in ["periodo_mes", "periodo_piso", "perido", "periodo"]:
    if c in pisos_cfg.columns:
        col_periodo_pisos = c
        break

if col_periodo_pisos is None:
    raise KeyError("No encontré columna de período en 'pisos_raw'.")

col_ejecutivo_pisos = None
for c in ["ejecutivo", "entityid", "Ejecutivo"]:
    if c in pisos_cfg.columns:
        col_ejecutivo_pisos = c
        break

if col_ejecutivo_pisos is None:
    raise KeyError("No encontré columna de ejecutivo en 'pisos_raw'.")

if "jefe" not in pisos_cfg.columns or "gerente" not in pisos_cfg.columns:
    raise KeyError("En 'pisos_raw' faltan las columnas 'jefe' y/o 'gerente'.")

pisos_cfg["periodo_resumen"] = normalizar_periodo_yyyy_mm(pisos_cfg[col_periodo_pisos])
pisos_cfg["ejecutivo_resumen"] = pisos_cfg[col_ejecutivo_pisos].astype("string").str.strip()
pisos_cfg["jefe_resumen"] = pisos_cfg["jefe"].astype("string").str.strip()
pisos_cfg["gerente_resumen"] = pisos_cfg["gerente"].astype("string").str.strip()

pisos_periodo = pisos_cfg[
    pisos_cfg["periodo_resumen"].eq(periodo_corte_resumen)
].copy()

if pisos_periodo.empty:
    raise ValueError(f"No encontré configuración en pisos_raw para el período {periodo_corte_resumen}.")

ejecutivos_master = unique_preserve_order(pisos_periodo["ejecutivo_resumen"].tolist())

responsables_master = unique_preserve_order(
    pisos_periodo["jefe_resumen"].tolist() +
    pisos_periodo["gerente_resumen"].tolist()
)

if not ejecutivos_master:
    raise ValueError(f"No encontré ejecutivos en pisos_raw para el período {periodo_corte_resumen}.")


# =========================
# 2) Base pagar del período
# =========================
required_pago_cols = [
    "Ejecutivo",
    "Decision Pago",
    "Periodo Corte",
    "Jefe",
    "Gerente",
    "Valor Comision Ejecutivo",
    "Valor Comision Jefe",
    "Valor Comision Gerente",
]

faltantes_pago = [c for c in required_pago_cols if c not in pago_resumen_base.columns]
if faltantes_pago:
    raise KeyError(f"Faltan columnas en la base de pago: {faltantes_pago}")

pago = pago_resumen_base.copy()
pago["Periodo Corte"] = normalizar_periodo_yyyy_mm(pago["Periodo Corte"])
pago["Ejecutivo"] = pago["Ejecutivo"].astype("string").str.strip()
pago["Jefe"] = pago["Jefe"].astype("string").str.strip()
pago["Gerente"] = pago["Gerente"].astype("string").str.strip()
pago["Decision Pago"] = pago["Decision Pago"].astype("string").str.strip().str.upper()

for c in [
    "Valor Comision Ejecutivo",
    "Valor Comision Jefe",
    "Valor Comision Gerente",
]:
    pago[c] = pd.to_numeric(pago[c], errors="coerce").fillna(0)

pago_periodo = pago[
    pago["Periodo Corte"].eq(periodo_corte_resumen)
    & pago["Decision Pago"].eq("PAGAR")
].copy()


# =========================
# 3) Comisión ejecutivo
# =========================
comision_ej = (
    pago_periodo.groupby("Ejecutivo", as_index=False)["Valor Comision Ejecutivo"]
    .sum()
    .rename(columns={"Valor Comision Ejecutivo": "COMISION"})
)


# =========================
# 4) Columnas responsables
# =========================
jefe_long = pago_periodo[["Ejecutivo", "Jefe", "Valor Comision Jefe"]].rename(
    columns={"Jefe": "responsable", "Valor Comision Jefe": "valor"}
)

gerente_long = pago_periodo[["Ejecutivo", "Gerente", "Valor Comision Gerente"]].rename(
    columns={"Gerente": "responsable", "Valor Comision Gerente": "valor"}
)

responsables_long = pd.concat([jefe_long, gerente_long], ignore_index=True)
responsables_long["responsable"] = responsables_long["responsable"].astype("string").str.strip()
responsables_long["valor"] = pd.to_numeric(responsables_long["valor"], errors="coerce").fillna(0)

responsables_long = responsables_long[
    responsables_long["Ejecutivo"].notna()
    & responsables_long["responsable"].notna()
    & responsables_long["responsable"].astype("string").str.strip().ne("")
].copy()

if responsables_long.empty:
    resp_pivot = pd.DataFrame(index=ejecutivos_master)
else:
    resp_pivot = (
        responsables_long.groupby(["Ejecutivo", "responsable"], as_index=False)["valor"]
        .sum()
        .pivot(index="Ejecutivo", columns="responsable", values="valor")
        .fillna(0)
    )

resp_pivot = resp_pivot.reindex(index=ejecutivos_master, fill_value=0)

for responsable in responsables_master:
    if responsable not in resp_pivot.columns:
        resp_pivot[responsable] = 0

resp_pivot = resp_pivot[responsables_master].copy()
resp_pivot = resp_pivot.reset_index()


# =========================
# 5) Armar resumen final
# =========================
resumen_pago = pd.DataFrame({
    "Representante de ventas": ejecutivos_master
})

resumen_pago = resumen_pago.merge(
    comision_ej.rename(columns={"Ejecutivo": "Representante de ventas"}),
    on="Representante de ventas",
    how="left",
)

resumen_pago["COMISION"] = pd.to_numeric(resumen_pago["COMISION"], errors="coerce").fillna(0)
resumen_pago["DESCUENTOS"] = 0.0
resumen_pago["VALOR A PAGAR"] = resumen_pago["COMISION"] - resumen_pago["DESCUENTOS"]

resumen_pago = resumen_pago.merge(
    resp_pivot.rename(columns={"Ejecutivo": "Representante de ventas"}),
    on="Representante de ventas",
    how="left",
)

cols_numericas = [
    c for c in resumen_pago.columns
    if c != "Representante de ventas"
]

for c in cols_numericas:
    resumen_pago[c] = pd.to_numeric(resumen_pago[c], errors="coerce").fillna(0)

# Fila TOTAL
fila_total = {"Representante de ventas": "TOTAL"}
for c in cols_numericas:
    fila_total[c] = resumen_pago[c].sum()

resumen_pago = pd.concat(
    [resumen_pago, pd.DataFrame([fila_total])],
    ignore_index=True
)

# Opcional: redondeo
for c in cols_numericas:
    resumen_pago[c] = resumen_pago[c].round(2)


# =========================
# 6) Exportar archivo nuevo
# =========================
base_export = Path.cwd() / "exports"
base_export.mkdir(parents=True, exist_ok=True)

archivo_resumen = base_export / f"resumen_pago_corte_{periodo_corte_resumen}.xlsx"

with pd.ExcelWriter(archivo_resumen, engine="openpyxl", mode="w") as writer:
    resumen_pago.to_excel(writer, sheet_name="resumen", index=False)


In [118]:
from review_bundle import save_dataframe_bundle

save_dataframe_bundle(
    {
        "detalle_reporte": detalle_reporte,
        "cumpl_reporte": cumpl_reporte,
        "para_pagar_full_reporte": para_pagar_full_reporte,
        "para_revision_reporte": para_revision_reporte,
        "cumplimiento_final": cumplimiento_final,
        "para_pago_full": para_pago_full,
    },
    out_dir="app_data",
)

WindowsPath('app_data/manifest.json')

In [119]:
from pathlib import Path
import importlib
import review_bundle

importlib.reload(review_bundle)

base_dir = Path(r"C:\Users\gloria.cabrera\OneDrive - Servimeters\Proyectos\2025\comisiones\app_data")

manifest_path = review_bundle.save_dataframe_bundle(
    {
        "detalle_reporte": detalle_reporte,
        "cumpl_reporte": cumpl_reporte,
        "para_pagar_full_reporte": para_pagar_full_reporte,
        "para_revision_reporte": para_revision_reporte,
        "cumplimiento_final": cumplimiento_final,
        "para_pago_full": para_pago_full,
    },
    out_dir=base_dir,
)

print("Manifest:", manifest_path)
print("Parquet generados:")
print(sorted([p.name for p in base_dir.glob("*.parquet")]))

Manifest: C:\Users\gloria.cabrera\OneDrive - Servimeters\Proyectos\2025\comisiones\app_data\manifest.json
Parquet generados:
[]


In [120]:
# =========================================================
# CELDA 9A - VALIDACION DE PAGO Y CORTE
# =========================================================

print("Resumen decision_pago:")
display(
    para_pago_full["decision_pago"]
    .value_counts(dropna=False)
    .rename_axis("decision_pago")
    .reset_index(name="filas")
)

print("\nResumen motivo_pago:")
display(
    para_pago_full["motivo_pago"]
    .value_counts(dropna=False)
    .rename_axis("motivo_pago")
    .reset_index(name="filas")
    .head(20)
)

print("\nResumen Periodo Corte:")
display(
    para_pago_full["Periodo Corte"]
    .astype("string")
    .value_counts(dropna=False)
    .rename_axis("Periodo Corte")
    .reset_index(name="filas")
    .head(20)
)

print("\nResumen factor:")
display(
    para_pago_full["factor"]
    .value_counts(dropna=False)
    .rename_axis("factor")
    .reset_index(name="filas")
)

print("\nResumen cumple piso:")
display(
    para_pago_full["cumple"]
    .value_counts(dropna=False)
    .rename_axis("cumple")
    .reset_index(name="filas")
)

print("\nResumen requiere_revision_pago:")
display(
    para_pago_full["requiere_revision_pago"]
    .value_counts(dropna=False)
    .rename_axis("requiere_revision_pago")
    .reset_index(name="filas")
)

cols_validacion = [
    "tranId",
    "doc_base_comision",
    "entityid",
    "linea_comision_key",
    "tranDate",
    "dueDate",
    "dueDate_base_pago",
    "Fecha Real Pago",
    "Fecha Contabilizado Pago",
    "Inicio Corte Contable",
    "Fin Corte Contable",
    "Periodo Corte",
    "Cumple Corte Contable",
    "dias_pago",
    "factor",
    "estado_piso",
    "cumple",
    "observacion_cumple",
    "decision_pago",
    "motivo_pago",
    "Saldo",
    "Pagado",
    "Valor",
    "valor_comision_ejecutivo",
]

cols_validacion = [c for c in cols_validacion if c in para_pago_full.columns]

print("\nMuestra general:")
display(
    para_pago_full[cols_validacion]
    .sort_values(
        [c for c in ["Fecha Contabilizado Pago", "entityid", "tranId"] if c in para_pago_full.columns],
        na_position="last",
    )
    .head(50)
)

print("\nCasos PARA REVISAR:")
display(
    para_pago_full.loc[
        para_pago_full["decision_pago"].eq("PARA REVISAR"),
        cols_validacion
    ]
    .sort_values(
        [c for c in ["Fecha Contabilizado Pago", "entityid", "tranId"] if c in para_pago_full.columns],
        na_position="last",
    )
    .head(50)
)

print("\nCasos PAGA 100%:")
display(
    para_pago_full.loc[
        para_pago_full["decision_pago"].eq("PAGA 100%"),
        cols_validacion
    ]
    .sort_values(
        [c for c in ["Fecha Contabilizado Pago", "entityid", "tranId"] if c in para_pago_full.columns],
        na_position="last",
    )
    .head(50)
)

print("\nCasos PAGA 50%:")
display(
    para_pago_full.loc[
        para_pago_full["decision_pago"].eq("PAGA 50%"),
        cols_validacion
    ]
    .sort_values(
        [c for c in ["Fecha Contabilizado Pago", "entityid", "tranId"] if c in para_pago_full.columns],
        na_position="last",
    )
    .head(50)
)

print("\nCasos NO PAGA:")
display(
    para_pago_full.loc[
        para_pago_full["decision_pago"].eq("NO PAGA"),
        cols_validacion
    ]
    .sort_values(
        [c for c in ["Fecha Contabilizado Pago", "entityid", "tranId"] if c in para_pago_full.columns],
        na_position="last",
    )
    .head(50)
)

Resumen decision_pago:


,decision_pago,filas
0,PARA REVISAR,71446
1,NO PAGA,9987
2,PAGA 100%,9360
3,PAGA 50%,580



Resumen motivo_pago:


,motivo_pago,filas
0,Sin configuración de piso para el período,69127
1,Cumple pago <= 30 días desde dueDate,9360
2,No cumple piso ni meta,4139
3,Hay Fecha Contabilizado Pago pero falta Fecha ...,2068
4,Es proforma; no paga comisión,1286
5,Sin pago registrado,769
6,Cumple pago entre 31 y 60 días desde dueDate,580
7,Pago después de 60 días desde dueDate,275
8,No cumple: no alcanzó el mayor piso configurad...,212
9,Sin mapeo de línea de comisión,207



Resumen Periodo Corte:


,Periodo Corte,filas
0,<NA>,70004
1,2026-05,1526
2,2025-06,1507
3,2025-09,1345
4,2025-01,1338
5,2025-08,1324
6,2026-01,1302
7,2025-11,1264
8,2025-05,1262
9,2025-02,1259


... (12 outputs left out)


Casos PAGA 50%:


,tranId,doc_base_comision,entityid,linea_comision_key,tranDate,dueDate,dueDate_base_pago,Fecha Real Pago,Fecha Contabilizado Pago,Inicio Corte Contable,Fin Corte Contable,Periodo Corte,Cumple Corte Contable,dias_pago,factor,estado_piso,cumple,observacion_cumple,decision_pago,motivo_pago,Saldo,Pagado,Valor,valor_comision_ejecutivo
8525,SM55291,SM55291,SERGIO ANDRES LOPEZ FUENTES,RETIE/LAP,2025-01-16,2025-01-16,2025-01-16,2025-02-18,2025-02-19,2025-01-23,2025-02-21,2025-02,True,33.0,0.5,suma,True,Cumple meta,PAGA 50%,Cumple pago entre 31 y 60 días desde dueDate,0,1428000,1200000.0,24000.00
6377,SM55292,SM55292,SERGIO ANDRES LOPEZ FUENTES,RETIE/LAP,2025-01-16,2025-01-16,2025-01-16,2025-02-18,2025-02-19,2025-01-23,2025-02-21,2025-02,True,33.0,0.5,suma,True,Cumple meta,PAGA 50%,Cumple pago entre 31 y 60 días desde dueDate,0,1428000,1200000.0,24000.00
42585,SM55330,SM55330,SERGIO ANDRES LOPEZ FUENTES,RETIE/LAP,2025-01-17,2025-01-17,2025-01-17,2025-02-20,2025-02-21,2025-01-23,2025-02-21,2025-02,True,34.0,0.5,suma,True,Cumple meta,PAGA 50%,Cumple pago entre 31 y 60 días desde dueDate,0,2737000,2300000.0,46000.00
73932,SM55350,SM55350,MARIA VICTORIA MILLAN CASTILLO,ESCAS,2025-01-20,2025-01-20,2025-01-20,2025-02-21,2025-02-24,2025-02-22,2025-03-20,2025-03,True,32.0,0.5,suma,True,Cumple piso (sin piso configurado para el ejec...,PAGA 50%,Cumple pago entre 31 y 60 días desde dueDate,0,3379600,2840000.0,NaN
2268,SM55189,SM55189,DEIBYS GUILLERMO JIMENEZ CHARRIS,RETIE/LAP,2025-01-14,2025-01-14,2025-01-14,2025-02-22,2025-02-25,2025-02-22,2025-03-20,2025-03,True,39.0,0.5,suma,True,Cumple meta,PAGA 50%,Cumple pago entre 31 y 60 días desde dueDate,0,1340999,1126890.0,22537.80
69365,SM55426,SM55426,MARIA VICTORIA MILLAN CASTILLO,ESCAS,2025-01-24,2025-01-24,2025-01-24,2025-02-25,2025-02-26,2025-02-22,2025-03-20,2025-03,True,32.0,0.5,suma,True,Cumple piso (sin piso configurado para el ejec...,PAGA 50%,Cumple pago entre 31 y 60 días desde dueDate,0,297500,250000.0,NaN
71383,SM55355,SM55355,JULY DANIELA MURCIA PEDRAZA,ESCAS,2025-01-20,2025-01-20,2025-01-20,2025-02-28,2025-02-28,2025-02-22,2025-03-20,2025-03,True,39.0,0.5,suma,True,Cumple meta,PAGA 50%,Cumple pago entre 31 y 60 días desde dueDate,0,321300,270000.0,5400.00
34360,SM55379,SM55379,MARIA JOSE PEREZ CERVANTES,LMAM,2025-01-20,2025-01-20,2025-01-20,2025-02-28,2025-02-28,2025-02-22,2025-03-20,2025-03,True,39.0,0.5,suma,True,Cumple meta,PAGA 50%,Cumple pago entre 31 y 60 días desde dueDate,0,1080520,908000.0,18160.00
68881,SM55455,SM55455,MARIA VICTORIA MILLAN CASTILLO,ESCAS,2025-01-22,2025-01-22,2025-01-22,2025-03-07,2025-03-07,2025-02-22,2025-03-20,2025-03,True,44.0,0.5,suma,True,Cumple piso (sin piso configurado para el ejec...,PAGA 50%,Cumple pago entre 31 y 60 días desde dueDate,0,273700,230000.0,NaN
19648,SM55252,SM55252,JULY DANIELA MURCIA PEDRAZA,ESCAS,2025-01-16,2025-01-16,2025-01-16,2025-03-03,2025-03-10,2025-02-22,2025-03-20,2025-03,True,46.0,0.5,suma,True,Cumple meta,PAGA 50%,Cumple pago entre 31 y 60 días desde dueDate,0,285600,240000.0,4800.00



Casos NO PAGA:


,tranId,doc_base_comision,entityid,linea_comision_key,tranDate,dueDate,dueDate_base_pago,Fecha Real Pago,Fecha Contabilizado Pago,Inicio Corte Contable,Fin Corte Contable,Periodo Corte,Cumple Corte Contable,dias_pago,factor,estado_piso,cumple,observacion_cumple,decision_pago,motivo_pago,Saldo,Pagado,Valor,valor_comision_ejecutivo
40387,SM55081,SM55081,LAURA ELENA RAMIREZ ROJAS,LVV,2025-01-08,2025-01-08,2025-01-08,2025-01-09,2025-01-13,2024-12-20,2025-01-22,2025-01,True,1.0,1.0,suma,False,No cumple piso ni meta,NO PAGA,No cumple piso ni meta,0,428400,360000.0,0.0
43916,SM55090,SM50909,MARIA JOSE PEREZ CERVANTES,LMAM,2025-01-10,2025-01-10,2024-09-04,2025-01-13,2025-01-14,2024-12-20,2025-01-22,2025-01,True,131.0,0.0,no suma,True,Cumple meta,NO PAGA,No suma por ser reemplazo de SM50909,0,468860,394000.0,0.0
58687,SM55172,SM55172,ADRIANA MARIA VELEZ ACEVEDO,NA,2025-01-13,2025-01-13,2025-01-13,2025-01-15,2025-01-16,2024-12-20,2025-01-22,2025-01,True,2.0,1.0,suma,False,No cumple: no alcanzó el mayor piso configurad...,NO PAGA,No cumple: no alcanzó el mayor piso configurad...,0,35700,30000.0,NaN
58744,SM55185,SM55185,ADRIANA MARIA VELEZ ACEVEDO,NA,2025-01-13,2025-01-13,2025-01-13,2025-01-15,2025-01-16,2024-12-20,2025-01-22,2025-01,True,2.0,1.0,suma,False,No cumple: no alcanzó el mayor piso configurad...,NO PAGA,No cumple: no alcanzó el mayor piso configurad...,0,35700,30000.0,NaN
58639,SM55186,SM55186,ADRIANA MARIA VELEZ ACEVEDO,NA,2025-01-13,2025-01-13,2025-01-13,2025-01-15,2025-01-16,2024-12-20,2025-01-22,2025-01,True,2.0,1.0,suma,False,No cumple: no alcanzó el mayor piso configurad...,NO PAGA,No cumple: no alcanzó el mayor piso configurad...,0,35700,30000.0,NaN
57597,SM55192,SM55192,ADRIANA MARIA VELEZ ACEVEDO,RETIE/LAP,2025-01-14,2025-01-14,2025-01-14,2025-01-15,2025-01-16,2024-12-20,2025-01-22,2025-01,True,1.0,1.0,suma,False,No cumple piso ni meta,NO PAGA,No cumple piso ni meta,0,2142000,1800000.0,0.0
56943,SM55238,SM55238,ADRIANA MARIA VELEZ ACEVEDO,RETIE/LAP,2025-01-14,2025-01-14,2025-01-14,2024-11-22,2025-01-16,2024-12-20,2025-01-22,2025-01,True,-53.0,1.0,suma,False,No cumple piso ni meta,NO PAGA,No cumple piso ni meta,0,1904000,1600000.0,0.0
39418,SM55249,SM55249,ADRIANA MARIA VELEZ ACEVEDO,RETIE/LAP,2025-01-15,2025-01-15,2025-01-15,2025-01-16,2025-01-16,2024-12-20,2025-01-22,2025-01,True,1.0,1.0,suma,False,No cumple piso ni meta,NO PAGA,No cumple piso ni meta,0,2739380,2302000.0,0.0
70863,SM55113,SM55113,ANGIE ALISON BERMUDEZ DIAZ,ESCAS,2025-01-10,2025-01-10,2025-01-10,2025-01-16,2025-01-16,2024-12-20,2025-01-22,2025-01,True,6.0,1.0,suma,False,No cumple piso ni meta,NO PAGA,No cumple piso ni meta,0,321300,270000.0,0.0
68312,SM55047,SM55047,AYLIN MARCELA MENDOZA RUIZ,ESCAS,2025-01-08,2025-01-08,2025-01-08,2024-12-27,2025-01-16,2024-12-20,2025-01-22,2025-01,True,-12.0,1.0,suma,False,No cumple piso ni meta,NO PAGA,No cumple piso ni meta,0,1071000,900000.0,0.0


In [121]:
# =========================================================
# CELDA 9B - HELPER DE AUDITORIA POR FACTURA Y LOTE
# =========================================================

def verificar_factura_pago(tranid: str, lote: str | None = None):
    tranid = str(tranid).strip()
    lote_norm = None if lote is None else str(lote).strip()

    base = para_pago_full[
        para_pago_full["tranId"].astype("string").str.strip().eq(tranid)
    ].copy()

    if lote_norm is not None and "lote" in base.columns:
        base = base[
            base["lote"].astype("string").str.strip().eq(lote_norm)
        ].copy()

    if base.empty:
        if lote_norm is None:
            print(f"No se encontró la factura {tranid} en para_pago_full.")
        else:
            print(f"No se encontró la factura {tranid} con lote {lote_norm} en para_pago_full.")
        return None

    cols_base = [
        "tranId",
        "lote",
        "Servicio",
        "doc_base_comision",
        "documento_origen",
        "entityid",
        "cc",
        "linea_comision_key",
        "tranDate",
        "dueDate",
        "dueDate_base_pago",
        "Fecha Real Pago",
        "Fecha Contabilizado Pago",
        "Inicio Corte Contable",
        "Fin Corte Contable",
        "Periodo Corte",
        "Cumple Corte Contable",
        "dias_pago",
        "factor",
        "estado_piso",
        "observacion_piso",
        "cumple",
        "observacion_cumple",
        "requiere_revision_cumplimiento",
        "motivo_revision_cumplimiento",
        "decision_pago",
        "motivo_pago",
        "Valor",
        "Pagado",
        "Saldo",
        "comision_ejecutivo",
        "valor_comision_ejecutivo",
        "comision_jefe",
        "valor_comision_jefe",
    ]
    cols_base = [c for c in cols_base if c in base.columns]

    print(f"Factura: {tranid}")
    if lote_norm is not None:
        print(f"Lote: {lote_norm}")
    print(f"Filas encontradas: {len(base)}")

    display(
        base[cols_base].sort_values(
            [c for c in ["lote", "Servicio"] if c in base.columns],
            na_position="last",
        )
    )

    doc_base_vals = (
        base["doc_base_comision"]
        .dropna()
        .astype("string")
        .str.strip()
        .unique()
        .tolist()
        if "doc_base_comision" in base.columns else []
    )

    if doc_base_vals:
        lookup_docs = set(doc_base_vals + [tranid])

        base_origen = ventas_piso[
            ventas_piso["tranId"].astype("string").str.strip().isin(lookup_docs)
        ].copy()

        if lote_norm is not None and "lote" in base_origen.columns:
            base_origen = base_origen[
                base_origen["lote"].astype("string").str.strip().eq(lote_norm)
            ].copy()

        if not base_origen.empty:
            cols_origen = [
                "tranId",
                "lote",
                "Servicio",
                "documento_origen",
                "reemplazo",
                "tipo_reemplazo",
                "tranDate",
                "dueDate",
                "estado_piso",
                "TotalParaPiso_final",
                "observacion_piso",
            ]
            cols_origen = [c for c in cols_origen if c in base_origen.columns]

            print("\nDocumento actual / base de comisión en ventas_piso:")
            display(
                base_origen[cols_origen].sort_values(
                    [c for c in ["tranDate", "tranId", "lote", "Servicio"] if c in base_origen.columns],
                    na_position="last",
                )
            )

    cartera_match = cartera_raw[
        cartera_raw["Factura"].astype("string").str.strip().eq(tranid)
    ].copy()

    if not cartera_match.empty:
        cols_cartera = [
            "Factura",
            "Fecha Real Pago",
            "Fecha Contabilizado Pago",
            "Pagado Con (Tipo)",
            "Pagado Con:",
            "Pagado",
            "Saldo",
        ]
        cols_cartera = [c for c in cols_cartera if c in cartera_match.columns]

        print("\nFilas de cartera asociadas a la factura:")
        display(
            cartera_match[cols_cartera].sort_values(
                [c for c in ["Fecha Contabilizado Pago", "Fecha Real Pago"] if c in cartera_match.columns],
                na_position="last",
            )
        )
    else:
        print("\nNo se encontraron filas en cartera_raw para esta factura.")

    ejecutivo_vals = (
        base["entityid"].dropna().astype("string").str.strip().unique().tolist()
        if "entityid" in base.columns else []
    )
    periodo_vals = (
        base["periodo_mes"].dropna().astype("string").unique().tolist()
        if "periodo_mes" in base.columns else []
    )
    linea_vals = (
        base["linea_comision_key"].dropna().astype("string").str.strip().unique().tolist()
        if "linea_comision_key" in base.columns else []
    )

    if ejecutivo_vals and periodo_vals and linea_vals:
        cumpl_match = cumplimiento_final.copy()
        cumpl_match["periodo_mes_key"] = cumpl_match["periodo_mes"].astype("string")
        cumpl_match["entityid_key"] = cumpl_match["entityid"].astype("string").str.strip()
        cumpl_match["linea_key"] = cumpl_match["linea_comision"].astype("string").str.strip()

        cumpl_match = cumpl_match[
            cumpl_match["periodo_mes_key"].isin(periodo_vals)
            & cumpl_match["entityid_key"].isin(ejecutivo_vals)
            & cumpl_match["linea_key"].isin(linea_vals)
        ].copy()

        if not cumpl_match.empty:
            cols_cumpl = [
                "periodo_mes",
                "entityid",
                "linea_comision",
                "TotalParaPiso",
                "meta",
                "piso",
                "cumple",
                "observacion_cumple",
                "requiere_revision_cumplimiento",
                "motivo_revision_cumplimiento",
            ]
            cols_cumpl = [c for c in cols_cumpl if c in cumpl_match.columns]

            print("\nCruce de cumplimiento asociado:")
            display(
                cumpl_match[cols_cumpl].sort_values(
                    [c for c in ["periodo_mes", "entityid", "linea_comision"] if c in cumpl_match.columns],
                    na_position="last",
                )
            )

    if "lote" in para_pago_full.columns:
        print("\nLotes disponibles para la factura:")
        display(
            para_pago_full[
                para_pago_full["tranId"].astype("string").str.strip().eq(tranid)
            ][["tranId", "lote", "Servicio", "decision_pago", "motivo_pago"]]
            .drop_duplicates()
            .sort_values(
                [c for c in ["lote", "Servicio"] if c in para_pago_full.columns],
                na_position="last",
            )
        )

    return {
        "para_pago": base,
        "cartera": cartera_match,
    }

# Ejemplos:
# verificar_factura_pago("SM38195")
# verificar_factura_pago("SM38195", "LOTE-123")


In [122]:
# =========================================================
# HELPER - VER PISO, META Y CUMPLIMIENTO DE UN EJECUTIVO EN UN MES
# =========================================================

def ver_piso_ejecutivo_mes(entityid: str, periodo_mes: str):
    ejecutivo = str(entityid).strip()
    ejecutivo_key = ejecutivo.upper()
    periodo = str(periodo_mes).strip()

    print(f"Ejecutivo: {ejecutivo}")
    print(f"Periodo: {periodo}")

    # =========================
    # 1) Resultado en cumplimiento_final
    # =========================
    base = cumplimiento_final.copy()
    base["entityid_key"] = base["entityid"].astype("string").str.strip().str.upper()
    base["periodo_mes_key"] = base["periodo_mes"].astype("string").str.strip()

    out_cumpl = base[
        base["entityid_key"].eq(ejecutivo_key)
        & base["periodo_mes_key"].eq(periodo)
    ].copy()

    if out_cumpl.empty:
        print("\nNo se encontró resultado en cumplimiento_final.")
        return None

    # =========================
    # 2) Mostrar detalle por línea
    # =========================
    cols_cumpl = [
        "periodo_mes",
        "entityid",
        "linea_comision",
        "TotalParaPiso",
        "meta",
        "piso",
        "cumple",
        "observacion_cumple",
        "requiere_revision_cumplimiento",
        "motivo_revision_cumplimiento",
    ]
    cols_cumpl = [c for c in cols_cumpl if c in out_cumpl.columns]

    print("\nDetalle por línea:")
    display(
        out_cumpl[cols_cumpl].sort_values(
            [c for c in ["linea_comision"] if c in out_cumpl.columns],
            na_position="last",
        )
    )

    # =========================
    # 3) Resumen claro
    # =========================
    print("\nResumen por línea:")
    for _, row in out_cumpl.sort_values("linea_comision", na_position="last").iterrows():
        linea = row.get("linea_comision", pd.NA)
        total = row.get("TotalParaPiso", pd.NA)
        piso = row.get("piso", pd.NA)
        meta = row.get("meta", pd.NA)
        cumple = row.get("cumple", pd.NA)
        obs = row.get("observacion_cumple", "")

        estado = "Sí cumplió" if bool(cumple) else "No cumplió"

        print(
            f"- {linea}: {estado} | "
            f"Total={total} | Piso={piso} | Meta={meta} | "
            f"Obs={obs}"
        )

    # =========================
    # 4) Resumen ejecutivo del período
    # =========================
    cumple_alguna = out_cumpl["cumple"].fillna(False).astype(bool).any()
    total_lineas = len(out_cumpl)

    print("\nConclusión del período:")
    if cumple_alguna:
        print(f"- El ejecutivo sí cumplió al menos una línea en {periodo}.")
    else:
        print(f"- El ejecutivo no cumplió ninguna línea en {periodo}.")

    print(f"- Líneas evaluadas: {total_lineas}")

    return out_cumpl


In [123]:

# Ejemplo:
ver_piso_ejecutivo_mes("ADRIANA MARITZA QUITIAN ROZO", "2025-04")

Ejecutivo: ADRIANA MARITZA QUITIAN ROZO
Periodo: 2025-04

Detalle por línea:


,periodo_mes,entityid,linea_comision,TotalParaPiso,meta,piso,cumple,observacion_cumple,requiere_revision_cumplimiento,motivo_revision_cumplimiento
3726,2025-04,ADRIANA MARITZA QUITIAN ROZO,FRONTERAS,3.900000e+06,3500000.0,3500000.0,True,Cumple meta,False,
3727,2025-04,ADRIANA MARITZA QUITIAN ROZO,NA,1.200000e+05,NaN,NaN,True,No cumple: no alcanzó el mayor piso configurad...,False,
3728,2025-04,ADRIANA MARITZA QUITIAN ROZO,RETIE/LAP,1.448062e+08,132650000.0,106120000.0,True,Cumple meta,False,
3729,2025-04,ADRIANA MARITZA QUITIAN ROZO,RITEL,7.708800e+06,12150000.0,12150000.0,True,Cumple por regla RETIE/LAP: al cumplir RETIE/L...,False,



Resumen por línea:
- FRONTERAS: Sí cumplió | Total=3900000.0 | Piso=3500000.0 | Meta=3500000.0 | Obs=Cumple meta
- NA: Sí cumplió | Total=120000.0 | Piso=nan | Meta=nan | Obs=No cumple: no alcanzó el mayor piso configurado del ejecutivo en el mes
- RETIE/LAP: Sí cumplió | Total=144806150.24 | Piso=106120000.0 | Meta=132650000.0 | Obs=Cumple meta
- RITEL: Sí cumplió | Total=7708799.99 | Piso=12150000.0 | Meta=12150000.0 | Obs=Cumple por regla RETIE/LAP: al cumplir RETIE/LAP se pagan las demás líneas configuradas del ejecutivo en el mes

Conclusión del período:
- El ejecutivo sí cumplió al menos una línea en 2025-04.
- Líneas evaluadas: 4


,periodo_mes,entityid,linea_comision,TotalParaPiso,costo,piso,meta,cumple,observacion_cumple,requiere_revision_cumplimiento,motivo_revision_cumplimiento,entityid_key,periodo_mes_key
3726,2025-04,ADRIANA MARITZA QUITIAN ROZO,FRONTERAS,3.900000e+06,307000.00,3500000.0,3500000.0,True,Cumple meta,False,,ADRIANA MARITZA QUITIAN ROZO,2025-04
3727,2025-04,ADRIANA MARITZA QUITIAN ROZO,NA,1.200000e+05,35286.00,NaN,NaN,True,No cumple: no alcanzó el mayor piso configurad...,False,,ADRIANA MARITZA QUITIAN ROZO,2025-04
3728,2025-04,ADRIANA MARITZA QUITIAN ROZO,RETIE/LAP,1.448062e+08,55160984.35,106120000.0,132650000.0,True,Cumple meta,False,,ADRIANA MARITZA QUITIAN ROZO,2025-04
3729,2025-04,ADRIANA MARITZA QUITIAN ROZO,RITEL,7.708800e+06,586824.14,12150000.0,12150000.0,True,Cumple por regla RETIE/LAP: al cumplir RETIE/L...,False,,ADRIANA MARITZA QUITIAN ROZO,2025-04


In [124]:
# =========================================================
# HELPER - VER FACTURAS DE UN EJECUTIVO EN UN MES
# =========================================================

def ver_facturas_ejecutivo_mes(entityid: str, periodo_mes: str):
    ejecutivo = str(entityid).strip()
    ejecutivo_key = ejecutivo.upper()
    periodo = str(periodo_mes).strip()

    base = ventas_piso.copy()

    if "periodo_factura" not in base.columns:
        base["periodo_factura"] = pd.to_datetime(base["tranDate"], errors="coerce").dt.to_period("M")

    base["periodo_mes_key"] = base["periodo_factura"].astype("string").str.strip()
    base["entityid_key"] = base["entityid"].astype("string").str.strip().str.upper()

    out = base[
        base["entityid_key"].eq(ejecutivo_key)
        & base["periodo_mes_key"].eq(periodo)
    ].copy()

    if out.empty:
        print(f"No se encontraron facturas para {ejecutivo} en {periodo}.")
        return None

    cols = [
        "tranId",
        "tranDate",
        "dueDate",
        "entityid",
        "Servicio",
        "cc",
        "lote",
        "custitem_sm_lineacomision",
        "Valor",
        "estado_piso",
        "TotalParaPiso_base",
        "TotalParaPiso_final",
        "observacion_piso",
        "documento_origen",
        "reemplazo",
        "tipo_reemplazo",
        "tranId_nc",
        "tranDate_nc",
        "Valor_nc",
        "tipo_afectacion_nc",
        "requiere_recalculo_piso",
        "periodo_recalculo",
    ]
    cols = [c for c in cols if c in out.columns]

    out = out.sort_values(
        [c for c in ["tranDate", "tranId", "lote", "Servicio"] if c in out.columns],
        na_position="last",
    )

    print(f"Ejecutivo: {ejecutivo}")
    print(f"Período: {periodo}")
    print(f"Filas encontradas: {len(out)}")
    print(f"Facturas distintas: {out['tranId'].nunique() if 'tranId' in out.columns else len(out)}")

    display(out[cols])

    return out

# Ejemplo:
ver_facturas_ejecutivo_mes("ADRIANA MARITZA QUITIAN ROZO", "2025-04")

Ejecutivo: ADRIANA MARITZA QUITIAN ROZO
Período: 2025-04
Filas encontradas: 52
Facturas distintas: 47


,tranId,tranDate,dueDate,entityid,Servicio,cc,lote,custitem_sm_lineacomision,Valor,estado_piso,TotalParaPiso_base,TotalParaPiso_final,observacion_piso,documento_origen,reemplazo,tipo_reemplazo,tranId_nc,tranDate_nc,Valor_nc,tipo_afectacion_nc,requiere_recalculo_piso,periodo_recalculo
39691,SM57594,2025-04-01,2025-04-01,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL ALTA CONCENTRACION ...,RETIE BOGOTA,181018,10,3100000.00,suma,3100000.00,3100000.00,Suma para piso,<NA>,False,ninguno,<NA>,NaT,NaN,ninguna,False,<NA>
58798,SM57643,2025-04-02,2025-04-02,ADRIANA MARITZA QUITIAN ROZO,REIMPRESION DICTAMEN RETIE/RETILAP,RETIE BOGOTA,181093,19,30000.00,suma,30000.00,30000.00,Suma para piso,<NA>,False,ninguno,<NA>,NaT,NaN,ninguna,False,<NA>
18124,SM57647,2025-04-03,2025-04-03,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL SISTEMAS FOTOVOLTAICOS,RETIE BARRANQUILLA,180937,10,1497000.00,suma,1497000.00,1497000.00,Suma para piso,<NA>,False,ninguno,<NA>,NaT,NaN,ninguna,False,<NA>
57273,SM57671,2025-04-04,2025-04-04,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL HOSPITALARIAS PERIO...,RETIE BOGOTA,179882,10,800000.00,no suma,800000.00,0.00,No suma por ser reemplazo de SM57469,SM57469,True,reemplazo_factura_nc,<NA>,NaT,NaN,ninguna,False,<NA>
4459,SM57672,2025-04-04,2025-04-04,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL BASICAS,RETIE BOGOTA,181116,10,1550000.00,suma,1550000.00,1550000.00,Suma para piso,<NA>,False,ninguno,<NA>,NaT,NaN,ninguna,False,<NA>
56031,SM57758,2025-04-07,2025-04-07,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL AREAS CLASIFICADAS ...,RETIE BOGOTA,181311,10,4600000.00,suma,4600000.00,4600000.00,Suma para piso,<NA>,False,ninguno,<NA>,NaT,NaN,ninguna,False,<NA>
77384,SM57761,2025-04-08,2025-05-08,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETILAP INTERIORES,RETILAP BOGOTA,178273,10,4600000.00,no suma,4600000.00,0.00,No suma por ser reemplazo de PR-8726,PR-8726,True,reemplazo_proforma_createdfrom,<NA>,NaT,NaN,ninguna,False,<NA>
77414,SM57762,2025-04-08,2025-05-08,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETILAP INTERIORES,RETILAP BOGOTA,178271,10,4600000.00,no suma,4600000.00,0.00,No suma por ser reemplazo de PR-8727,PR-8727,True,reemplazo_proforma_createdfrom,<NA>,NaT,NaN,ninguna,False,<NA>
10795,SM57798,2025-04-09,2025-05-09,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL BASICAS,RETIE BOGOTA,176832,10,8836800.00,no suma,8836800.00,0.00,No suma por ser reemplazo de PR-8685,PR-8685,True,reemplazo_proforma_createdfrom,<NA>,NaT,NaN,ninguna,False,<NA>
56465,ND0069,2025-04-10,2025-04-10,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL BASICAS PERIODICA,RETIE BOGOTA,179269,10,3400000.00,no suma,3400000.00,0.00,No suma por ser reemplazo de SM56870,SM56870,True,reemplazo_fallback,<NA>,NaT,NaN,ninguna,False,<NA>


,tranDate,dueDate,tranId,entity,entityid,Servicio,Precio,quantity,Valor,createdfrom,id,status,custbody_sm_factura_reemplazo,custbody_ks_sm_proforma,type,custitem_sm_lineacomision,cc,lote,custbody_sm_reemplaza_a,memo,origen_documento,es_factura,es_proforma,periodo_factura,Servicio_key,lote_key,valor_abs,createdfrom_key,id_key,reemplaza_a_key,factura_reemplazo_key,__rowid,Valor_nc,tranId_nc,tranDate_nc,factura_reemplazo_key_nc,factura_reemplazo_tranId_nc,tipo_match_nc,hay_nc_del_serv,nc_count,alarma_nc_servicio,motivo_alarma_nc,documento_origen,fecha_origen,documento_reemplazado,tipo_reemplazo,reemplazo,observacion_reemplazo,periodo_nc,documento_anulado,tipo_afectacion_nc,requiere_recalculo_piso,periodo_recalculo,observacion_nc,tranId_key,tranId_nc_ncp,tranDate_nc_ncp,tranId_nc_cpf,tranDate_nc_cpf,TotalParaPiso_base,estado_piso,TotalParaPiso_final,observacion_piso,periodo_mes_key,entityid_key
39691,2025-04-01,2025-04-01,SM57594,104272,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL ALTA CONCENTRACION ...,3100000.00,-1.0,3100000.00,1803409,1803519,B,0,True,CustInvc,10,RETIE BOGOTA,181018,0,COTIZACION 181018 DEL PROYECTO OFICINAS EDF PI...,factura,True,False,2025-04,INSPECCION RETIE USO FINAL ALTA CONCENTRACION ...,181018,3100000.00,1803409,1803519,0,0,39691,NaN,<NA>,NaT,<NA>,NaN,NaN,False,0,False,,<NA>,NaT,<NA>,ninguno,False,,NaT,False,ninguna,False,<NA>,,SM57594,<NA>,NaT,<NA>,NaT,3100000.00,suma,3100000.00,Suma para piso,2025-04,ADRIANA MARITZA QUITIAN ROZO
58798,2025-04-02,2025-04-02,SM57643,179338,ADRIANA MARITZA QUITIAN ROZO,REIMPRESION DICTAMEN RETIE/RETILAP,30000.00,-1.0,30000.00,1804706,1804736,B,0,True,CustInvc,19,RETIE BOGOTA,181093,0,Reimpresión dictamen asociado a la oferta #179...,factura,True,False,2025-04,REIMPRESION DICTAMEN RETIE/RETILAP,181093,30000.00,1804706,1804736,0,0,58798,NaN,<NA>,NaT,<NA>,NaN,NaN,False,0,False,,<NA>,NaT,<NA>,ninguno,False,,NaT,False,ninguna,False,<NA>,,SM57643,<NA>,NaT,<NA>,NaT,30000.00,suma,30000.00,Suma para piso,2025-04,ADRIANA MARITZA QUITIAN ROZO
18124,2025-04-03,2025-04-03,SM57647,15953,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL SISTEMAS FOTOVOLTAICOS,1497000.00,-1.0,1497000.00,1803692,1804742,B,0,True,CustInvc,10,RETIE BARRANQUILLA,180937,0,COTIZACION 180937 DEL PROYECTO SISTEMA FOTOVOL...,factura,True,False,2025-04,INSPECCION RETIE USO FINAL SISTEMAS FOTOVOLTAICOS,180937,1497000.00,1803692,1804742,0,0,18124,NaN,<NA>,NaT,<NA>,NaN,NaN,False,0,False,,<NA>,NaT,<NA>,ninguno,False,,NaT,False,ninguna,False,<NA>,,SM57647,<NA>,NaT,<NA>,NaT,1497000.00,suma,1497000.00,Suma para piso,2025-04,ADRIANA MARITZA QUITIAN ROZO
57273,2025-04-04,2025-04-04,SM57671,192230,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL HOSPITALARIAS PERIO...,800000.00,-1.0,800000.00,1805555,1805876,B,0,True,CustInvc,10,RETIE BOGOTA,179882,0,COTIZACION 179882 DEL PROYECTO CONSULTORIO 1019,factura,True,False,2025-04,INSPECCION RETIE USO FINAL HOSPITALARIAS PERIO...,179882,800000.00,1805555,1805876,0,0,57273,NaN,<NA>,NaT,<NA>,NaN,NaN,False,0,False,,SM57469,2025-03-26,SM57469,reemplazo_factura_nc,True,No suma por ser reemplazo de SM57469,NaT,False,ninguna,False,<NA>,,SM57671,<NA>,NaT,<NA>,NaT,800000.00,no suma,0.00,No suma por ser reemplazo de SM57469,2025-04,ADRIANA MARITZA QUITIAN ROZO
4459,2025-04-04,2025-04-04,SM57672,63803,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL BASICAS,1550000.00,-1.0,1550000.00,1805525,1805888,B,0,True,CustInvc,10,RETIE BOGOTA,181116,0,COTIZACION 181116 DEL PROYECTO CLARET REAL VI,factura,True,False,2025-04,INSPECCION RETIE USO FINAL BASICAS,181116,1550000.00,1805525,1805888,0,0,4459,NaN,<NA>,NaT,<NA>,NaN,NaN,False,0,False,,<NA>,NaT,<NA>,ninguno,False,,NaT,False,ninguna,False,<NA>,,SM57672,<NA>,NaT,<NA>,NaT,1550000.00,suma,1550000.00,Suma para piso,2025-04,ADRIANA MARITZA QUITIAN ROZO
56031,2025-04-07,2025-04-07,SM57758,26412,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL AREAS CLASIFICADAS ...,4600000.00,-1.0,4600000.00,1807813,1808479,B,0,True,C

In [125]:

# Ejemplo:
verificar_factura_pago("SM58100")

Factura: SM58100
Filas encontradas: 1


,tranId,lote,Servicio,doc_base_comision,documento_origen,entityid,cc,linea_comision_key,tranDate,dueDate,dueDate_base_pago,Fecha Real Pago,Fecha Contabilizado Pago,Inicio Corte Contable,Fin Corte Contable,Periodo Corte,Cumple Corte Contable,dias_pago,factor,estado_piso,observacion_piso,cumple,observacion_cumple,requiere_revision_cumplimiento,motivo_revision_cumplimiento,decision_pago,motivo_pago,Valor,Pagado,Saldo,comision_ejecutivo,valor_comision_ejecutivo,comision_jefe,valor_comision_jefe
19993,SM58100,181476,INSPECCION ASCENSOR 8-14 PARADAS,SM58100,<NA>,JULY DANIELA MURCIA PEDRAZA,ESCALERAS Y ASCENSORES BOGOTA,ESCAS,2025-04-14,2025-04-14,2025-04-14,2025-04-30,2025-04-30,2025-04-22,2025-05-20,2025-05,True,16.0,1.0,suma,Suma para piso,True,Cumple meta,False,,PAGA 100%,Cumple pago <= 30 días desde dueDate,540000.0,642600,0,0.04,21600.0,0.01,5400.0



Documento actual / base de comisión en ventas_piso:


,tranId,lote,Servicio,documento_origen,reemplazo,tipo_reemplazo,tranDate,dueDate,estado_piso,TotalParaPiso_final,observacion_piso
19993,SM58100,181476,INSPECCION ASCENSOR 8-14 PARADAS,<NA>,False,ninguno,2025-04-14,2025-04-14,suma,540000.0,Suma para piso



Filas de cartera asociadas a la factura:


,Factura,Fecha Real Pago,Fecha Contabilizado Pago,Pagado Con (Tipo),Pagado Con:,Pagado,Saldo
19778,SM58100,2025-04-30,2025-04-30,Pago,Pago #35296,642600,0



Cruce de cumplimiento asociado:


,periodo_mes,entityid,linea_comision,TotalParaPiso,meta,piso,cumple,observacion_cumple,requiere_revision_cumplimiento,motivo_revision_cumplimiento
3763,2025-04,JULY DANIELA MURCIA PEDRAZA,ESCAS,26375200.0,25960000.0,25960000.0,True,Cumple meta,False,



Lotes disponibles para la factura:


,tranId,lote,Servicio,decision_pago,motivo_pago
19993,SM58100,181476,INSPECCION ASCENSOR 8-14 PARADAS,PAGA 100%,Cumple pago <= 30 días desde dueDate


{'para_pago':         tranDate    dueDate   tranId  entity                     entityid                          Servicio    Precio  quantity     Valor createdfrom       id status custbody_sm_factura_reemplazo  \
 19993 2025-04-14 2025-04-14  SM58100   35447  JULY DANIELA MURCIA PEDRAZA  INSPECCION ASCENSOR 8-14 PARADAS  270000.0      -2.0  540000.0     1811137  1813180      B                             0   
 
        custbody_ks_sm_proforma      type custitem_sm_lineacomision                             cc    lote custbody_sm_reemplaza_a  memo origen_documento  es_factura  es_proforma periodo_factura  \
 19993                     True  CustInvc                         9  ESCALERAS Y ASCENSORES BOGOTA  181476                       0  <NA>          factura        True        False         2025-04   
 
                            Servicio_key lote_key  valor_abs createdfrom_key   id_key reemplaza_a_key factura_reemplazo_key  __rowid  Valor_nc tranId_nc tranDate_nc factura_reemplazo_key_

In [126]:
# =========================================================
# HELPER - AUDITORIA COMPLETA POR LOTE
# =========================================================

def verificar_lote(lote: str, documento: str | None = None):
    lote_norm = str(lote).strip()
    doc_norm = None if documento is None else str(documento).strip().upper()

    print(f"Lote: {lote_norm}")
    if doc_norm:
        print(f"Documento filtro: {doc_norm}")

    # =========================
    # 1) Ventas piso
    # =========================
    vp = ventas_piso.copy()
    vp["lote_key"] = vp["lote"].astype("string").str.strip()
    vp["tranId_key"] = vp["tranId"].astype("string").str.strip().str.upper()

    vp = vp[vp["lote_key"].eq(lote_norm)].copy()
    if doc_norm is not None:
        vp = vp[vp["tranId_key"].eq(doc_norm)].copy()

    if vp.empty:
        print("\nNo hay filas en ventas_piso para ese lote/documento.")
    else:
        print("\n1) Detalle en ventas_piso:")
        cols_vp = [
            "tranId",
            "lote",
            "Servicio",
            "entityid",
            "tranDate",
            "dueDate",
            "es_proforma",
            "es_factura",
            "documento_origen",
            "reemplazo",
            "tipo_reemplazo",
            "tranId_nc",
            "tipo_afectacion_nc",
            "estado_piso",
            "TotalParaPiso_base",
            "TotalParaPiso_final",
            "observacion_piso",
            "custitem_sm_lineacomision",
        ]
        cols_vp = [c for c in cols_vp if c in vp.columns]
        display(
            vp[cols_vp].sort_values(
                [c for c in ["tranDate", "tranId", "Servicio"] if c in vp.columns],
                na_position="last",
            )
        )

        print("\nResumen ventas_piso:")
        resumen_vp = (
            vp.groupby(["tranId", "estado_piso"], dropna=False, as_index=False)
            .agg(
                valor=("Valor", "sum"),
                total_para_piso=("TotalParaPiso_final", "sum"),
                servicios=("Servicio", "nunique"),
            )
            .sort_values(["tranId", "estado_piso"], na_position="last")
        )
        display(resumen_vp)

    # =========================
    # 2) Costo del lote
    # =========================
    cw = costo.copy()
    cw["lote_key"] = cw["lote"].astype("string").str.strip()
    cw["costo_abs"] = pd.to_numeric(cw["Valor"], errors="coerce").abs()

    cw = cw[cw["lote_key"].eq(lote_norm)].copy()

    if cw.empty:
        print("\n2) No se encontró costo para el lote.")
    else:
        print("\n2) Costo encontrado para el lote:")
        display(cw)

        print("\nResumen costo:")
        display(
            cw.groupby("lote_key", as_index=False)
            .agg(
                costo_original=("Valor", "sum"),
                costo_abs=("costo_abs", "sum"),
                filas=("lote_key", "size"),
            )
        )

    # =========================
    # 3) Cumplimiento asociado
    # =========================
    if not vp.empty:
        periodos = vp["periodo_factura"].astype("string").dropna().unique().tolist() if "periodo_factura" in vp.columns else []
        ejecutivos = vp["entityid"].astype("string").str.strip().dropna().unique().tolist()

        # si en ventas_piso no está linea_comision en texto, la reconstruimos
        rel = pd.DataFrame({
            "ID INTERNO": [14, 13, 7, 8, 18, 9, 17, 16, 2, 1, 3, 6, 5, 19, 12, 10, 11, 15, 4],
            "NOMBRE": [
                "ADMON", "CALIBRACION", "CAPACITACION", "CERTIFICACION", "CSG",
                "ESCAS", "FRONTERAS", "INTER", "LMA", "LMAM", "LME", "LVM",
                "LVV", "NA", "PRUEBAS", "RETIE/LAP", "RIESGO", "RITEL",
                "VARIABLES ELECTRICAS"
            ],
        })
        rel["codigo_key"] = rel["ID INTERNO"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
        rel["linea_key"] = rel["NOMBRE"].astype("string").str.strip().str.upper()

        vp_lineas = vp.copy()
        vp_lineas["codigo_key"] = vp_lineas["custitem_sm_lineacomision"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
        vp_lineas = vp_lineas.merge(
            rel[["codigo_key", "linea_key"]],
            on="codigo_key",
            how="left",
            validate="m:1",
        )

        lineas = vp_lineas["linea_key"].dropna().astype("string").str.strip().unique().tolist()

        cf = cumplimiento_final.copy()
        cf["periodo_key"] = cf["periodo_mes"].astype("string").str.strip()
        cf["entityid_key"] = cf["entityid"].astype("string").str.strip()
        cf["linea_key"] = cf["linea_comision"].astype("string").str.strip().str.upper()

        cf = cf[
            cf["periodo_key"].isin(periodos)
            & cf["entityid_key"].isin(ejecutivos)
            & cf["linea_key"].isin(lineas)
        ].copy()

        if cf.empty:
            print("\n3) No se encontró cruce en cumplimiento_final.")
        else:
            print("\n3) Cumplimiento asociado al lote:")
            cols_cf = [
                "periodo_mes",
                "entityid",
                "linea_comision",
                "TotalParaPiso",
                "costo",
                "TotalParaPiso_menos_costo",
                "piso",
                "meta",
                "cumple",
                "observacion_cumple",
                "requiere_revision_cumplimiento",
                "motivo_revision_cumplimiento",
            ]
            cols_cf = [c for c in cols_cf if c in cf.columns]
            display(
                cf[cols_cf].sort_values(
                    [c for c in ["periodo_mes", "entityid", "linea_comision"] if c in cf.columns],
                    na_position="last",
                )
            )

    # =========================
    # 4) Pago asociado
    # =========================
    if "para_pago_full" in globals():
        pp = para_pago_full.copy()
        pp["lote_key"] = pp["lote"].astype("string").str.strip()
        pp["tranId_key"] = pp["tranId"].astype("string").str.strip().str.upper()

        pp = pp[pp["lote_key"].eq(lote_norm)].copy()
        if doc_norm is not None:
            pp = pp[pp["tranId_key"].eq(doc_norm)].copy()

        if pp.empty:
            print("\n4) No hay filas en para_pago_full para ese lote/documento.")
        else:
            print("\n4) Resultado en para_pago_full:")
            cols_pp = [
                "tranId",
                "lote",
                "Servicio",
                "entityid",
                "linea_comision_key",
                "tranDate",
                "dueDate",
                "dueDate_base_pago",
                "Fecha Real Pago",
                "Fecha Contabilizado Pago",
                "Periodo Corte",
                "dias_pago",
                "factor",
                "estado_piso",
                "cumple",
                "costo",
                "TotalParaPiso",
                "TotalParaPiso_menos_costo",
                "decision_pago",
                "motivo_pago",
            ]
            cols_pp = [c for c in cols_pp if c in pp.columns]
            display(
                pp[cols_pp].sort_values(
                    [c for c in ["tranDate", "tranId", "Servicio"] if c in pp.columns],
                    na_position="last",
                )
            )

    return True

# Ejemplos:
verificar_lote("181849")
# verificar_lote("181849", "PR-8858")

Lote: 181849

1) Detalle en ventas_piso:


,tranId,lote,Servicio,entityid,tranDate,dueDate,es_proforma,es_factura,documento_origen,reemplazo,tipo_reemplazo,tranId_nc,tipo_afectacion_nc,estado_piso,TotalParaPiso_base,TotalParaPiso_final,observacion_piso,custitem_sm_lineacomision
85313,PR-8858,181849,INSPECCION RETIE USO FINAL BASICAS,ADRIANA MARITZA QUITIAN ROZO,2025-04-29,2025-06-29,True,False,<NA>,False,ninguno,<NA>,ninguna,suma,1330000.0,1330000.0,Suma para piso,10
15266,SM60761,181849,INSPECCION RETIE USO FINAL BASICAS,ADRIANA MARITZA QUITIAN ROZO,2025-07-14,2025-08-13,False,True,PR-8858,True,reemplazo_proforma_createdfrom,<NA>,ninguna,no suma,1330000.0,0.0,No suma por ser reemplazo de PR-8858,10



Resumen ventas_piso:


,tranId,estado_piso,valor,total_para_piso,servicios
0,PR-8858,suma,1330000.0,1330000.0,1
1,SM60761,no suma,1330000.0,0.0,1



2) Costo encontrado para el lote:


,lote,Valor,lote_key,costo_abs
10308,181849,-399000.0,181849,399000.0


... (3 outputs left out)

,periodo_mes,entityid,linea_comision,TotalParaPiso,costo,piso,meta,cumple,observacion_cumple,requiere_revision_cumplimiento,motivo_revision_cumplimiento
3728,2025-04,ADRIANA MARITZA QUITIAN ROZO,RETIE/LAP,1.448062e+08,55160984.35,106120000.0,132650000.0,True,Cumple meta,False,
3974,2025-07,ADRIANA MARITZA QUITIAN ROZO,RETIE/LAP,2.456837e+08,58140180.24,138600000.0,173250000.0,True,Cumple meta,False,



4) Resultado en para_pago_full:


,tranId,lote,Servicio,entityid,linea_comision_key,tranDate,dueDate,dueDate_base_pago,Fecha Real Pago,Fecha Contabilizado Pago,Periodo Corte,dias_pago,factor,estado_piso,cumple,costo,TotalParaPiso,decision_pago,motivo_pago
85313,PR-8858,181849,INSPECCION RETIE USO FINAL BASICAS,ADRIANA MARITZA QUITIAN ROZO,RETIE/LAP,2025-04-29,2025-06-29,2025-06-29,NaT,NaT,<NA>,NaN,NaN,suma,True,55160984.35,1.448062e+08,NO PAGA,Es proforma; no paga comisión
15266,SM60761,181849,INSPECCION RETIE USO FINAL BASICAS,ADRIANA MARITZA QUITIAN ROZO,RETIE/LAP,2025-07-14,2025-08-13,2025-06-29,2025-08-25,2025-08-26,2025-09,57.0,0.5,no suma,True,58140180.24,2.456837e+08,NO PAGA,No suma por ser reemplazo de PR-8858


True

In [127]:
# =========================================================
# HELPER - VER LOTES Y COSTOS DE UN EJECUTIVO EN UN MES
# =========================================================

def ver_lotes_costos_ejecutivo_mes(entityid: str, periodo_mes: str):
    ejecutivo = str(entityid).strip()
    ejecutivo_key = ejecutivo.upper()
    periodo = str(periodo_mes).strip()

    base = ventas_piso.copy()
    base["entityid_key"] = base["entityid"].astype("string").str.strip().str.upper()

    if "periodo_factura" not in base.columns:
        base["periodo_factura"] = pd.to_datetime(base["tranDate"], errors="coerce").dt.to_period("M")

    base["periodo_mes_key"] = base["periodo_factura"].astype("string").str.strip()
    base["lote_key"] = base["lote"].astype("string").str.strip()

    base = base[
        base["entityid_key"].eq(ejecutivo_key)
        & base["periodo_mes_key"].eq(periodo)
    ].copy()

    if base.empty:
        print(f"No se encontraron filas para {ejecutivo} en {periodo}.")
        return None

    # costos
    cw = costo.copy()
    cw.columns = [str(c).strip() for c in cw.columns]
    cw["lote_key"] = cw["lote"].astype("string").str.strip()
    cw["costo_original"] = pd.to_numeric(cw["Valor"], errors="coerce")
    cw["costo_abs"] = cw["costo_original"].abs()

    costo_lote = (
        cw.groupby("lote_key", as_index=False)
        .agg(
            costo_original=("costo_original", "sum"),
            costo_abs=("costo_abs", "sum"),
        )
    )

    # primer periodo del lote en ventas_piso
    primer_periodo = (
        ventas_piso.copy()
        .assign(
            periodo_piso=lambda d: pd.to_datetime(d["tranDate"], errors="coerce").dt.to_period("M"),
            lote_key=lambda d: d["lote"].astype("string").str.strip(),
        )[["lote_key", "periodo_piso"]]
        .dropna(subset=["periodo_piso"])
        .sort_values(["lote_key", "periodo_piso"])
        .drop_duplicates(subset=["lote_key"], keep="first")
        .rename(columns={"periodo_piso": "primer_periodo_lote"})
    )

    detalle = base.merge(
        costo_lote,
        on="lote_key",
        how="left",
        validate="m:1",
    ).merge(
        primer_periodo,
        on="lote_key",
        how="left",
        validate="m:1",
    )

    detalle["costo_aplicaria_en_este_mes"] = np.where(
        detalle["primer_periodo_lote"].astype("string").eq(periodo),
        detalle["costo_abs"].fillna(0),
        0,
    )

    print(f"Ejecutivo: {ejecutivo}")
    print(f"Periodo: {periodo}")

    print("\nDetalle por factura / servicio / lote:")
    cols_detalle = [
        "tranId",
        "tranDate",
        "lote",
        "Servicio",
        "Valor",
        "estado_piso",
        "TotalParaPiso_final",
        "observacion_piso",
        "costo_original",
        "costo_abs",
        "primer_periodo_lote",
        "costo_aplicaria_en_este_mes",
    ]
    cols_detalle = [c for c in cols_detalle if c in detalle.columns]

    display(
        detalle[cols_detalle].sort_values(
            [c for c in ["lote", "tranDate", "tranId", "Servicio"] if c in detalle.columns],
            na_position="last",
        )
    )

    print("\nResumen por lote:")
    resumen_lote = (
        detalle.groupby(
            ["lote_key", "primer_periodo_lote"],
            dropna=False,
            as_index=False,
        )
        .agg(
            facturas=("tranId", lambda s: " | ".join(sorted(set(s.astype("string"))))),
            valor_factura=("Valor", "sum"),
            total_para_piso=("TotalParaPiso_final", "sum"),
            costo_original=("costo_original", "max"),
            costo_abs=("costo_abs", "max"),
            costo_aplicaria_en_este_mes=("costo_aplicaria_en_este_mes", "max"),
            servicios=("Servicio", "nunique"),
        )
        .sort_values(["lote_key"], na_position="last")
    )

    display(resumen_lote)

    return detalle

# Ejemplo:
ver_lotes_costos_ejecutivo_mes("ADRIANA MARITZA QUITIAN ROZO", "2025-04")

Ejecutivo: ADRIANA MARITZA QUITIAN ROZO
Periodo: 2025-04

Detalle por factura / servicio / lote:


,tranId,tranDate,lote,Servicio,Valor,estado_piso,TotalParaPiso_final,observacion_piso,costo_original,costo_abs,primer_periodo_lote,costo_aplicaria_en_este_mes
48,PR-8848,2025-04-29,105658,INSPECCION RETILAP INTERNAS BASICAS,1817707.56,no suma,0.00,No suma por anulación NC-000863 del mismo periodo,-59253.00,59253.00,2022-06,0.00
19,SM58478,2025-04-29,105658,INSPECCION RETILAP INTERNAS BASICAS,3468907.56,no suma,0.00,No suma por nota crédito NC4148 del mismo periodo,-59253.00,59253.00,2022-06,0.00
24,SM58481,2025-04-29,105658,INSPECCION RETILAP INTERNAS BASICAS,3468907.56,no suma,0.00,No suma por ser reemplazo de SM57115,-59253.00,59253.00,2022-06,0.00
46,PR-8850,2025-04-29,119283,INSPECCION RETILAP INTERNAS BASICAS,4950000.00,suma,4950000.00,Suma para piso,-640423.80,640423.80,2025-04,640423.80
25,SM58017,2025-04-11,132113,INSPECCION RETILAP INTERNAS BASICAS,3720000.00,suma,3720000.00,Suma para piso,NaN,NaN,2024-12,0.00
11,SM58017,2025-04-11,133681,INSPECCION RETIE USO FINAL BASICAS,1803121.00,suma,1803121.00,Suma para piso,-149056.00,149056.00,2024-06,0.00
1,SM58017,2025-04-11,133683,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,2700000.00,suma,2700000.00,Suma para piso,NaN,NaN,2023-06,0.00
7,SM58092,2025-04-14,149904,INSPECCION RETIE USO FINAL BASICAS,17100000.00,no suma,0.00,No suma por ser reemplazo de SM57570,-7360109.01,7360109.01,2024-01,0.00
8,SM58276,2025-04-22,149904,INSPECCION RETIE USO FINAL BASICAS,15663025.21,no suma,0.00,No suma por ser reemplazo de SM58092,-7360109.01,7360109.01,2024-01,0.00
21,SM58092,2025-04-14,149911,INSPECCION RETILAP INTERNAS BASICAS,6600000.00,no suma,0.00,No suma por ser reemplazo de SM57570,-22254.00,22254.00,2024-01,0.00



Resumen por lote:


,lote_key,primer_periodo_lote,facturas,valor_factura,total_para_piso,costo_original,costo_abs,costo_aplicaria_en_este_mes,servicios
0,105658,2022-06,PR-8848 | SM58478 | SM58481,8755522.68,0.00,-59253.00,59253.00,0.00,1
1,119283,2025-04,PR-8850,4950000.00,4950000.00,-640423.80,640423.80,640423.80,1
2,132113,2024-12,SM58017,3720000.00,3720000.00,NaN,NaN,0.00,1
3,133681,2024-06,SM58017,1803121.00,1803121.00,-149056.00,149056.00,0.00,1
4,133683,2023-06,SM58017,2700000.00,2700000.00,NaN,NaN,0.00,1
5,149904,2024-01,SM58092 | SM58276,32763025.21,0.00,-7360109.01,7360109.01,0.00,1
6,149911,2024-01,SM58092 | SM58276,12645378.15,0.00,-22254.00,22254.00,0.00,1
7,152775,2024-02,PR-8851,17016209.00,17016209.00,NaN,NaN,0.00,1
8,157441,2024-04,PR-8857,8084333.00,8084333.00,-6725736.62,6725736.62,0.00,1
9,161960,2024-05,SM57953,3150000.00,0.00,NaN,NaN,0.00,1


,tranDate,dueDate,tranId,entity,entityid,Servicio,Precio,quantity,Valor,createdfrom,id,status,custbody_sm_factura_reemplazo,custbody_ks_sm_proforma,type,custitem_sm_lineacomision,cc,lote,custbody_sm_reemplaza_a,memo,origen_documento,es_factura,es_proforma,periodo_factura,Servicio_key,lote_key,valor_abs,createdfrom_key,id_key,reemplaza_a_key,factura_reemplazo_key,__rowid,Valor_nc,tranId_nc,tranDate_nc,factura_reemplazo_key_nc,factura_reemplazo_tranId_nc,tipo_match_nc,hay_nc_del_serv,nc_count,alarma_nc_servicio,motivo_alarma_nc,documento_origen,fecha_origen,documento_reemplazado,tipo_reemplazo,reemplazo,observacion_reemplazo,periodo_nc,documento_anulado,tipo_afectacion_nc,requiere_recalculo_piso,periodo_recalculo,observacion_nc,tranId_key,tranId_nc_ncp,tranDate_nc_ncp,tranId_nc_cpf,tranDate_nc_cpf,TotalParaPiso_base,estado_piso,TotalParaPiso_final,observacion_piso,entityid_key,periodo_mes_key,costo_original,costo_abs,primer_periodo_lote,costo_aplicaria_en_este_mes
0,2025-04-16,2025-06-15,SM58211,3234,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,5008799.99,-1.0,5008799.99,1815487,1815489,B,0,True,CustInvc,15,RITEL BOGOTA,166958,0,COTIZACION 166962 y 166958 DEL PROYECTO Veramo...,factura,True,False,2025-04,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,166958,5008799.99,1815487,1815489,0,0,513,NaN,<NA>,NaT,<NA>,NaN,NaN,False,0,False,,<NA>,NaT,<NA>,ninguno,False,,NaT,False,ninguna,False,<NA>,,SM58211,<NA>,NaT,<NA>,NaT,5008799.99,suma,5008799.99,Suma para piso,ADRIANA MARITZA QUITIAN ROZO,2025-04,-586824.14,586824.14,2025-02,0.00
1,2025-04-11,2025-05-11,SM58017,3222,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,NaN,-1.0,2700000.00,1812239,1812257,B,0,True,CustInvc,15,RITEL BOGOTA,133683,0,COTIZACION 133683 - 133681 - 132113 DEL PROYEC...,factura,True,False,2025-04,INSPECCION RITEL INFRAESTRUCTURA SOPORTE Y TDT,133683,2700000.00,1812239,1812257,0,0,588,NaN,<NA>,NaT,<NA>,NaN,NaN,False,0,False,,<NA>,NaT,<NA>,ninguno,False,,NaT,False,ninguna,False,<NA>,,SM58017,<NA>,NaT,<NA>,NaT,2700000.00,suma,2700000.00,Suma para piso,ADRIANA MARITZA QUITIAN ROZO,2025-04,NaN,NaN,2023-06,0.00
2,2025-04-16,2025-05-16,SM58202,68218,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL BASICAS,6200000.00,-1.0,6200000.00,1758062,1815342,B,0,True,CustInvc,10,RETIE PEREIRA,177076,0,Proyecto:3 SISTEMAS FOTOVOLTAICOS ETEX (PROYEC...,factura,True,False,2025-04,INSPECCION RETIE USO FINAL BASICAS,177076,6200000.00,1758062,1815342,0,0,2417,-6200000.00,NC4352,2025-07-23,<NA>,NaN,servicio_unica,True,1,False,,SM56427,2025-02-21,SM56427,reemplazo_factura_nc,True,No suma por ser reemplazo de SM56427,2025-07,False,factura_nc_otro_mes_sin_reemplazo,True,2025-04,Suma en su mes original; recalcular piso de 20...,SM58202,<NA>,NaT,<NA>,NaT,6200000.00,no suma,0.00,No suma por ser reemplazo de SM56427,ADRIANA MARITZA QUITIAN ROZO,2025-04,-294000.00,294000.00,2025-02,0.00
3,2025-04-04,2025-04-04,SM57672,63803,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL BASICAS,1550000.00,-1.0,1550000.00,1805525,1805888,B,0,True,CustInvc,10,RETIE BOGOTA,181116,0,COTIZACION 181116 DEL PROYECTO CLARET REAL VI,factura,True,False,2025-04,INSPECCION RETIE USO FINAL BASICAS,181116,1550000.00,1805525,1805888,0,0,4459,NaN,<NA>,NaT,<NA>,NaN,NaN,False,0,False,,<NA>,NaT,<NA>,ninguno,False,,NaT,False,ninguna,False,<NA>,,SM57672,<NA>,NaT,<NA>,NaT,1550000.00,suma,1550000.00,Suma para piso,ADRIANA MARITZA QUITIAN ROZO,2025-04,-465000.00,465000.00,2025-04,465000.00
4,2025-04-29,2025-05-14,SM58469,193076,ADRIANA MARITZA QUITIAN ROZO,INSPECCION RETIE USO FINAL BASICAS,3829171.00,-1.0,3829171.00,1824086,1824149,B,0,True,CustInvc,10,RETIE BOGOTA,169988,0,COTIZACION 169988 DEL PROYECTO RETIE RESERVA D...,factura,True,False,2025-04,INSPECCION RETIE USO FINAL BASICAS,169988,3829171.00,1824086,1824149,0,0,6531,NaN,<NA>,NaT,<NA>,NaN,NaN,False,0,False,,<NA>,NaT,<NA>,ninguno,False,,NaT,False,ninguna,False,<NA>,,SM58469,<NA>,NaT,<NA>,NaT,3829171.00,